# **Notebook 02 — Telecom RAG V1 Build & Retrieval Architecture**

**Module:** 2 — Telecom RAG V1  
**Purpose:** Build and validate the frozen RAG V1 knowledge/retrieval layer that supports the later Module 2 model experiments.

### Scope
This notebook covers corpus preparation, chunking, BGE-M3 embedding generation, FAISS indexing, retrieval validation and the engineering experiments used to select the final embedding pipeline.

### Benchmark continuity
Module 2 reuses both frozen Module 1 benchmark tracks:
- Track 1 — 20 custom telecom engineering questions
- Track 2 — 32 industry benchmark questions

### Final architectural baseline
The final RAG V1 architecture uses `BAAI/bge-m3`, 1,024-dimensional embeddings, FAISS retrieval and Top-7 evidence retrieval. Exploratory optimization trials are retained where they materially explain the final design choice.

### Reproducibility note
Large corpus, embedding and FAISS artifacts remain external to Git. The notebook documents the build process and persisted-artifact workflow without embedding those large binaries in the repository.


# **Environment & Development Setup**

## **Install All Required Libraries**

In [1]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 2.1 — INSTALL ADDITIONAL RAG DEPENDENCIES
# =============================================================================

# Install the libraries required for telecom RAG data acquisition,
# document processing, embeddings and vector retrieval.
#
# The current runtime is CPU-based because inference is not yet being
# performed. GPU-specific acceleration can be enabled later when required.

!pip install -q \
    transformers \
    accelerate \
    bitsandbytes \
    huggingface_hub \
    safetensors \
    sentencepiece \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    python-docx \
    python-pptx \
    beautifulsoup4 \
    pyarrow \
    tqdm

print("All required Module 2 libraries installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 48.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 88.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 9.2 MB/s eta 0:00:00
All required Module 2 libraries installed successfully.


## **Import All Required Libraries**

In [3]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 2.2 — IMPORT REQUIRED LIBRARIES
# =============================================================================

# ---------------------------------------------------------------------------
# Core scientific stack
# ---------------------------------------------------------------------------
import numpy as np
import scipy
import pandas as pd

# ---------------------------------------------------------------------------
# Standard Python libraries
# ---------------------------------------------------------------------------
import os
import gc
import json
import shutil
from pathlib import Path

# ---------------------------------------------------------------------------
# Progress monitoring
# ---------------------------------------------------------------------------
from tqdm.auto import tqdm

# Data / document processing
# ---------------------------------------------------------------------------
import pyarrow
import pyarrow.parquet as pq
from docx import Document
from bs4 import BeautifulSoup
from pypdf import PdfReader

# ---------------------------------------------------------------------------
# HTTP / source acquisition
# ---------------------------------------------------------------------------
import requests

# ---------------------------------------------------------------------------
# PyTorch
# ---------------------------------------------------------------------------
import torch

# ---------------------------------------------------------------------------
# Hugging Face / LLM inference
# ---------------------------------------------------------------------------
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

from huggingface_hub import login, HfApi, snapshot_download

# ---------------------------------------------------------------------------
# Embeddings
# ---------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------------------------
# Vector similarity search
# ---------------------------------------------------------------------------
import faiss

# ---------------------------------------------------------------------------
# PDF document processing
# ---------------------------------------------------------------------------
from pypdf import PdfReader

print("All required libraries imported successfully.")

# Display key package versions for reproducibility
print("\nPackage Versions")
print("-" * 40)
print(f"NumPy           : {np.__version__}")
print(f"SciPy           : {scipy.__version__}")
print(f"Pandas          : {pd.__version__}")
print(f"PyTorch         : {torch.__version__}")
print(f"FAISS           : {faiss.__version__}")
print(f"PyArrow         : {pyarrow.__version__}")

All required libraries imported successfully.

Package Versions
----------------------------------------
NumPy           : 2.0.2
SciPy           : 1.16.3
Pandas          : 2.3.3
PyTorch         : 2.10.0+cu128
FAISS           : 1.15.0
PyArrow         : 24.0.0


### **Observation**

The RAG environment initialized successfully in the Colab GPU runtime. All required libraries imported without errors, and the core package versions were verified for reproducibility. The environment is ready for baseline model and Judge LLM configuration.

## **GPU Verification**

In [3]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# GPU / CUDA VERIFICATION
# =============================================================================

import torch


print("=" * 80)
print("GPU / CUDA VERIFICATION")
print("=" * 80)

print(f"PyTorch version   : {torch.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")

if torch.cuda.is_available():

    print(f"CUDA version      : {torch.version.cuda}")
    print(f"GPU count         : {torch.cuda.device_count()}")

    for i in range(torch.cuda.device_count()):

        props = torch.cuda.get_device_properties(i)

        total_memory = (
            props.total_memory / (1024**3)
        )

        allocated = (
            torch.cuda.memory_allocated(i)
            / (1024**3)
        )

        reserved = (
            torch.cuda.memory_reserved(i)
            / (1024**3)
        )

        free_memory = max(
            0,
            total_memory - reserved
        )

        print(f"\nGPU {i}")
        print(f"  Name           : {props.name}")
        print(
            f"  Total Memory   : "
            f"{total_memory:.2f} GB"
        )
        print(
            f"  Free Memory*   : "
            f"{free_memory:.2f} GB"
        )
        print(
            f"  Allocated      : "
            f"{allocated:.2f} GB"
        )
        print(
            f"  Reserved       : "
            f"{reserved:.2f} GB"
        )
        print(
            f"  Compute Cap.   : "
            f"{props.major}.{props.minor}"
        )

    # -------------------------------------------------------------------------
    # Select first available GPU
    # -------------------------------------------------------------------------

    DEVICE = torch.device("cuda:0")

    torch.cuda.set_device(0)

    print(f"\nSelected device  : {DEVICE}")

else:

    DEVICE = torch.device("cpu")

    print("\nWARNING: CUDA GPU not detected.")
    print(
        "In Colab, enable GPU via "
        "Runtime → Change runtime type → T4 GPU "
        "(or the GPU available to your runtime)."
    )

print("\nGPU verification completed.")

GPU / CUDA VERIFICATION
PyTorch version   : 2.11.0+cu128
CUDA available    : True
CUDA version      : 12.8
GPU count         : 1

GPU 0
  Name           : Tesla T4
  Total Memory   : 14.56 GB
  Free Memory*   : 14.56 GB
  Allocated      : 0.00 GB
  Reserved       : 0.00 GB
  Compute Cap.   : 7.5

Selected device  : cuda:0

GPU verification completed.


### **Observation — Colab GPU Verification**

- **1 × Tesla T4 GPU** detected.
- **14.56 GB GPU memory** available.
- **CUDA 12.8** with PyTorch 2.11.0.
- GPU memory currently unused.
- **Selected device:** `cuda:0`

# **LLM Model Loading**

## **Model Configuration**

In [ ]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 3.1 — BASELINE MODEL & JUDGE CONFIGURATION
# =============================================================================

# Retain the exact models established and validated in Module 1.
# This preserves experimental consistency between the baseline and RAG stages.

# ---------------------------------------------------------------------------
# Model identifiers
# ---------------------------------------------------------------------------

GENERAL_LLM_ID = "EssentialAI/rnj-1-instruct"
OTEL_LLM_ID = "farbodtavakkoli/OTel-LLM-8.3B-IT"
JUDGE_LLM_ID = "Qwen/Qwen2.5-7B-Instruct"

# ---------------------------------------------------------------------------
# Device configuration
# ---------------------------------------------------------------------------

MODEL_DEVICE = DEVICE

# ---------------------------------------------------------------------------
# Generation configuration retained from Module 1
# ---------------------------------------------------------------------------

TEMPERATURE = 0.0
TOP_K = 50
TOP_P = 0.95

print("=" * 80)
print("BASELINE MODEL & JUDGE CONFIGURATION")
print("=" * 80)

print(f"General LLM      : {GENERAL_LLM_ID}")
print(f"OTel 1.0 LLM     : {OTEL_LLM_ID}")
print(f"Judge LLM        : {JUDGE_LLM_ID}")
print(f"Primary device   : {MODEL_DEVICE}")

print("\nGeneration Configuration")
print("-" * 40)
print(f"Temperature      : {TEMPERATURE}")
print(f"Top-k            : {TOP_K}")
print(f"Top-p            : {TOP_P}")

BASELINE MODEL & JUDGE CONFIGURATION
General LLM      : EssentialAI/rnj-1-instruct
OTel 1.0 LLM     : farbodtavakkoli/OTel-LLM-8.3B-IT
Judge LLM        : Qwen/Qwen2.5-7B-Instruct
Primary device   : cuda:0

Generation Configuration
----------------------------------------
Temperature      : 0.0
Top-k            : 50
Top-p            : 0.95


### Observation

The validated Module 1 model configuration has been carried forward unchanged. The General LLM, OTel 1.0 LLM and independent Judge LLM are configured with the established generation parameters, supporting controlled comparison between the baseline and RAG stages.

## **Quantisation Configuration**

In [ ]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 3.2 — QUANTISATION CONFIGURATION
# =============================================================================

# 4-bit NF4 quantisation reduces GPU memory requirements while retaining
# practical inference quality for the baseline and RAG experiments.

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("=" * 80)
print("4-BIT QUANTISATION CONFIGURATION")
print("=" * 80)

print("Load in 4-bit          : True")
print("Quantisation type      : NF4")
print("Compute dtype          : float16")
print("Double quantisation   : True")

4-BIT QUANTISATION CONFIGURATION
Load in 4-bit          : True
Quantisation type      : NF4
Compute dtype          : float16
Double quantisation   : True


### Observation

4-bit NF4 quantisation is configured with float16 computation and double quantisation to reduce GPU memory usage while supporting efficient inference across the available Kaggle GPUs.

## **Load General LLM**

In [ ]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 3.3 — LOAD GENERAL LLM
# =============================================================================

print("=" * 80)
print("LOADING GENERAL LLM")
print("=" * 80)

# Load tokenizer using the model's native configuration.
general_tokenizer = AutoTokenizer.from_pretrained(
    GENERAL_LLM_ID,
    trust_remote_code=True,
)

# Load the General LLM using 4-bit quantisation.
general_model = AutoModelForCausalLM.from_pretrained(
    GENERAL_LLM_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded successfully : {GENERAL_LLM_ID}")
print(f"Model device map          : {general_model.hf_device_map}")

LOADING GENERAL LLM


config.json:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


tokenizer_config.json:   0%|          | 0.00/55.9k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/418 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/226 [00:00<?, ?B/s]

Model loaded successfully : EssentialAI/rnj-1-instruct
Model device map          : {'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'model.rotary_emb': 1}


### Observation

The General LLM loaded successfully in 4-bit NF4 quantised mode. The model weights were distributed across both available Tesla T4 GPUs using automatic device mapping. Hugging Face authentication and RoPE configuration warnings were observed but did not prevent successful model loading.

## **Load OTel 1.0 LLM**

In [ ]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 3.4 — LOAD OTel 1.0 LLM
# =============================================================================

print("=" * 80)
print("LOADING OTel 1.0 LLM")
print("=" * 80)

# Load tokenizer using the model's native configuration.
otel_tokenizer = AutoTokenizer.from_pretrained(
    OTEL_LLM_ID,
    trust_remote_code=True,
)

# Load OTel 1.0 using the same quantisation approach.
otel_model = AutoModelForCausalLM.from_pretrained(
    OTEL_LLM_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded successfully : {OTEL_LLM_ID}")
print(f"Model device map          : {otel_model.hf_device_map}")

LOADING OTel 1.0 LLM


config.json:   0%|          | 0.00/2.64k [00:00<?, ?B/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'sliding_attention', 'full_attention'}


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

...

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/419 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/248 [00:00<?, ?B/s]

Model loaded successfully : farbodtavakkoli/OTel-LLM-8.3B-IT
Model device map          : {'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'model.rotary_emb': 1}


### Observation

The OTel 1.0 LLM loaded successfully in 4-bit NF4 quantised mode and was automatically distributed across both available Tesla T4 GPUs. RoPE and generation-configuration warnings were reported but did not prevent successful model loading or GPU placement.

## **Load Independent Judge LLM**

In [ ]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 3.5 — LOAD INDEPENDENT JUDGE LLM
# =============================================================================

print("=" * 80)
print("LOADING INDEPENDENT JUDGE LLM")
print("=" * 80)

# The Judge LLM remains independent from the two models being evaluated.
judge_tokenizer = AutoTokenizer.from_pretrained(
    JUDGE_LLM_ID,
    trust_remote_code=True,
)

# Use the same 4-bit strategy to control GPU memory consumption.
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_LLM_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Model loaded successfully : {JUDGE_LLM_ID}")
print(f"Model device map          : {judge_model.hf_device_map}")

LOADING INDEPENDENT JUDGE LLM


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully : Qwen/Qwen2.5-7B-Instruct
Model device map          : {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 1, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


### Observation

The independent Judge LLM loaded successfully in 4-bit NF4 quantised mode and was automatically distributed across both Tesla T4 GPUs. The complete model weights were loaded successfully, confirming that the Judge LLM is ready for evaluation.

## **Post LLM Load GPU Verification**

In [ ]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 3.6 — MODEL & GPU PLACEMENT VERIFICATION
# =============================================================================

print("=" * 80)
print("MODEL & GPU PLACEMENT VERIFICATION")
print("=" * 80)

# ---------------------------------------------------------------------------
# Display device maps for each loaded model
# ---------------------------------------------------------------------------

print("\nGeneral LLM Device Map")
print("-" * 40)
print(general_model.hf_device_map)

print("\nOTel 1.0 LLM Device Map")
print("-" * 40)
print(otel_model.hf_device_map)

print("\nJudge LLM Device Map")
print("-" * 40)
print(judge_model.hf_device_map)

# ---------------------------------------------------------------------------
# Report GPU memory utilisation
# ---------------------------------------------------------------------------

if torch.cuda.is_available():

    print("\nGPU Memory Utilisation")
    print("-" * 40)

    for i in range(torch.cuda.device_count()):

        allocated = torch.cuda.memory_allocated(i) / (1024 ** 3)
        reserved = torch.cuda.memory_reserved(i) / (1024 ** 3)

        print(f"GPU {i}")
        print(f"  Name       : {torch.cuda.get_device_name(i)}")
        print(f"  Allocated  : {allocated:.2f} GB")
        print(f"  Reserved   : {reserved:.2f} GB")

else:
    print("\nCUDA is not available.")

print("\nModel and GPU placement verification completed.")

MODEL & GPU PLACEMENT VERIFICATION

General LLM Device Map
----------------------------------------
{'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'model.rotary_emb': 1}

OTel 1.0 LLM Device Map
----------------------------------------
{'model.embed_tokens': 0, 'lm_head': 0, 'model.layers.0': 0,

### Observation

All three models are successfully loaded and distributed across both Tesla T4 GPUs using automatic device mapping. GPU memory utilisation remains within the available capacity, confirming stable multi-GPU placement and readiness for the RAG implementation.

# **Telecom Knowledge Base Preparation**

## **Mount Google Drive for Data Storage**

In [11]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 4.4.12 — MOUNT GOOGLE DRIVE
# =============================================================================

from google.colab import drive
from pathlib import Path

print("=" * 80)
print("MOUNTING GOOGLE DRIVE")
print("=" * 80)

drive.mount("/content/drive")

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/telecom_knowledge_base"
)

DRIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("\nGoogle Drive mounted successfully.")
print(f"Persistent project location : {DRIVE_ROOT}")

MOUNTING GOOGLE DRIVE
Mounted at /content/drive

Google Drive mounted successfully.
Persistent project location : /content/drive/MyDrive/telecom_knowledge_base


In [4]:
from pathlib import Path

INPUT_DIR = Path(
    "/kaggle/input/datasets/cliffordimaguezegie/telecom-reconciled-chunks"
)

print("=" * 80)
print("KAGGLE DATASET VERIFICATION")
print("=" * 80)

print(f"Path exists : {INPUT_DIR.exists()}")

shards = sorted(
    INPUT_DIR.rglob("*.jsonl")
)

print(f"JSONL shards: {len(shards):,}")

if shards:
    print(f"First shard : {shards[0].name}")
    print(f"Last shard  : {shards[-1].name}")

print("=" * 80)

KAGGLE DATASET VERIFICATION
Path exists : True
JSONL shards: 151
First shard : chunks_000001.jsonl
Last shard  : chunks_000151.jsonl


### **Check Google Drive Capacity**

In [12]:
# =============================================================================
# CHECK GOOGLE DRIVE CAPACITY
# =============================================================================

import shutil
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/telecom_knowledge_base"
)

total, used, free = shutil.disk_usage(
    "/content/drive/MyDrive"
)

print("=" * 80)
print("GOOGLE DRIVE STORAGE STATUS")
print("=" * 80)

print(
    f"Total Drive storage : {total / (1024**3):.2f} GB"
)

print(
    f"Used Drive storage  : {used / (1024**3):.2f} GB"
)

print(
    f"Free Drive storage  : {free / (1024**3):.2f} GB"
)

print(
    f"Project location    : {DRIVE_ROOT}"
)

GOOGLE DRIVE STORAGE STATUS
Total Drive storage : 15.00 GB
Used Drive storage  : 9.44 GB
Free Drive storage  : 5.56 GB
Project location    : /content/drive/MyDrive/telecom_knowledge_base


## **Define Knowledge Base Scope**

In [3]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 4.1 — DEFINE TELECOM KNOWLEDGE BASE SCOPE
# =============================================================================

# Define the technical scope of the RAG knowledge base.
# The scope combines authoritative standards, practical engineering material,
# academic foundations and selected telecom operations/API knowledge.

KNOWLEDGE_BASE_SCOPE = {
    "5G_and_3GPP": [
        "5G System Architecture",
        "5G Core Network",
        "AMF, SMF, UPF and related Network Functions",
        "Service-Based Architecture (SBA)",
        "Registration and PDU Session procedures",
        "Network slicing",
        "5G security",
        "5G NR and RAN architecture",
    ],

    "RAN_and_Open_RAN": [
        "5G RAN architecture",
        "O-RAN architecture",
        "O-RAN interfaces",
        "Near-RT RIC and Non-RT RIC",
        "SMO and O-Cloud",
        "RAN disaggregation",
        "AI/ML in RAN",
    ],

    "Cloud_Native_Telecom": [
        "NFV",
        "VNFs and CNFs",
        "Telco Cloud",
        "Containerisation",
        "Kubernetes",
        "Cloud-native network functions",
        "Network orchestration",
        "Lifecycle management",
    ],

    "Telecom_Operations_and_Management": [
        "Network management",
        "Service management",
        "Fault management",
        "Performance management",
        "Configuration management",
        "Network automation",
        "Closed-loop automation",
        "OSS/BSS integration concepts",
    ],

    "Network_APIs_and_Exposure": [
        "Telecom APIs",
        "Network exposure",
        "CAMARA APIs",
        "TM Forum Open APIs",
        "5G API enablement",
        "Programmable networks",
        "Network capability exposure",
    ],

    "Telecom_Engineering_Foundations": [
        "Wireless communications",
        "Radio systems engineering",
        "Propagation and channel modelling",
        "MIMO and diversity",
        "OFDM",
        "Modulation and coding",
        "Digital communications",
        "Telecom engineering principles",
    ],
}

print("=" * 80)
print("TELECOM KNOWLEDGE BASE SCOPE")
print("=" * 80)

for domain, topics in KNOWLEDGE_BASE_SCOPE.items():
    print(f"\n{domain.replace('_', ' ')}")
    print("-" * 60)

    for topic in topics:
        print(f"  • {topic}")

print("\nKnowledge base scope defined successfully.")
print(f"Total domains : {len(KNOWLEDGE_BASE_SCOPE)}")
print(
    f"Total topics  : "
    f"{sum(len(topics) for topics in KNOWLEDGE_BASE_SCOPE.values())}"
)

TELECOM KNOWLEDGE BASE SCOPE

5G and 3GPP
------------------------------------------------------------
  • 5G System Architecture
  • 5G Core Network
  • AMF, SMF, UPF and related Network Functions
  • Service-Based Architecture (SBA)
  • Registration and PDU Session procedures
  • Network slicing
  • 5G security
  • 5G NR and RAN architecture

RAN and Open RAN
------------------------------------------------------------
  • 5G RAN architecture
  • O-RAN architecture
  • O-RAN interfaces
  • Near-RT RIC and Non-RT RIC
  • SMO and O-Cloud
  • RAN disaggregation
  • AI/ML in RAN

Cloud Native Telecom
------------------------------------------------------------
  • NFV
  • VNFs and CNFs
  • Telco Cloud
  • Containerisation
  • Kubernetes
  • Cloud-native network functions
  • Network orchestration
  • Lifecycle management

Telecom Operations and Management
------------------------------------------------------------
  • Network management
  • Service management
  • Fault management
  • Pe

### **Observation**

The knowledge base scope has been successfully defined across six telecom domains covering 46 technical topics. The scope provides a balanced foundation spanning standards, RAN/Open RAN, cloud-native technologies, operations, network APIs and core telecommunications engineering principles.

## **Define Knowledge Sources / Source Registry**

In [4]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 4.2 — DEFINE KNOWLEDGE SOURCES / SOURCE REGISTRY
# =============================================================================

# Register approved knowledge sources before automated acquisition.
# Evaluation datasets are recorded separately and will NOT form part of the
# primary RAG knowledge corpus to avoid benchmark contamination.

KNOWLEDGE_SOURCES = [

    # -------------------------------------------------------------------------
    # Tier 1 — Authoritative / Vendor-Neutral Standards
    # -------------------------------------------------------------------------
    {
        "source_id": "GSMA_3GPP",
        "name": "GSMA 3GPP Specifications",
        "type": "standards",
        "provider": "3GPP / GSMA",
        "access": "Hugging Face Dataset",
        "uri": "GSMA/3GPP",
        "domains": ["5G_and_3GPP"],
        "rag_use": True,
    },

    {
        "source_id": "GSMA_ORAN",
        "name": "O-RAN Specifications and Technical Reports",
        "type": "standards",
        "provider": "O-RAN Alliance / GSMA",
        "access": "Hugging Face Dataset",
        "uri": "GSMA/oran",
        "domains": ["RAN_and_Open_RAN"],
        "rag_use": True,
    },

    {
        "source_id": "GSMA_ETSI",
        "name": "ETSI Telecom Standards",
        "type": "standards",
        "provider": "ETSI / GSMA",
        "access": "Hugging Face Dataset",
        "uri": "GSMA/etsi",
        "domains": ["Cloud_Native_Telecom"],
        "rag_use": True,
    },

    {
        "source_id": "GSMA_ITU",
        "name": "ITU-T Telecommunications Standards",
        "type": "standards",
        "provider": "ITU-T / GSMA",
        "access": "Hugging Face Dataset",
        "uri": "GSMA/itu",
        "domains": [
            "5G_and_3GPP",
            "Cloud_Native_Telecom",
            "Telecom_Operations_and_Management",
        ],
        "rag_use": True,
    },

    {
        "source_id": "GSMA_GSMA",
        "name": "GSMA Specifications and Industry Guidelines",
        "type": "standards",
        "provider": "GSMA",
        "access": "Hugging Face Dataset",
        "uri": "GSMA/gsma",
        "domains": [
            "5G_and_3GPP",
            "Telecom_Operations_and_Management",
        ],
        "rag_use": True,
    },

    {
        "source_id": "GSMA_TMF",
        "name": "TM Forum Standards and Open APIs",
        "type": "standards",
        "provider": "TM Forum / GSMA",
        "access": "Hugging Face Dataset",
        "uri": "GSMA/tmforum",
        "domains": [
            "Telecom_Operations_and_Management",
            "Network_APIs_and_Exposure",
        ],
        "rag_use": True,
    },

    {
        "source_id": "GSMA_CAMARA",
        "name": "CAMARA Network APIs",
        "type": "standards",
        "provider": "CAMARA / GSMA",
        "access": "Hugging Face Dataset",
        "uri": "GSMA/camara",
        "domains": ["Network_APIs_and_Exposure"],
        "rag_use": True,
    },

    # -------------------------------------------------------------------------
    # Tier 2 — Open / Practical Telecom Engineering
    # -------------------------------------------------------------------------
    {
        "source_id": "SRARAN",
        "name": "srsRAN Project Documentation",
        "type": "open_source",
        "provider": "srsRAN",
        "access": "Official Documentation",
        "uri": "https://docs.srsran.com/",
        "domains": [
            "RAN_and_Open_RAN",
            "5G_and_3GPP",
        ],
        "rag_use": True,
    },

    {
        "source_id": "OPEN5GS",
        "name": "Open5GS Documentation",
        "type": "open_source",
        "provider": "Open5GS",
        "access": "Official Documentation",
        "uri": "https://open5gs.org/",
        "domains": [
            "5G_and_3GPP",
            "Cloud_Native_Telecom",
        ],
        "rag_use": True,
    },

    {
        "source_id": "FREE5GC",
        "name": "free5GC Documentation",
        "type": "open_source",
        "provider": "free5GC",
        "access": "Official Documentation",
        "uri": "https://free5gc.org/",
        "domains": [
            "5G_and_3GPP",
            "Cloud_Native_Telecom",
        ],
        "rag_use": True,
    },

    {
        "source_id": "OAI",
        "name": "OpenAirInterface Documentation",
        "type": "open_source",
        "provider": "OpenAirInterface",
        "access": "Official Documentation",
        "uri": "https://openairinterface.org/",
        "domains": [
            "RAN_and_Open_RAN",
            "5G_and_3GPP",
        ],
        "rag_use": True,
    },

    {
        "source_id": "ONAP",
        "name": "ONAP Documentation",
        "type": "open_source",
        "provider": "Linux Foundation",
        "access": "Official Documentation",
        "uri": "https://docs.onap.org/",
        "domains": [
            "Cloud_Native_Telecom",
            "Telecom_Operations_and_Management",
        ],
        "rag_use": True,
    },

    # -------------------------------------------------------------------------
    # Tier 3 — Academic / Open Educational Material
    # -------------------------------------------------------------------------
    {
        "source_id": "MIT_WIRELESS",
        "name": "MIT OpenCourseWare — Principles of Wireless Communications",
        "type": "academic",
        "provider": "MIT OpenCourseWare",
        "access": "Open Course Material",
        "uri": "https://ocw.mit.edu/",
        "domains": ["Telecom_Engineering_Foundations"],
        "rag_use": True,
    },

    {
        "source_id": "MIT_DIGITAL_COMM",
        "name": "MIT OpenCourseWare — Principles of Digital Communications",
        "type": "academic",
        "provider": "MIT OpenCourseWare",
        "access": "Open Course Material",
        "uri": "https://ocw.mit.edu/courses/6-450-principles-of-digital-communications-i-fall-2006/",
        "domains": ["Telecom_Engineering_Foundations"],
        "rag_use": True,
    },

    # -------------------------------------------------------------------------
    # Tier 4 — Vendor Engineering References
    # -------------------------------------------------------------------------
    {
        "source_id": "ERICSSON",
        "name": "Ericsson Technical Documentation",
        "type": "vendor",
        "provider": "Ericsson",
        "access": "Public Technical Material",
        "uri": "https://www.ericsson.com/",
        "domains": [
            "5G_and_3GPP",
            "RAN_and_Open_RAN",
            "Cloud_Native_Telecom",
        ],
        "rag_use": True,
    },

    {
        "source_id": "NOKIA",
        "name": "Nokia Technical Documentation",
        "type": "vendor",
        "provider": "Nokia",
        "access": "Public Technical Material",
        "uri": "https://www.nokia.com/",
        "domains": [
            "5G_and_3GPP",
            "Cloud_Native_Telecom",
            "Network_APIs_and_Exposure",
        ],
        "rag_use": True,
    },

    {
        "source_id": "CISCO",
        "name": "Cisco 5G and Telco Cloud Documentation",
        "type": "vendor",
        "provider": "Cisco",
        "access": "Public Technical Documentation",
        "uri": "https://www.cisco.com/",
        "domains": [
            "5G_and_3GPP",
            "Cloud_Native_Telecom",
            "Telecom_Operations_and_Management",
        ],
        "rag_use": True,
    },

    # -------------------------------------------------------------------------
    # Separate — Evaluation / Benchmark Sources
    # -------------------------------------------------------------------------
    {
        "source_id": "GSMA_OT_FULL",
        "name": "GSMA Open-Telco Full Benchmark",
        "type": "evaluation",
        "provider": "GSMA",
        "access": "Hugging Face Dataset",
        "uri": "GSMA/ot-full",
        "domains": ["Evaluation"],
        "rag_use": False,
    },

    {
        "source_id": "TELEQNA",
        "name": "TeleQnA",
        "type": "evaluation",
        "provider": "Netop",
        "access": "Hugging Face Dataset",
        "uri": "netop/TeleQnA",
        "domains": ["Evaluation"],
        "rag_use": False,
    },

    {
        "source_id": "TELEMATH",
        "name": "TeleMath",
        "type": "evaluation",
        "provider": "Netop",
        "access": "Hugging Face Dataset",
        "uri": "netop/TeleMath",
        "domains": ["Evaluation"],
        "rag_use": False,
    },

    {
        "source_id": "TELELOGS",
        "name": "TeleLogs",
        "type": "evaluation",
        "provider": "Netop",
        "access": "Hugging Face Dataset",
        "uri": "netop/TeleLogs",
        "domains": [
            "Evaluation",
            "Telecom_Operations_and_Management",
        ],
        "rag_use": False,
    },
]

# =============================================================================
# DISPLAY SOURCE REGISTRY
# =============================================================================

print("=" * 80)
print("TELECOM KNOWLEDGE SOURCE REGISTRY")
print("=" * 80)

for source in KNOWLEDGE_SOURCES:
    print(
        f"{source['source_id']:<18} | "
        f"{source['type']:<12} | "
        f"RAG={str(source['rag_use']):<5} | "
        f"{source['name']}"
    )

rag_sources = [s for s in KNOWLEDGE_SOURCES if s["rag_use"]]

print("\n" + "-" * 80)
print(f"Total registered sources : {len(KNOWLEDGE_SOURCES)}")
print(f"RAG knowledge sources   : {len(rag_sources)}")
print(
    f"Evaluation-only sources : "
    f"{len(KNOWLEDGE_SOURCES) - len(rag_sources)}"
)

TELECOM KNOWLEDGE SOURCE REGISTRY
GSMA_3GPP          | standards    | RAG=True  | GSMA 3GPP Specifications
GSMA_ORAN          | standards    | RAG=True  | O-RAN Specifications and Technical Reports
GSMA_ETSI          | standards    | RAG=True  | ETSI Telecom Standards
GSMA_ITU           | standards    | RAG=True  | ITU-T Telecommunications Standards
GSMA_GSMA          | standards    | RAG=True  | GSMA Specifications and Industry Guidelines
GSMA_TMF           | standards    | RAG=True  | TM Forum Standards and Open APIs
GSMA_CAMARA        | standards    | RAG=True  | CAMARA Network APIs
SRARAN             | open_source  | RAG=True  | srsRAN Project Documentation
OPEN5GS            | open_source  | RAG=True  | Open5GS Documentation
FREE5GC            | open_source  | RAG=True  | free5GC Documentation
OAI                | open_source  | RAG=True  | OpenAirInterface Documentation
ONAP               | open_source  | RAG=True  | ONAP Documentation
MIT_WIRELESS       | academic     | RAG=True

### **Observation**

The knowledge source registry has been successfully established with **21 registered sources**: 17 designated for RAG knowledge acquisition and 4 retained exclusively for evaluation. The registry provides a balanced mix of standards, open-source engineering documentation, academic material and vendor references while maintaining a clear separation between knowledge sources and benchmark data.

## **Locate and Load RAG Documents**

### **Verifying Disk Space Available**

In [8]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 2.3 — COLAB DISK + KNOWLEDGE BASE USAGE
# =============================================================================

import shutil
from pathlib import Path


def folder_size_gb(path):
    """
    Calculate the total size of files within a directory in GB.
    """

    path = Path(path)

    if not path.exists():
        return 0.0

    total_bytes = sum(
        file.stat().st_size
        for file in path.rglob("*")
        if file.is_file()
    )

    return total_bytes / (1024 ** 3)


# ---------------------------------------------------------------------------
# Colab disk usage
# ---------------------------------------------------------------------------

total, used, free = shutil.disk_usage("/content")

# ---------------------------------------------------------------------------
# Knowledge-base usage
# ---------------------------------------------------------------------------

kb_path = Path("/content/telecom_knowledge_base")
kb_size = folder_size_gb(kb_path)

print("=" * 80)
print("COLAB STORAGE STATUS")
print("=" * 80)

print(f"Total disk space        : {total / (1024**3):.2f} GB")
print(f"Used disk space         : {used / (1024**3):.2f} GB")
print(f"Free disk space         : {free / (1024**3):.2f} GB")
print(f"RAG knowledge base      : {kb_size:.2f} GB")

COLAB STORAGE STATUS
Total disk space        : 107.72 GB
Used disk space         : 20.38 GB
Free disk space         : 87.32 GB
RAG knowledge base      : 0.00 GB


#### **Observation**

- The Colab runtime provides **107.72 GB of total storage**, with **87.32 GB currently free**.
- The RAG knowledge-base directory is currently empty, confirming that the new Colab environment has been established independently and is ready for controlled corpus acquisition.

**Recommendation:**
- Proceed with knowledge-base acquisition and processing in Colab, using the available disk capacity while maintaining the previously established memory-safe streaming approach for the Telco Common Corpus.

### **Standard Knowledge Base Acquisition**

In [6]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# BULK KNOWLEDGE SOURCE ACQUISITION
# =============================================================================

from pathlib import Path
import shutil
import subprocess


# =============================================================================
# KNOWLEDGE-BASE STRUCTURE
# =============================================================================

KB_ROOT = Path("/content/telecom_knowledge_base")
STANDARDS_DIR = KB_ROOT / "standards"

KB_ROOT.mkdir(parents=True, exist_ok=True)
STANDARDS_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def folder_size_gb(path):
    """Return total file size within a directory in GB."""

    path = Path(path)

    if not path.exists():
        return 0.0

    return sum(
        file.stat().st_size
        for file in path.rglob("*")
        if file.is_file()
    ) / (1024 ** 3)


def source_status(path):
    """Return file count and total size for an acquired source."""

    path = Path(path)

    if not path.exists():
        return 0, 0.0

    files = [
        file
        for file in path.rglob("*")
        if file.is_file()
    ]

    size_gb = sum(
        file.stat().st_size
        for file in files
    ) / (1024 ** 3)

    return len(files), size_gb


def acquire_hf_source(
    repo_id,
    destination,
    include_pattern=None,
):
    """
    Acquire a Hugging Face dataset using the hf CLI.

    Output is suppressed to keep the notebook dashboard clean.
    """

    destination = Path(destination)

    command = [
        "hf",
        "download",
        f"datasets/{repo_id}",
        "--local-dir",
        str(destination),
    ]

    if include_pattern:
        command.extend([
            "--include",
            include_pattern,
        ])

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode != 0:

        error_text = (
            result.stderr.strip()
            or result.stdout.strip()
            or "Unknown acquisition error"
        )

        raise RuntimeError(
            f"Acquisition failed for {repo_id}:\n{error_text}"
        )


# =============================================================================
# SOURCE DEFINITIONS
# =============================================================================

SOURCES = [
    {
        "source": "GSMA",
        "repo": "GSMA/gsma",
        "destination": STANDARDS_DIR / "gsma",
        "include": "marked/**/raw.md",
    },
    {
        "source": "O-RAN",
        "repo": "GSMA/oran",
        "destination": STANDARDS_DIR / "oran",
        "include": "marked/**/raw.md",
    },
    {
        "source": "TM Forum",
        "repo": "GSMA/tmforum",
        "destination": STANDARDS_DIR / "tmforum",
        "include": "marked/**/raw.md",
    },
    {
        "source": "CAMARA",
        "repo": "GSMA/camara",
        "destination": STANDARDS_DIR / "camara",
        "include": "marked/**/raw.md",
    },
    {
        "source": "3GPP Release 18",
        "repo": "GSMA/3GPP-REL18",
        "destination": STANDARDS_DIR / "3gpp_rel18",
        "include": None,
    },
    {
        "source": "Telco Common Corpus",
        "repo": "GSMA/Telco-Common-Corpus",
        "destination": KB_ROOT / "GSMA-Telco-Common-Corpus",
        "include": None,
    },
    {
        "source": "ETSI",
        "repo": "GSMA/etsi",
        "destination": STANDARDS_DIR / "etsi",
        "include": "marked/**/raw.md",
    },
    {
        "source": "ITU-T",
        "repo": "GSMA/itu",
        "destination": STANDARDS_DIR / "itu",
        "include": "marked/**/raw.md",
    },
]


# =============================================================================
# DASHBOARD
# =============================================================================

def display_dashboard():

    total_disk, used_disk, free_disk = shutil.disk_usage(
        "/content"
    )

    print("\n" + "=" * 90)
    print("TELECOM KNOWLEDGE BASE — ACQUISITION DASHBOARD")
    print("=" * 90)

    print(
        f"{'SOURCE':<24}"
        f"{'STATUS':<12}"
        f"{'FILES':>12}"
        f"{'SIZE (GB)':>14}"
    )

    print("-" * 90)

    total_kb_size = 0

    for source in SOURCES:

        files, size_gb = source_status(
            source["destination"]
        )

        total_kb_size += size_gb

        status = "READY" if files else "PENDING"

        print(
            f"{source['source']:<24}"
            f"{status:<12}"
            f"{files:>12,}"
            f"{size_gb:>14.3f}"
        )

    print("-" * 90)

    print(
        f"{'TOTAL KNOWLEDGE BASE':<36}"
        f"{total_kb_size:>14.3f} GB"
    )

    print(
        f"{'COLAB FREE STORAGE':<36}"
        f"{free_disk / (1024**3):>14.2f} GB"
    )

    print("=" * 90)


# =============================================================================
# INITIAL STATUS
# =============================================================================

display_dashboard()


# =============================================================================
# ACQUISITION
# =============================================================================

for index, source in enumerate(SOURCES, start=1):

    files, size_gb = source_status(
        source["destination"]
    )

    # Skip complete existing sources.
    if files > 0:

        print(
            f"\n[{index}/{len(SOURCES)}] "
            f"{source['source']} — already present "
            f"({files:,} files / {size_gb:.3f} GB)"
        )

        continue

    print(
        f"\n[{index}/{len(SOURCES)}] "
        f"Acquiring {source['source']}..."
    )

    try:

        acquire_hf_source(
            repo_id=source["repo"],
            destination=source["destination"],
            include_pattern=source["include"],
        )

        files, size_gb = source_status(
            source["destination"]
        )

        print(
            f"✓ {source['source']} completed — "
            f"{files:,} files / {size_gb:.3f} GB"
        )

    except Exception as exc:

        print(
            f"✗ {source['source']} failed — "
            f"{type(exc).__name__}: {exc}"
        )


# =============================================================================
# FINAL DASHBOARD
# =============================================================================

display_dashboard()


TELECOM KNOWLEDGE BASE — ACQUISITION DASHBOARD
SOURCE                  STATUS             FILES     SIZE (GB)
------------------------------------------------------------------------------------------
GSMA                    PENDING                0         0.000
O-RAN                   PENDING                0         0.000
TM Forum                PENDING                0         0.000
CAMARA                  PENDING                0         0.000
3GPP Release 18         PENDING                0         0.000
Telco Common Corpus     PENDING                0         0.000
ETSI                    PENDING                0         0.000
ITU-T                   PENDING                0         0.000
------------------------------------------------------------------------------------------
TOTAL KNOWLEDGE BASE                         0.000 GB
COLAB FREE STORAGE                           87.32 GB

[1/8] Acquiring GSMA...
✓ GSMA completed — 768 files / 0.061 GB

[2/8] Acquiring O-RAN...
✓ O-

#### **Observation**

The bulk standards acquisition completed successfully across **8 major sources**, producing a combined corpus of **12.71 GB** while retaining **74.39 GB of free Colab storage**. ETSI and ITU-T have added substantial standards coverage without creating storage pressure.

**Recommendation:** Proceed to the non-standard knowledge layers — open cloud-native, academic/textbook, and broader vendor material — before beginning corpus processing.

### **Standard Knowledge Base Verification**

In [7]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 4.3.3 — KNOWLEDGE BASE VERIFICATION DASHBOARD
# =============================================================================

# Verify the current state of all acquired knowledge sources.
# This cell is safe to run while acquisition is in progress; sources that are
# incomplete or not yet present are reported accordingly.

from pathlib import Path
import shutil


# =============================================================================
# KNOWLEDGE-BASE LOCATIONS
# =============================================================================

KB_ROOT = Path("/content/telecom_knowledge_base")

ACQUIRED_SOURCES = {
    "GSMA": KB_ROOT / "standards" / "gsma",
    "O-RAN": KB_ROOT / "standards" / "oran",
    "TM Forum": KB_ROOT / "standards" / "tmforum",
    "CAMARA": KB_ROOT / "standards" / "camara",
    "3GPP Release 18": KB_ROOT / "standards" / "3gpp_rel18",
    "Telco Common Corpus": KB_ROOT / "GSMA-Telco-Common-Corpus",
    "ETSI": KB_ROOT / "standards" / "etsi",
    "ITU-T": KB_ROOT / "standards" / "itu",
    "Vendor Sources": KB_ROOT / "vendor",
}


# =============================================================================
# SOURCE VERIFICATION
# =============================================================================

def inspect_source(path):
    """
    Return file count and total size for a source directory.
    """

    path = Path(path)

    if not path.exists():
        return 0, 0.0

    files = [
        file
        for file in path.rglob("*")
        if file.is_file()
    ]

    total_bytes = sum(
        file.stat().st_size
        for file in files
    )

    return (
        len(files),
        total_bytes / (1024 ** 3),
    )


# =============================================================================
# DASHBOARD
# =============================================================================

print("=" * 100)
print("TELECOM KNOWLEDGE BASE — VERIFICATION DASHBOARD")
print("=" * 100)

print(
    f"{'SOURCE':<25}"
    f"{'STATUS':<12}"
    f"{'FILES':>12}"
    f"{'SIZE (GB)':>14}"
    f"  LOCATION"
)

print("-" * 100)

total_files = 0
total_size_gb = 0.0


for source_name, source_path in ACQUIRED_SOURCES.items():

    file_count, size_gb = inspect_source(
        source_path
    )

    total_files += file_count
    total_size_gb += size_gb

    # ---------------------------------------------------------------
    # Determine status
    # ---------------------------------------------------------------

    if file_count == 0:
        status = "NOT FOUND"

    elif source_name == "Telco Common Corpus":

        # TCC is large and can be recognised by its Parquet corpus.
        parquet_count = sum(
            1
            for file in source_path.rglob("*.parquet")
        )

        status = (
            "READY"
            if parquet_count > 0
            else "PARTIAL"
        )

    else:
        status = "READY"

    print(
        f"{source_name:<25}"
        f"{status:<12}"
        f"{file_count:>12,}"
        f"{size_gb:>14.3f}"
        f"  {source_path}"
    )


# =============================================================================
# COLAB STORAGE
# =============================================================================

total_disk, used_disk, free_disk = shutil.disk_usage(
    "/content"
)

print("-" * 100)

print(
    f"{'KNOWLEDGE BASE TOTAL':<37}"
    f"{total_files:>12,}"
    f"{total_size_gb:>14.3f} GB"
)

print(
    f"{'COLAB TOTAL STORAGE':<52}"
    f"{total_disk / (1024**3):>14.2f} GB"
)

print(
    f"{'COLAB USED STORAGE':<52}"
    f"{used_disk / (1024**3):>14.2f} GB"
)

print(
    f"{'COLAB FREE STORAGE':<52}"
    f"{free_disk / (1024**3):>14.2f} GB"
)

print("=" * 100)

print("\nKnowledge-base verification completed.")

TELECOM KNOWLEDGE BASE — VERIFICATION DASHBOARD
SOURCE                   STATUS             FILES     SIZE (GB)  LOCATION
----------------------------------------------------------------------------------------------------
GSMA                     READY                768         0.061  /content/telecom_knowledge_base/standards/gsma
O-RAN                    READY                510         0.054  /content/telecom_knowledge_base/standards/oran
TM Forum                 READY                336         0.019  /content/telecom_knowledge_base/standards/tmforum
CAMARA                   READY              2,283         0.006  /content/telecom_knowledge_base/standards/camara
3GPP Release 18          READY              1,668         0.861  /content/telecom_knowledge_base/standards/3gpp_rel18
Telco Common Corpus      READY                309         8.838  /content/telecom_knowledge_base/GSMA-Telco-Common-Corpus
ETSI                     READY             24,600         2.147  /content/telecom_kn

#### **Observation**

The standards corpus is fully acquired and verified, comprising **48,738 files across 8 sources** and **12.71 GB** of data. Colab retains **74.39 GB free**, providing ample capacity for the remaining vendor, cloud-native and academic knowledge layers.

### **Open Source Telecom Knowledge Base Acquisition**

In [8]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# OPEN-SOURCE TELECOM KNOWLEDGE ACQUISITION
# =============================================================================

from pathlib import Path
import subprocess

OPEN_SOURCE_ROOT = (
    KB_ROOT / "open_source"
)

OPEN_SOURCE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

OPEN_SOURCE_REPOS = {
    "srsRAN_Project_docs":
        "https://github.com/srsran/srsRAN_Project_docs.git",

    "open5gs":
        "https://github.com/open5gs/open5gs.git",

    "free5gc_docs":
        "https://github.com/free5gc/free5gc.github.io.git",

    "free5gc_labs":
        "https://github.com/free5gc/free5GLabs.git",
}

print("=" * 90)
print("OPEN-SOURCE TELECOM KNOWLEDGE ACQUISITION")
print("=" * 90)

for name, repo_url in OPEN_SOURCE_REPOS.items():

    destination = OPEN_SOURCE_ROOT / name

    if destination.exists():
        print(f"\n✓ {name} — already present")
        continue

    print(f"\n▶ Acquiring {name}...")

    result = subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            repo_url,
            str(destination),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode == 0:
        print(f"✓ {name} — acquired")
    else:
        print(
            f"✗ {name} — acquisition failed\n"
            f"{result.stderr[-500:]}"
        )

print("\n" + "=" * 90)
print("OPEN-SOURCE ACQUISITION COMPLETE")
print("=" * 90)

OPEN-SOURCE TELECOM KNOWLEDGE ACQUISITION

▶ Acquiring srsRAN_Project_docs...
✓ srsRAN_Project_docs — acquired

▶ Acquiring open5gs...
✓ open5gs — acquired

▶ Acquiring free5gc_docs...
✓ free5gc_docs — acquired

▶ Acquiring free5gc_labs...
✓ free5gc_labs — acquired

OPEN-SOURCE ACQUISITION COMPLETE


### **Open-Source Telecom Verification**

In [9]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# OPEN-SOURCE TELECOM VERIFICATION
# =============================================================================

from pathlib import Path
import shutil


# =============================================================================
# OPEN-SOURCE KNOWLEDGE BASE
# =============================================================================

OPEN_SOURCE_ROOT = (
    KB_ROOT / "open_source"
)


OPEN_SOURCE_SOURCES = {
    "srsRAN Project Docs":
        OPEN_SOURCE_ROOT / "srsRAN_Project_docs",

    "Open5GS":
        OPEN_SOURCE_ROOT / "open5gs",

    "free5GC Docs":
        OPEN_SOURCE_ROOT / "free5gc_docs",

    "free5GC Labs":
        OPEN_SOURCE_ROOT / "free5gc_labs",
}


# =============================================================================
# SOURCE INSPECTION
# =============================================================================

def inspect_source(path):

    path = Path(path)

    if not path.exists():
        return 0, 0.0

    files = [
        file
        for file in path.rglob("*")
        if file.is_file()
    ]

    total_bytes = sum(
        file.stat().st_size
        for file in files
    )

    return (
        len(files),
        total_bytes / (1024 ** 3)
    )


# =============================================================================
# DASHBOARD
# =============================================================================

print("=" * 100)
print("OPEN-SOURCE TELECOM — VERIFICATION DASHBOARD")
print("=" * 100)

print(
    f"{'SOURCE':<25}"
    f"{'STATUS':<12}"
    f"{'FILES':>12}"
    f"{'SIZE (GB)':>14}"
)

print("-" * 100)

total_files = 0
total_size = 0.0

for source_name, source_path in OPEN_SOURCE_SOURCES.items():

    file_count, size_gb = inspect_source(
        source_path
    )

    total_files += file_count
    total_size += size_gb

    if file_count == 0:
        status = "NOT FOUND"
    else:
        status = "READY"

    print(
        f"{source_name:<25}"
        f"{status:<12}"
        f"{file_count:>12,}"
        f"{size_gb:>14.3f}"
    )

# =============================================================================
# STORAGE
# =============================================================================

total_disk, used_disk, free_disk = (
    shutil.disk_usage("/content")
)

print("-" * 100)

print(
    f"{'OPEN-SOURCE TOTAL':<37}"
    f"{total_files:>12,}"
    f"{total_size:>14.3f} GB"
)

print(
    f"{'COLAB FREE STORAGE':<37}"
    f"{free_disk / (1024**3):>14.2f} GB"
)

print("=" * 100)

print("\nOpen-source telecom verification completed.")

OPEN-SOURCE TELECOM — VERIFICATION DASHBOARD
SOURCE                   STATUS             FILES     SIZE (GB)
----------------------------------------------------------------------------------------------------
srsRAN Project Docs      READY                351         0.040
Open5GS                  READY             11,637         0.164
free5GC Docs             READY                717         0.229
free5GC Labs             READY                 96         0.006
----------------------------------------------------------------------------------------------------
OPEN-SOURCE TOTAL                          12,801         0.438 GB
COLAB FREE STORAGE                            73.92 GB

Open-source telecom verification completed.


#### **Observation**

The open-source telecom layer was successfully acquired and verified, comprising **12,801 files across srsRAN, Open5GS, free5GC documentation and free5GC Labs**, with a combined footprint of **0.438 GB**. Colab retains **73.92 GB of free storage**.

**Recommendation:** Proceed with verification of the newly acquired cloud-native infrastructure layer, then continue to the academic/textbook knowledge layer.

### **Network Vendor Knowledge Base Acquisition**

In [10]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# NETWORK VENDOR KNOWLEDGE ACQUISITION
# =============================================================================

# Acquire a representative set of public technical references from major
# network vendors. Vendor sources are kept separate from standards, cloud
# platforms and open-source projects to preserve provenance.

import json
import requests
from pathlib import Path


# =============================================================================
# VENDOR KNOWLEDGE BASE
# =============================================================================

VENDOR_ROOT = KB_ROOT / "vendor_network"

VENDOR_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# SOURCE REGISTRY
# =============================================================================

NETWORK_VENDOR_SOURCES = [

    # =========================================================================
    # ERICSSON
    # =========================================================================
    {
        "vendor": "Ericsson",
        "title": "Security in 5G RAN and Core deployments",
        "url": (
            "https://www.ericsson.com/4ac68f/assets/local/"
            "reports-papers/white-papers/ericsson-whitepaper-5gran.pdf"
        ),
        "format": "pdf",
        "filename": "ericsson_5g_ran_core_security.pdf",
        "topics": [
            "5G",
            "RAN",
            "5G Core",
            "security",
            "O-RAN",
            "Cloud RAN",
        ],
    },

    {
        "vendor": "Ericsson",
        "title": "Security considerations of Cloud RAN",
        "url": (
            "https://www.ericsson.com/en/reports-and-papers/"
            "further-insights/cloud-ran-security"
        ),
        "format": "html",
        "filename": "ericsson_cloud_ran_security.html",
        "topics": [
            "Cloud RAN",
            "vRAN",
            "O-RAN",
            "cloud-native",
            "security",
        ],
    },

    {
        "vendor": "Ericsson",
        "title": "SMO enabling intelligent RAN operations",
        "url": (
            "https://www.ericsson.com/en/reports-and-papers/"
            "white-papers/smo-enabling-intelligent-ran-operations"
        ),
        "format": "html",
        "filename": "ericsson_smo_intelligent_ran_operations.html",
        "topics": [
            "O-RAN",
            "SMO",
            "automation",
            "RAN",
            "rApps",
            "intelligent operations",
        ],
    },

    {
        "vendor": "Ericsson",
        "title": "Cloud RAN",
        "url": (
            "https://www.ericsson.com/en/ran/cloud"
        ),
        "format": "html",
        "filename": "ericsson_cloud_ran.html",
        "topics": [
            "Cloud RAN",
            "vRAN",
            "cloud-native",
            "RAN",
        ],
    },

    {
        "vendor": "Ericsson",
        "title": "5G security for public and hybrid cloud deployments",
        "url": (
            "https://www.ericsson.com/en/reports-and-papers/"
            "further-insights/5g-security-for-hybrid-cloud"
        ),
        "format": "html",
        "filename": "ericsson_5g_hybrid_cloud_security.html",
        "topics": [
            "5G",
            "hybrid cloud",
            "security",
            "cloud",
            "network functions",
        ],
    },


    # =========================================================================
    # NOKIA
    # =========================================================================
    {
        "vendor": "Nokia",
        "title": "5G Core explained",
        "url": (
            "https://www.nokia.com/core-networks/5g-core/"
            "5g-core-explained/"
        ),
        "format": "html",
        "filename": "nokia_5g_core_explained.html",
        "topics": [
            "5G Core",
            "cloud-native",
            "microservices",
            "automation",
            "edge",
        ],
    },

    {
        "vendor": "Nokia",
        "title": "Open RAN",
        "url": (
            "https://www.nokia.com/radio-access/anyran/open-ran/"
        ),
        "format": "html",
        "filename": "nokia_open_ran.html",
        "topics": [
            "Open RAN",
            "RIC",
            "SMO",
            "O-RAN",
            "interoperability",
        ],
    },

    {
        "vendor": "Nokia",
        "title": "Cloud Native RAN",
        "url": (
            "https://www.nokia.com/networks/training/"
            "cloud-native-ran-professional/"
        ),
        "format": "html",
        "filename": "nokia_cloud_native_ran.html",
        "topics": [
            "Cloud RAN",
            "cloud-native",
            "microservices",
            "orchestration",
            "lifecycle management",
        ],
    },

    {
        "vendor": "Nokia",
        "title": "Cloud AI-RAN",
        "url": (
            "https://www.nokia.com/radio-access/anyran/cloud-ai-ran/"
        ),
        "format": "html",
        "filename": "nokia_cloud_ai_ran.html",
        "topics": [
            "AI-RAN",
            "Cloud RAN",
            "AI",
            "GPU",
            "cloud-native",
        ],
    },

    {
        "vendor": "Nokia",
        "title": "Open RAN Standardization",
        "url": (
            "https://www.nokia.com/standardization/"
            "technology-standards/o-ran/"
        ),
        "format": "html",
        "filename": "nokia_open_ran_standardization.html",
        "topics": [
            "Open RAN",
            "standardization",
            "O-RAN",
            "3GPP",
        ],
    },


    # =========================================================================
    # CISCO
    # =========================================================================
    {
        "vendor": "Cisco",
        "title": "What Is Open RAN",
        "url": (
            "https://www.cisco.com/site/us/en/learn/topics/"
            "networking/what-is-open-ran-oran.html"
        ),
        "format": "html",
        "filename": "cisco_open_ran.html",
        "topics": [
            "Open RAN",
            "vRAN",
            "RAN",
            "interoperability",
        ],
    },

    {
        "vendor": "Cisco",
        "title": "5G Transport",
        "url": (
            "https://www.cisco.com/c/en/us/solutions/"
            "service-provider/mobile-internet/5g-transport.html"
        ),
        "format": "html",
        "filename": "cisco_5g_transport.html",
        "topics": [
            "5G transport",
            "fronthaul",
            "midhaul",
            "backhaul",
            "IP",
            "Ethernet",
        ],
    },


    # =========================================================================
    # SAMSUNG
    # =========================================================================
    {
        "vendor": "Samsung",
        "title": "Samsung 5G Core Vision",
        "url": (
            "https://images.samsung.com/is/content/samsung/"
            "p5/global/business/networks/insights/white-paper/"
            "5g-core-vision/white-paper_5g-core-vision.pdf"
        ),
        "format": "pdf",
        "filename": "samsung_5g_core_vision.pdf",
        "topics": [
            "5G Core",
            "cloud-native",
            "automation",
            "network slicing",
            "MEC",
        ],
    },

    {
        "vendor": "Samsung",
        "title": "5G Core User Plane Performance",
        "url": (
            "https://images.samsung.com/is/content/samsung/"
            "assets/global/business/networks/insights/"
            "white-papers/samsung-exceeds-1tbps-in-5g-core-user-plane-"
            "with-intel/samsung-exceeds-1tbps-in-5g-core-user-plane-"
            "with-intel.pdf"
        ),
        "format": "pdf",
        "filename": "samsung_5g_core_user_plane_performance.pdf",
        "topics": [
            "5G Core",
            "user plane",
            "cloud-native",
            "performance",
        ],
    },
]


# =============================================================================
# DOWNLOAD FUNCTION
# =============================================================================

def download_vendor_source(source):

    vendor_dir = (
        VENDOR_ROOT
        / source["vendor"].lower()
    )

    vendor_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output_file = (
        vendor_dir
        / source["filename"]
    )

    # ---------------------------------------------------------------
    # Resume / skip
    # ---------------------------------------------------------------

    if (
        output_file.exists()
        and output_file.stat().st_size > 0
    ):
        return {
            "status": "EXISTING",
            "size_mb": (
                output_file.stat().st_size
                / (1024 ** 2)
            ),
            "path": str(output_file),
        }

    # ---------------------------------------------------------------
    # Download
    # ---------------------------------------------------------------

    try:

        response = requests.get(
            source["url"],
            stream=True,
            timeout=180,
            headers={
                "User-Agent":
                    "Mozilla/5.0 "
                    "(compatible; Telecom-RAG-Research/1.0)"
            },
        )

        response.raise_for_status()

        with open(
            output_file,
            "wb"
        ) as output:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):

                if chunk:
                    output.write(chunk)

        return {
            "status": "DOWNLOADED",
            "size_mb": (
                output_file.stat().st_size
                / (1024 ** 2)
            ),
            "path": str(output_file),
        }

    except Exception as exc:

        # Remove incomplete output if one was created.
        if output_file.exists():
            output_file.unlink()

        return {
            "status": "FAILED",
            "size_mb": 0.0,
            "path": "",
            "error": (
                f"{type(exc).__name__}: {exc}"
            ),
        }


# =============================================================================
# ACQUISITION DASHBOARD
# =============================================================================

results = []

print("=" * 100)
print("NETWORK VENDOR KNOWLEDGE ACQUISITION")
print("=" * 100)

for source in NETWORK_VENDOR_SOURCES:

    result = download_vendor_source(source)

    results.append({
        "vendor": source["vendor"],
        "title": source["title"],
        **result,
    })

    print(
        f"{source['vendor']:<12} | "
        f"{result['status']:<10} | "
        f"{result['size_mb']:>8.2f} MB | "
        f"{source['title']}"
    )

    if result["status"] == "FAILED":
        print(
            f"              Error: "
            f"{result.get('error', 'Unknown error')}"
        )


# =============================================================================
# SOURCE REGISTRY
# =============================================================================

registry_path = (
    VENDOR_ROOT
    / "vendor_source_registry.json"
)

with open(
    registry_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        NETWORK_VENDOR_SOURCES,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# FINAL DASHBOARD
# =============================================================================

total_disk, used_disk, free_disk = (
    shutil.disk_usage("/content")
)

downloaded_count = sum(
    result["status"] in {
        "DOWNLOADED",
        "EXISTING",
    }
    for result in results
)

downloaded_size = sum(
    result["size_mb"]
    for result in results
)

print("\n" + "=" * 100)
print("NETWORK VENDOR ACQUISITION SUMMARY")
print("=" * 100)

print(f"Sources defined       : {len(NETWORK_VENDOR_SOURCES)}")
print(f"Sources available     : {downloaded_count}")
print(f"Acquired size         : {downloaded_size:.2f} MB")
print(f"Colab free storage    : {free_disk / (1024**3):.2f} GB")
print(f"Registry              : {registry_path}")

print("=" * 100)

NETWORK VENDOR KNOWLEDGE ACQUISITION
Ericsson     | FAILED     |     0.00 MB | Security in 5G RAN and Core deployments
              Error: HTTPError: 403 Client Error: Forbidden for url: https://www.ericsson.com/4ac68f/assets/local/reports-papers/white-papers/ericsson-whitepaper-5gran.pdf
Ericsson     | FAILED     |     0.00 MB | Security considerations of Cloud RAN
              Error: HTTPError: 403 Client Error: Forbidden for url: https://www.ericsson.com/en/reports-and-papers/further-insights/cloud-ran-security
Ericsson     | FAILED     |     0.00 MB | SMO enabling intelligent RAN operations
              Error: HTTPError: 403 Client Error: Forbidden for url: https://www.ericsson.com/en/reports-and-papers/white-papers/smo-enabling-intelligent-ran-operations
Ericsson     | FAILED     |     0.00 MB | Cloud RAN
              Error: HTTPError: 403 Client Error: Forbidden for url: https://www.ericsson.com/en/ran/cloud
Ericsson     | FAILED     |     0.00 MB | 5G security for public and

#### **Observation**

The network-vendor layer acquired **7 technical references successfully**, comprising five Nokia and two Samsung sources. Automated retrieval of Ericsson and Cisco references was blocked by HTTP 403 responses from their public web endpoints.

**Recommendation:** Retain the successfully acquired vendor sources and defer blocked Ericsson/Cisco references rather than introducing additional acquisition complexity. Vendor material remains a supplementary layer to the substantially larger standards, Telco Common Corpus and open-source engineering collections.

### **Network Vendor Knowledge Base Verification**

In [11]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# NETWORK VENDOR KNOWLEDGE VERIFICATION
# =============================================================================

from pathlib import Path
import json
import shutil

VENDOR_ROOT = KB_ROOT / "vendor_network"


# =============================================================================
# SOURCE INSPECTION
# =============================================================================

def inspect_vendor_directory(path):
    """Return file count and total size for a vendor directory."""

    path = Path(path)

    if not path.exists():
        return 0, 0.0

    files = [
        file
        for file in path.rglob("*")
        if file.is_file()
    ]

    total_bytes = sum(
        file.stat().st_size
        for file in files
    )

    return (
        len(files),
        total_bytes / (1024 ** 3),
    )


# =============================================================================
# DASHBOARD
# =============================================================================

print("=" * 100)
print("NETWORK VENDOR KNOWLEDGE — VERIFICATION DASHBOARD")
print("=" * 100)

print(
    f"{'VENDOR':<15}"
    f"{'FILES':>10}"
    f"{'SIZE (GB)':>14}"
    f"  STATUS"
)

print("-" * 100)

total_files = 0
total_size_gb = 0.0

vendor_directories = [
    "ericsson",
    "nokia",
    "cisco",
    "samsung",
]

for vendor in vendor_directories:

    vendor_path = VENDOR_ROOT / vendor

    file_count, size_gb = inspect_vendor_directory(
        vendor_path
    )

    total_files += file_count
    total_size_gb += size_gb

    status = "READY" if file_count > 0 else "NOT FOUND"

    print(
        f"{vendor.capitalize():<15}"
        f"{file_count:>10,}"
        f"{size_gb:>14.3f}"
        f"  {status}"
    )


# =============================================================================
# REGISTRY VERIFICATION
# =============================================================================

registry_path = (
    VENDOR_ROOT
    / "vendor_source_registry.json"
)

registry_count = 0

if registry_path.exists():

    with open(
        registry_path,
        "r",
        encoding="utf-8",
    ) as file:

        registry = json.load(file)

    registry_count = len(registry)

else:

    registry = []


# =============================================================================
# STORAGE
# =============================================================================

total_disk, used_disk, free_disk = (
    shutil.disk_usage("/content")
)

print("-" * 100)

print(
    f"{'TOTAL VENDOR FILES':<25}"
    f"{total_files:>10,}"
    f"{total_size_gb:>14.3f} GB"
)

print(
    f"{'REGISTERED SOURCES':<25}"
    f"{registry_count:>10,}"
)

print(
    f"{'COLAB FREE STORAGE':<39}"
    f"{free_disk / (1024**3):>14.2f} GB"
)

print("=" * 100)

print("\nNetwork vendor verification completed.")

NETWORK VENDOR KNOWLEDGE — VERIFICATION DASHBOARD
VENDOR              FILES     SIZE (GB)  STATUS
----------------------------------------------------------------------------------------------------
Ericsson                0         0.000  NOT FOUND
Nokia                   5         0.001  READY
Cisco                   0         0.000  NOT FOUND
Samsung                 2         0.002  READY
----------------------------------------------------------------------------------------------------
TOTAL VENDOR FILES                7         0.004 GB
REGISTERED SOURCES               14
COLAB FREE STORAGE                              73.91 GB

Network vendor verification completed.


#### **Observation**

The network-vendor layer is partially complete, with **7 technical references acquired** from Nokia and Samsung. Ericsson and Cisco sources remain unavailable through automated retrieval, while **73.91 GB of Colab storage remains free**.

**Recommendation:** Retain the available vendor sources and proceed with the academic/textbook knowledge layer.

### **Cloud Platform Knowledge Base Acquisition**

In [12]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CLOUD PLATFORM KNOWLEDGE ACQUISITION
# =============================================================================

# Acquire a focused set of official telecom/cloud technical references from
# AWS, Microsoft Azure and Google Cloud.
#
# These sources complement the standards and network-vendor layers with
# practical cloud-native telecom deployment, orchestration and edge knowledge.

import json
import requests
from pathlib import Path

CLOUD_ROOT = KB_ROOT / "cloud_platform"

CLOUD_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# SOURCE REGISTRY
# =============================================================================

CLOUD_SOURCES = [

    # =========================================================================
    # AWS
    # =========================================================================
    {
        "provider": "AWS",
        "title": "AWS Telco Lens",
        "url": (
            "https://docs.aws.amazon.com/wellarchitected/latest/"
            "telco-lens/telco-lens.html"
        ),
        "filename": "aws_telco_lens.html",
        "topics": [
            "telecom",
            "5G",
            "cloud",
            "architecture",
            "operations",
            "resilience",
        ],
    },

    {
        "provider": "AWS",
        "title": "5G Core Deployment on AWS",
        "url": (
            "https://docs.aws.amazon.com/wellarchitected/latest/"
            "telco-lens/g-core-deployment-on-aws.html"
        ),
        "filename": "aws_5g_core_deployment.html",
        "topics": [
            "5G Core",
            "SBA",
            "CNF",
            "VNF",
            "cloud-native",
            "edge",
        ],
    },

    {
        "provider": "AWS",
        "title": "Automatic Lifecycle Management of Telco Network Functions",
        "url": (
            "https://docs.aws.amazon.com/wellarchitected/latest/"
            "telco-lens/automatic-lifecycle-management-of-telco-network-functions.html"
        ),
        "filename": "aws_telco_network_function_lifecycle.html",
        "topics": [
            "automation",
            "lifecycle management",
            "RAN",
            "5G Core",
            "network functions",
        ],
    },

    {
        "provider": "AWS",
        "title": "AWS Telco Lens Scenarios",
        "url": (
            "https://docs.aws.amazon.com/wellarchitected/latest/"
            "telco-lens/scenarios.html"
        ),
        "filename": "aws_telco_lens_scenarios.html",
        "topics": [
            "5G",
            "network automation",
            "NFV",
            "CNF",
            "orchestration",
        ],
    },


    # =========================================================================
    # MICROSOFT AZURE
    # =========================================================================
    {
        "provider": "Microsoft Azure",
        "title": "Azure Operator Service Manager",
        "url": (
            "https://learn.microsoft.com/en-us/azure/"
            "operator-service-manager/"
        ),
        "filename": "azure_operator_service_manager.html",
        "topics": [
            "CNF",
            "VNF",
            "orchestration",
            "lifecycle management",
            "hybrid cloud",
            "telecom",
        ],
    },

    {
        "provider": "Microsoft Azure",
        "title": "Azure Operator Service Manager Overview",
        "url": (
            "https://learn.microsoft.com/en-us/azure/"
            "operator-service-manager/azure-operator-service-manager-overview"
        ),
        "filename": "azure_operator_service_manager_overview.html",
        "topics": [
            "orchestration",
            "CNF",
            "VNF",
            "network functions",
            "automation",
        ],
    },

    {
        "provider": "Microsoft Azure",
        "title": "Azure Operator Nexus",
        "url": (
            "https://learn.microsoft.com/en-us/azure/operator-nexus/overview"
        ),
        "filename": "azure_operator_nexus.html",
        "topics": [
            "telco cloud",
            "hybrid cloud",
            "CNF",
            "VNF",
            "Kubernetes",
            "observability",
        ],
    },

    {
        "provider": "Microsoft Azure",
        "title": "Azure Operator Nexus Documentation",
        "url": (
            "https://learn.microsoft.com/en-us/azure/operator-nexus/"
        ),
        "filename": "azure_operator_nexus_documentation.html",
        "topics": [
            "telecom",
            "Kubernetes",
            "observability",
            "security",
            "edge",
        ],
    },


    # =========================================================================
    # GOOGLE CLOUD
    # =========================================================================
    {
        "provider": "Google Cloud",
        "title": "Google Cloud Telecommunications",
        "url": (
            "https://cloud.google.com/solutions/telecommunications"
        ),
        "filename": "google_cloud_telecommunications.html",
        "topics": [
            "telecom",
            "5G",
            "edge",
            "hybrid cloud",
            "AI",
        ],
    },

    {
        "provider": "Google Cloud",
        "title": "5G Edge Computing Study",
        "url": (
            "https://cloud.google.com/resources/"
            "telecom-5g-edge-enterprise-report"
        ),
        "filename": "google_cloud_5g_edge.html",
        "topics": [
            "5G",
            "edge",
            "CSP",
            "enterprise",
        ],
    },
]


# =============================================================================
# ACQUISITION
# =============================================================================

def download_cloud_source(source):

    provider_dir = (
        CLOUD_ROOT
        / source["provider"]
            .lower()
            .replace(" ", "_")
    )

    provider_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output_file = (
        provider_dir
        / source["filename"]
    )

    if (
        output_file.exists()
        and output_file.stat().st_size > 0
    ):
        return "EXISTING", output_file.stat().st_size

    try:

        response = requests.get(
            source["url"],
            timeout=120,
            headers={
                "User-Agent":
                    "Mozilla/5.0 "
                    "(compatible; Telecom-RAG-Research/1.0)"
            },
        )

        response.raise_for_status()

        output_file.write_bytes(
            response.content
        )

        return "DOWNLOADED", output_file.stat().st_size

    except Exception as exc:

        if output_file.exists():
            output_file.unlink()

        return (
            f"FAILED: {type(exc).__name__}",
            0,
        )


# =============================================================================
# DASHBOARD
# =============================================================================

results = []

print("=" * 100)
print("CLOUD PLATFORM KNOWLEDGE ACQUISITION")
print("=" * 100)

for source in CLOUD_SOURCES:

    status, size_bytes = download_cloud_source(
        source
    )

    results.append({
        "provider": source["provider"],
        "title": source["title"],
        "status": status,
        "size_mb": size_bytes / (1024 ** 2),
    })

    print(
        f"{source['provider']:<18} | "
        f"{status:<24} | "
        f"{size_bytes / (1024 ** 2):>8.2f} MB | "
        f"{source['title']}"
    )


# =============================================================================
# REGISTRY
# =============================================================================

registry_path = (
    CLOUD_ROOT
    / "cloud_source_registry.json"
)

with open(
    registry_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        CLOUD_SOURCES,
        file,
        indent=2,
        ensure_ascii=False,
    )


print("\n" + "=" * 100)
print("CLOUD PLATFORM ACQUISITION COMPLETED")
print("=" * 100)

successful = sum(
    result["status"] in {"DOWNLOADED", "EXISTING"}
    for result in results
)

total_mb = sum(
    result["size_mb"]
    for result in results
)

print(f"Sources defined    : {len(CLOUD_SOURCES)}")
print(f"Sources available  : {successful}")
print(f"Total acquired     : {total_mb:.2f} MB")
print(f"Registry           : {registry_path}")

CLOUD PLATFORM KNOWLEDGE ACQUISITION
AWS                | DOWNLOADED               |     0.02 MB | AWS Telco Lens
AWS                | DOWNLOADED               |     0.02 MB | 5G Core Deployment on AWS
AWS                | DOWNLOADED               |     0.01 MB | Automatic Lifecycle Management of Telco Network Functions
AWS                | DOWNLOADED               |     0.01 MB | AWS Telco Lens Scenarios
Microsoft Azure    | DOWNLOADED               |     0.04 MB | Azure Operator Service Manager
Microsoft Azure    | DOWNLOADED               |     0.05 MB | Azure Operator Service Manager Overview
Microsoft Azure    | DOWNLOADED               |     0.05 MB | Azure Operator Nexus
Microsoft Azure    | DOWNLOADED               |     0.03 MB | Azure Operator Nexus Documentation
Google Cloud       | DOWNLOADED               |     2.38 MB | Google Cloud Telecommunications
Google Cloud       | DOWNLOADED               |     2.48 MB | 5G Edge Computing Study

CLOUD PLATFORM ACQUISITION COMPLETE

### **Cloud Platform Knowledge Base Verification**

In [13]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 4.3.X — CLOUD PLATFORM KNOWLEDGE VERIFICATION
# =============================================================================

from pathlib import Path
import shutil


# =============================================================================
# CLOUD PLATFORM KNOWLEDGE BASE
# =============================================================================

CLOUD_ROOT = KB_ROOT / "cloud_platform"

CLOUD_SOURCES = {
    "AWS": CLOUD_ROOT / "aws",
    "Microsoft Azure": CLOUD_ROOT / "microsoft_azure",
    "Google Cloud": CLOUD_ROOT / "google_cloud",
}


# =============================================================================
# SOURCE INSPECTION
# =============================================================================

def inspect_source(path):
    """Return file count and total size for a source directory."""

    path = Path(path)

    if not path.exists():
        return 0, 0.0

    files = [
        file
        for file in path.rglob("*")
        if file.is_file()
    ]

    total_bytes = sum(
        file.stat().st_size
        for file in files
    )

    return (
        len(files),
        total_bytes / (1024 ** 3),
    )


# =============================================================================
# VERIFICATION DASHBOARD
# =============================================================================

print("=" * 100)
print("CLOUD PLATFORM KNOWLEDGE — VERIFICATION DASHBOARD")
print("=" * 100)

print(
    f"{'PROVIDER':<22}"
    f"{'STATUS':<12}"
    f"{'FILES':>12}"
    f"{'SIZE (GB)':>14}"
)

print("-" * 100)

total_files = 0
total_size_gb = 0.0

for provider, source_path in CLOUD_SOURCES.items():

    file_count, size_gb = inspect_source(
        source_path
    )

    total_files += file_count
    total_size_gb += size_gb

    status = (
        "READY"
        if file_count > 0
        else "NOT FOUND"
    )

    print(
        f"{provider:<22}"
        f"{status:<12}"
        f"{file_count:>12,}"
        f"{size_gb:>14.3f}"
    )


# =============================================================================
# STORAGE
# =============================================================================

total_disk, used_disk, free_disk = (
    shutil.disk_usage("/content")
)

print("-" * 100)

print(
    f"{'TOTAL CLOUD SOURCES':<34}"
    f"{total_files:>12,}"
    f"{total_size_gb:>14.3f} GB"
)

print(
    f"{'COLAB FREE STORAGE':<48}"
    f"{free_disk / (1024**3):>14.2f} GB"
)

print("=" * 100)

print("\nCloud platform verification completed.")

CLOUD PLATFORM KNOWLEDGE — VERIFICATION DASHBOARD
PROVIDER              STATUS             FILES     SIZE (GB)
----------------------------------------------------------------------------------------------------
AWS                   READY                  4         0.000
Microsoft Azure       READY                  4         0.000
Google Cloud          READY                  2         0.005
----------------------------------------------------------------------------------------------------
TOTAL CLOUD SOURCES                         10         0.005 GB
COLAB FREE STORAGE                                       73.91 GB

Cloud platform verification completed.


#### **Observation**

The cloud-platform layer was successfully acquired and verified, comprising **10 technical references across AWS, Microsoft Azure and Google Cloud** with a negligible **0.005 GB footprint**. Colab retains **73.91 GB of free storage**.

**Recommendation:** Retain the cloud-platform layer and proceed with the remaining **academic/textbook knowledge acquisition**.

### **Open Cloud-Native Infrastructure Knowledge Base Acquisition**

In [14]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# OPEN CLOUD-NATIVE KNOWLEDGE ACQUISITION
# =============================================================================

# Acquire official documentation repositories for major cloud-native projects.
# Documentation is retained for RAG; source code, binaries and build artifacts
# will be excluded during post-processing.

from pathlib import Path
import subprocess

CLOUD_NATIVE_ROOT = (
    KB_ROOT / "cloud_native"
)

CLOUD_NATIVE_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# DOCUMENTATION REPOSITORIES
# =============================================================================

CLOUD_NATIVE_REPOS = {
    "kubernetes":
        "https://github.com/kubernetes/website.git",

    "opentelemetry":
        "https://github.com/open-telemetry/opentelemetry.io.git",

    "opentelemetry_specification":
        "https://github.com/open-telemetry/opentelemetry-specification.git",

    "prometheus":
        "https://github.com/prometheus/docs.git",

    "helm":
        "https://github.com/helm/helm-www.git",

    "istio":
        "https://github.com/istio/istio.io.git",
}


# =============================================================================
# ACQUISITION
# =============================================================================

print("=" * 100)
print("OPEN CLOUD-NATIVE KNOWLEDGE ACQUISITION")
print("=" * 100)

results = []

for name, repo_url in CLOUD_NATIVE_REPOS.items():

    destination = (
        CLOUD_NATIVE_ROOT / name
    )

    if destination.exists():

        print(
            f"\n✓ {name:<30} ALREADY PRESENT"
        )

        results.append({
            "project": name,
            "status": "EXISTING",
        })

        continue

    print(
        f"\n▶ Acquiring {name}..."
    )

    result = subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            repo_url,
            str(destination),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode == 0:

        print(
            f"✓ {name:<30} ACQUIRED"
        )

        results.append({
            "project": name,
            "status": "ACQUIRED",
        })

    else:

        print(
            f"✗ {name:<30} FAILED"
        )

        print(
            result.stderr[-500:]
        )

        results.append({
            "project": name,
            "status": "FAILED",
        })


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("OPEN CLOUD-NATIVE ACQUISITION SUMMARY")
print("=" * 100)

for result in results:

    print(
        f"{result['project']:<35}"
        f"{result['status']}"
    )

print("\nCloud-native acquisition completed.")

OPEN CLOUD-NATIVE KNOWLEDGE ACQUISITION

▶ Acquiring kubernetes...
✓ kubernetes                     ACQUIRED

▶ Acquiring opentelemetry...
✓ opentelemetry                  ACQUIRED

▶ Acquiring opentelemetry_specification...
✓ opentelemetry_specification    ACQUIRED

▶ Acquiring prometheus...
✓ prometheus                     ACQUIRED

▶ Acquiring helm...
✓ helm                           ACQUIRED

▶ Acquiring istio...
✓ istio                          ACQUIRED

OPEN CLOUD-NATIVE ACQUISITION SUMMARY
kubernetes                         ACQUIRED
opentelemetry                      ACQUIRED
opentelemetry_specification        ACQUIRED
prometheus                         ACQUIRED
helm                               ACQUIRED
istio                              ACQUIRED

Cloud-native acquisition completed.


### **Open Cloud-Native Infrastructure Knowledge Base Verification**

In [15]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CLOUD-NATIVE KNOWLEDGE VERIFICATION
# =============================================================================

from pathlib import Path
import shutil

CLOUD_NATIVE_ROOT = KB_ROOT / "cloud_native"

CLOUD_NATIVE_SOURCES = {
    "Kubernetes": CLOUD_NATIVE_ROOT / "kubernetes",
    "OpenTelemetry": CLOUD_NATIVE_ROOT / "opentelemetry",
    "OTel Specification": CLOUD_NATIVE_ROOT / "opentelemetry_specification",
    "Prometheus": CLOUD_NATIVE_ROOT / "prometheus",
    "Helm": CLOUD_NATIVE_ROOT / "helm",
    "Istio": CLOUD_NATIVE_ROOT / "istio",
}


# =============================================================================
# SOURCE INSPECTION
# =============================================================================

def inspect_source(path):
    """Return file count and total size for a source directory."""

    path = Path(path)

    if not path.exists():
        return 0, 0.0

    files = [
        file
        for file in path.rglob("*")
        if file.is_file()
    ]

    total_bytes = sum(
        file.stat().st_size
        for file in files
    )

    return (
        len(files),
        total_bytes / (1024 ** 3),
    )


# =============================================================================
# VERIFICATION DASHBOARD
# =============================================================================

print("=" * 100)
print("CLOUD-NATIVE KNOWLEDGE — VERIFICATION DASHBOARD")
print("=" * 100)

print(
    f"{'PROJECT':<25}"
    f"{'STATUS':<12}"
    f"{'FILES':>12}"
    f"{'SIZE (GB)':>14}"
)

print("-" * 100)

total_files = 0
total_size_gb = 0.0

for project, source_path in CLOUD_NATIVE_SOURCES.items():

    file_count, size_gb = inspect_source(
        source_path
    )

    total_files += file_count
    total_size_gb += size_gb

    status = (
        "READY"
        if file_count > 0
        else "NOT FOUND"
    )

    print(
        f"{project:<25}"
        f"{status:<12}"
        f"{file_count:>12,}"
        f"{size_gb:>14.3f}"
    )


# =============================================================================
# COLAB STORAGE
# =============================================================================

total_disk, used_disk, free_disk = (
    shutil.disk_usage("/content")
)

print("-" * 100)

print(
    f"{'CLOUD-NATIVE TOTAL':<37}"
    f"{total_files:>12,}"
    f"{total_size_gb:>14.3f} GB"
)

print(
    f"{'COLAB FREE STORAGE':<51}"
    f"{free_disk / (1024**3):>14.2f} GB"
)

print("=" * 100)

print("\nCloud-native knowledge verification completed.")

CLOUD-NATIVE KNOWLEDGE — VERIFICATION DASHBOARD
PROJECT                  STATUS             FILES     SIZE (GB)
----------------------------------------------------------------------------------------------------
Kubernetes               READY             13,224         0.732
OpenTelemetry            READY              4,016         0.156
OTel Specification       READY                376         0.032
Prometheus               READY                372         0.043
Helm                     READY              2,245         0.037
Istio                    READY            108,243         7.226
----------------------------------------------------------------------------------------------------
CLOUD-NATIVE TOTAL                        128,476         8.226 GB
COLAB FREE STORAGE                                          65.09 GB

Cloud-native knowledge verification completed.


#### **Observation**

The cloud-native layer was successfully acquired and verified, comprising **128,476 files across Kubernetes, OpenTelemetry, Prometheus, Helm and Istio**, with a combined footprint of **8.226 GB**. Colab retains **65.09 GB of free storage**.

**Recommendation:** Retain the cloud-native corpus and proceed to the **academic/textbook knowledge layer**. Given Istio's size, its documentation should later be filtered to documentation content before chunking and embedding.

### **Academic / Textbook Knowledge Base Acquisition**

In [16]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# ACADEMIC / TEXTBOOK KNOWLEDGE ACQUISITION
# =============================================================================

# Acquire open-access academic and textbook material relevant to telecom,
# wireless communications, networking and emerging 5G/6G technologies.
#
# Sources are kept separate from standards, vendors and cloud-native material
# so provenance can be preserved during RAG processing.

import json
import re
import shutil
import subprocess
from pathlib import Path

import requests
from bs4 import BeautifulSoup


# =============================================================================
# KNOWLEDGE-BASE STRUCTURE
# =============================================================================

ACADEMIC_ROOT = (
    KB_ROOT / "academic"
)

ACADEMIC_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 1. OPEN TEXTBOOK
# =============================================================================

print("=" * 100)
print("ACADEMIC / TEXTBOOK KNOWLEDGE ACQUISITION")
print("=" * 100)

TEXTBOOK_ROOT = (
    ACADEMIC_ROOT / "textbooks"
)

TEXTBOOK_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

SYSTEMS_APPROACH_DEST = (
    TEXTBOOK_ROOT / "computer_networks_systems_approach"
)

if not SYSTEMS_APPROACH_DEST.exists():

    print("\n▶ Acquiring Computer Networks: A Systems Approach...")

    result = subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/SystemsApproach/book.git",
            str(SYSTEMS_APPROACH_DEST),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    if result.returncode == 0:
        print("✓ Systems Approach acquired.")
    else:
        print("✗ Systems Approach acquisition failed.")
        print(result.stderr[-500:])

else:
    print(
        "\n✓ Systems Approach already present."
    )


# =============================================================================
# 2. MIT OPENCOURSEWARE
# =============================================================================

MIT_ROOT = (
    ACADEMIC_ROOT / "mit_ocw"
)

MIT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

MIT_COURSES = {
    "principles_wireless_communications": (
        "https://ocw.mit.edu/courses/"
        "6-452-principles-of-wireless-communications-spring-2006/"
    ),

    "data_communication_networks": (
        "https://ocw.mit.edu/courses/"
        "6-263j-data-communication-networks-fall-2002/"
    ),
}


def download_mit_pdfs(course_name, course_url):
    """
    Discover PDF lecture materials from an MIT OCW course page and download
    the linked PDFs.
    """

    course_dir = (
        MIT_ROOT / course_name
    )

    course_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        f"\n▶ Inspecting MIT OCW: {course_name}"
    )

    response = requests.get(
        course_url,
        timeout=120,
        headers={
            "User-Agent":
                "Mozilla/5.0 "
                "(compatible; Telecom-RAG-Research/1.0)"
        },
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    pdf_links = {}

    for anchor in soup.find_all("a"):

        href = anchor.get("href")

        if not href:
            continue

        if ".pdf" not in href.lower():
            continue

        if href.startswith("/"):
            href = (
                "https://ocw.mit.edu"
                + href
            )

        pdf_links[href] = (
            anchor.get_text(
                " ",
                strip=True
            )
            or Path(href).name
        )

    print(
        f"PDF resources discovered : "
        f"{len(pdf_links)}"
    )

    downloaded = 0

    for pdf_url, label in pdf_links.items():

        filename = Path(
            pdf_url.split("?")[0]
        ).name

        if not filename.lower().endswith(".pdf"):
            continue

        output_file = (
            course_dir / filename
        )

        if (
            output_file.exists()
            and output_file.stat().st_size > 0
        ):
            continue

        try:

            pdf_response = requests.get(
                pdf_url,
                timeout=180,
                headers={
                    "User-Agent":
                        "Mozilla/5.0 "
                        "(compatible; Telecom-RAG-Research/1.0)"
                },
            )

            pdf_response.raise_for_status()

            output_file.write_bytes(
                pdf_response.content
            )

            downloaded += 1

        except Exception as exc:

            print(
                f"  ! Failed: {filename} — "
                f"{type(exc).__name__}"
            )

    print(
        f"✓ {course_name}: "
        f"{downloaded} new PDFs acquired."
    )


for course_name, course_url in MIT_COURSES.items():

    try:
        download_mit_pdfs(
            course_name,
            course_url,
        )

    except Exception as exc:

        print(
            f"✗ MIT course failed: "
            f"{course_name} — "
            f"{type(exc).__name__}: {exc}"
        )


# =============================================================================
# 3. OPEN RESEARCH — ARXIV
# =============================================================================

ARXIV_ROOT = (
    ACADEMIC_ROOT / "arxiv"
)

ARXIV_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

ARXIV_PAPERS = {
    "1502.07228":
        "5G mmWave Communications Survey",

    "2004.08549":
        "6G Wireless Communications Survey",

    "2009.04943":
        "AI for 5G Wireless Systems",

    "2104.08834":
        "5G Network Slicing Testbeds",

    "2111.13754":
        "Open RAN — What O-RAN Can and Cannot Do",

    "2405.03555":
        "Comprehensive O-RAN Survey",
}


print("\n" + "=" * 100)
print("ACQUIRING OPEN RESEARCH")
print("=" * 100)


for arxiv_id, title in ARXIV_PAPERS.items():

    output_file = (
        ARXIV_ROOT
        / f"arxiv_{arxiv_id}.pdf"
    )

    print(
        f"\n▶ {arxiv_id} — {title}"
    )

    if (
        output_file.exists()
        and output_file.stat().st_size > 0
    ):

        print("  Already present.")
        continue

    pdf_url = (
        f"https://arxiv.org/pdf/{arxiv_id}"
    )

    try:

        response = requests.get(
            pdf_url,
            timeout=180,
            headers={
                "User-Agent":
                    "Mozilla/5.0 "
                    "(compatible; Telecom-RAG-Research/1.0)"
            },
        )

        response.raise_for_status()

        output_file.write_bytes(
            response.content
        )

        print(
            f"  ✓ Downloaded "
            f"({output_file.stat().st_size / (1024**2):.2f} MB)"
        )

    except Exception as exc:

        print(
            f"  ✗ Failed — "
            f"{type(exc).__name__}: {exc}"
        )


# =============================================================================
# REGISTRY
# =============================================================================

registry = {
    "textbooks": {
        "computer_networks_systems_approach": {
            "source":
                "https://github.com/SystemsApproach/book",
            "license":
                "CC BY 4.0",
        },
    },

    "mit_ocw": MIT_COURSES,

    "arxiv": ARXIV_PAPERS,
}

registry_path = (
    ACADEMIC_ROOT
    / "academic_source_registry.json"
)

with open(
    registry_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        registry,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# STORAGE SUMMARY
# =============================================================================

total, used, free = shutil.disk_usage(
    "/content"
)

print("\n" + "=" * 100)
print("ACADEMIC / TEXTBOOK ACQUISITION COMPLETED")
print("=" * 100)

print(
    f"Academic corpus size : "
    f"{sum(
        file.stat().st_size
        for file in ACADEMIC_ROOT.rglob('*')
        if file.is_file()
    ) / (1024**3):.3f} GB"
)

print(
    f"Colab free storage   : "
    f"{free / (1024**3):.2f} GB"
)

print(
    f"Registry             : "
    f"{registry_path}"
)

ACADEMIC / TEXTBOOK KNOWLEDGE ACQUISITION

▶ Acquiring Computer Networks: A Systems Approach...
✓ Systems Approach acquired.

▶ Inspecting MIT OCW: principles_wireless_communications
PDF resources discovered : 0
✓ principles_wireless_communications: 0 new PDFs acquired.

▶ Inspecting MIT OCW: data_communication_networks
PDF resources discovered : 0
✓ data_communication_networks: 0 new PDFs acquired.

ACQUIRING OPEN RESEARCH

▶ 1502.07228 — 5G mmWave Communications Survey
  ✓ Downloaded (0.76 MB)

▶ 2004.08549 — 6G Wireless Communications Survey
  ✓ Downloaded (5.06 MB)

▶ 2009.04943 — AI for 5G Wireless Systems
  ✓ Downloaded (0.63 MB)

▶ 2104.08834 — 5G Network Slicing Testbeds
  ✓ Downloaded (3.97 MB)

▶ 2111.13754 — Open RAN — What O-RAN Can and Cannot Do
  ✓ Downloaded (1.16 MB)

▶ 2405.03555 — Comprehensive O-RAN Survey
  ✓ Downloaded (0.80 MB)

ACADEMIC / TEXTBOOK ACQUISITION COMPLETED
Academic corpus size : 0.384 GB
Colab free storage   : 64.70 GB
Registry             : /content

#### **Observation**

The academic/textbook layer was partially acquired successfully, including the **Computer Networks: A Systems Approach** open textbook and **six open-access telecom research papers** covering 5G/mmWave, 6G, AI, network slicing and O-RAN. MIT OCW PDF discovery returned no downloadable resources through the current method.

The academic layer currently occupies **0.384 GB**, with **64.70 GB of Colab storage remaining**.

### **Academic / Textbook Knowledge Base Verification**

In [17]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# ACADEMIC / TEXTBOOK KNOWLEDGE VERIFICATION
# =============================================================================

from pathlib import Path
import shutil


# =============================================================================
# ACADEMIC KNOWLEDGE BASE
# =============================================================================

ACADEMIC_ROOT = KB_ROOT / "academic"

ACADEMIC_SOURCES = {
    "Textbooks":
        ACADEMIC_ROOT / "textbooks",

    "MIT OpenCourseWare":
        ACADEMIC_ROOT / "mit_ocw",

    "Open Research":
        ACADEMIC_ROOT / "arxiv",
}


# =============================================================================
# SOURCE INSPECTION
# =============================================================================

def inspect_source(path):
    """Return file count and total size for a source directory."""

    path = Path(path)

    if not path.exists():
        return 0, 0.0

    files = [
        file
        for file in path.rglob("*")
        if file.is_file()
    ]

    total_bytes = sum(
        file.stat().st_size
        for file in files
    )

    return (
        len(files),
        total_bytes / (1024 ** 3),
    )


# =============================================================================
# VERIFICATION DASHBOARD
# =============================================================================

print("=" * 100)
print("ACADEMIC / TEXTBOOK KNOWLEDGE — VERIFICATION DASHBOARD")
print("=" * 100)

print(
    f"{'SOURCE':<25}"
    f"{'STATUS':<12}"
    f"{'FILES':>12}"
    f"{'SIZE (GB)':>14}"
)

print("-" * 100)

total_files = 0
total_size_gb = 0.0

for source_name, source_path in ACADEMIC_SOURCES.items():

    file_count, size_gb = inspect_source(
        source_path
    )

    total_files += file_count
    total_size_gb += size_gb

    status = (
        "READY"
        if file_count > 0
        else "NOT FOUND"
    )

    print(
        f"{source_name:<25}"
        f"{status:<12}"
        f"{file_count:>12,}"
        f"{size_gb:>14.3f}"
    )


# =============================================================================
# REGISTRY CHECK
# =============================================================================

registry_path = (
    ACADEMIC_ROOT
    / "academic_source_registry.json"
)

print("-" * 100)

print(
    f"{'ACADEMIC TOTAL':<37}"
    f"{total_files:>12,}"
    f"{total_size_gb:>14.3f} GB"
)

print(
    f"{'REGISTRY':<37}"
    f"{'PRESENT' if registry_path.exists() else 'MISSING'}"
)


# =============================================================================
# COLAB STORAGE
# =============================================================================

total_disk, used_disk, free_disk = (
    shutil.disk_usage("/content")
)

print(
    f"{'COLAB FREE STORAGE':<51}"
    f"{free_disk / (1024**3):>14.2f} GB"
)

print("=" * 100)

print("\nAcademic / textbook verification completed.")

ACADEMIC / TEXTBOOK KNOWLEDGE — VERIFICATION DASHBOARD
SOURCE                   STATUS             FILES     SIZE (GB)
----------------------------------------------------------------------------------------------------
Textbooks                READY                682         0.372
MIT OpenCourseWare       NOT FOUND              0         0.000
Open Research            READY                  6         0.012
----------------------------------------------------------------------------------------------------
ACADEMIC TOTAL                                688         0.384 GB
REGISTRY                             PRESENT
COLAB FREE STORAGE                                          64.70 GB

Academic / textbook verification completed.


#### **Observation**

The academic/textbook layer was successfully verified with **688 files totaling 0.384 GB**, comprising the open textbook and six research papers. MIT OpenCourseWare resources were not acquired through the current discovery method. **64.70 GB of Colab storage remains available**.

**Recommendation:** Retain the acquired academic layer and proceed to final corpus inventory before beginning document processing and normalisation.

### **Knowledge Base Inventory**

In [18]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# FINAL KNOWLEDGE BASE INVENTORY
# =============================================================================

from pathlib import Path
import shutil
from collections import defaultdict

KB_ROOT = Path("/content/telecom_knowledge_base")


# =============================================================================
# DIRECTORY DEFINITIONS
# =============================================================================

KNOWLEDGE_SOURCES = {
    "Standards / GSMA Ecosystem":
        KB_ROOT / "standards",

    "Telco Common Corpus":
        KB_ROOT / "GSMA-Telco-Common-Corpus",

    "Open-Source Telecom":
        KB_ROOT / "open_source",

    "Cloud-Native":
        KB_ROOT / "cloud_native",

    "Network Vendors":
        KB_ROOT / "vendor_network",

    "Cloud Platforms":
        KB_ROOT / "cloud_platform",

    "Academic / Textbooks":
        KB_ROOT / "academic",

    "Other Vendor Sources":
        KB_ROOT / "vendor",
}


# =============================================================================
# SOURCE INSPECTION
# =============================================================================

def inspect_source(path):

    path = Path(path)

    if not path.exists():
        return 0, 0.0, defaultdict(int)

    files = [
        file
        for file in path.rglob("*")
        if file.is_file()
    ]

    total_bytes = sum(
        file.stat().st_size
        for file in files
    )

    extensions = defaultdict(int)

    for file in files:
        extension = (
            file.suffix.lower()
            if file.suffix
            else "[no extension]"
        )

        extensions[extension] += 1

    return (
        len(files),
        total_bytes / (1024 ** 3),
        extensions,
    )


# =============================================================================
# INVENTORY DASHBOARD
# =============================================================================

print("=" * 110)
print("TELECOM RAG — FINAL KNOWLEDGE BASE INVENTORY")
print("=" * 110)

print(
    f"{'SOURCE CATEGORY':<30}"
    f"{'STATUS':<12}"
    f"{'FILES':>12}"
    f"{'SIZE (GB)':>14}"
)

print("-" * 110)

grand_total_files = 0
grand_total_gb = 0.0

inventory = {}

for category, path in KNOWLEDGE_SOURCES.items():

    file_count, size_gb, extensions = inspect_source(
        path
    )

    status = (
        "READY"
        if file_count > 0
        else "NOT FOUND"
    )

    grand_total_files += file_count
    grand_total_gb += size_gb

    inventory[category] = {
        "path": str(path),
        "status": status,
        "files": file_count,
        "size_gb": size_gb,
        "extensions": dict(extensions),
    }

    print(
        f"{category:<30}"
        f"{status:<12}"
        f"{file_count:>12,}"
        f"{size_gb:>14.3f}"
    )


# =============================================================================
# STORAGE
# =============================================================================

total_disk, used_disk, free_disk = (
    shutil.disk_usage("/content")
)

print("-" * 110)

print(
    f"{'TOTAL KNOWLEDGE BASE':<42}"
    f"{grand_total_files:>12,}"
    f"{grand_total_gb:>14.3f} GB"
)

print(
    f"{'COLAB TOTAL STORAGE':<56}"
    f"{total_disk / (1024**3):>14.2f} GB"
)

print(
    f"{'COLAB USED STORAGE':<56}"
    f"{used_disk / (1024**3):>14.2f} GB"
)

print(
    f"{'COLAB FREE STORAGE':<56}"
    f"{free_disk / (1024**3):>14.2f} GB"
)

print("=" * 110)


# =============================================================================
# EXTENSION SUMMARY
# =============================================================================

extension_totals = defaultdict(int)

for category_data in inventory.values():

    for extension, count in category_data["extensions"].items():
        extension_totals[extension] += count

print("\nFile Type Summary")
print("-" * 60)

for extension, count in sorted(
    extension_totals.items(),
    key=lambda item: item[1],
    reverse=True,
):

    print(
        f"{extension:<20} : {count:>12,}"
    )


print("\nFinal knowledge-base inventory completed.")

TELECOM RAG — FINAL KNOWLEDGE BASE INVENTORY
SOURCE CATEGORY               STATUS             FILES     SIZE (GB)
--------------------------------------------------------------------------------------------------------------
Standards / GSMA Ecosystem    READY             48,429         3.872
Telco Common Corpus           READY                309         8.838
Open-Source Telecom           READY             12,801         0.438
Cloud-Native                  READY            128,476         8.226
Network Vendors               READY                  8         0.004
Cloud Platforms               READY                 11         0.005
Academic / Textbooks          READY                689         0.384
Other Vendor Sources          NOT FOUND              0         0.000
--------------------------------------------------------------------------------------------------------------
TOTAL KNOWLEDGE BASE                           190,723        21.767 GB
COLAB TOTAL STORAGE                     

#### **Observation**

- The final knowledge base now contains **190,723 files across 7 active source categories**, occupying **21.767 GB**, with **64.70 GB of Colab storage remaining**.
- The corpus is sufficiently broad for the RAG experiment, spanning standards, Telco Common Corpus, open-source telecom, cloud-native, vendors, cloud platforms and academic material.

- The file-type inventory also shows substantial non-document content from Git repositories, including source code, images, build artifacts, locks, metadata and configuration files.
- These should **not** enter the RAG corpus.

**Recommendation:**
- Freeze acquisition and proceed to **Document Content Selection & Extraction**, where we separate knowledge-bearing documents from repository artifacts and apply format-specific extraction before chunking.

# **Document Chunking and Preprocessing**

## **Extract and Normalize Text**

### **Document Eligibility Inventory**

In [20]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# DOCUMENT ELIGIBILITY & CONTENT INVENTORY
# =============================================================================

# Build a source-aware inventory of knowledge-bearing content.
#
# IMPORTANT:
# - Telco Common Corpus is handled separately as a streaming Parquet source.
# - Other sources are classified by document format.
# - Repository artefacts, source code, images, binaries and build files
#   are excluded from the document-extraction pipeline.
#
# No document contents are loaded in this cell.

from pathlib import Path
from collections import Counter

# ---------------------------------------------------------------------------
# Knowledge-base root
# ---------------------------------------------------------------------------

KB_ROOT = Path(
    "/content/telecom_knowledge_base"
)

# ---------------------------------------------------------------------------
# TCC location
# ---------------------------------------------------------------------------

TCC_ROOT = (
    KB_ROOT / "GSMA-Telco-Common-Corpus"
)


# =============================================================================
# ELIGIBLE DOCUMENT FORMATS
# =============================================================================

ELIGIBLE_EXTENSIONS = {
    ".md",
    ".mdx",
    ".html",
    ".htm",
    ".txt",
    ".rst",
    ".pdf",
    ".docx",
    ".ppt",
    ".pptx",
}


# =============================================================================
# EXCLUDED DIRECTORIES
# =============================================================================

EXCLUDED_DIRECTORIES = {
    ".git",
    ".github",
    ".gitlab",
    ".svn",
    "node_modules",
    "__pycache__",
    "build",
    "dist",
    "target",
    "venv",
    ".venv",
    "coverage",
    "site",
    "_site",
}


# =============================================================================
# EXCLUDED FILE NAMES
# =============================================================================

EXCLUDED_FILENAMES = {
    "package-lock.json",
    "yarn.lock",
    "pnpm-lock.yaml",
}


# =============================================================================
# SOURCE CATEGORY DETECTION
# =============================================================================

def classify_source(path):
    """
    Determine the broad knowledge category from the file path.
    """

    parts = {
        part.lower()
        for part in Path(path).parts
    }

    if "standards" in parts:
        return "standards"

    if "open_source" in parts:
        return "open_source"

    if "cloud_native" in parts:
        return "cloud_native"

    if "vendor_network" in parts:
        return "vendor_network"

    if "cloud_platform" in parts:
        return "cloud_platform"

    if "academic" in parts:
        return "academic"

    if "vendor" in parts:
        return "vendor"

    return "other"


# =============================================================================
# TCC INSPECTION
# =============================================================================

tcc_parquet_files = []

if TCC_ROOT.exists():

    tcc_parquet_files = sorted(
        TCC_ROOT.rglob("*.parquet")
    )

tcc_file_count = len(tcc_parquet_files)

tcc_size_bytes = sum(
    file.stat().st_size
    for file in tcc_parquet_files
)


# =============================================================================
# DOCUMENT INVENTORY
# =============================================================================

eligible_files = []
excluded_files = []

eligible_by_source = Counter()
eligible_by_extension = Counter()
excluded_by_extension = Counter()


# =============================================================================
# SCAN NON-TCC SOURCES
# =============================================================================

for file in KB_ROOT.rglob("*"):

    if not file.is_file():
        continue

    # ---------------------------------------------------------------
    # TCC is handled separately as a streaming dataset.
    # ---------------------------------------------------------------

    try:
        file.relative_to(TCC_ROOT)
        continue
    except ValueError:
        pass

    # ---------------------------------------------------------------
    # Skip known repository/build directories.
    # ---------------------------------------------------------------

    if any(
        directory in EXCLUDED_DIRECTORIES
        for directory in file.parts
    ):
        excluded_files.append(file)
        continue

    # ---------------------------------------------------------------
    # Skip specific known artefact filenames.
    # ---------------------------------------------------------------

    if file.name in EXCLUDED_FILENAMES:
        excluded_files.append(file)
        continue

    extension = file.suffix.lower()

    # ---------------------------------------------------------------
    # Document eligibility
    # ---------------------------------------------------------------

    if extension in ELIGIBLE_EXTENSIONS:

        source = classify_source(file)

        eligible_files.append(file)

        eligible_by_source[source] += 1
        eligible_by_extension[extension] += 1

    else:

        excluded_files.append(file)
        excluded_by_extension[
            extension or "[no extension]"
        ] += 1


# =============================================================================
# DASHBOARD
# =============================================================================

print("=" * 100)
print("DOCUMENT ELIGIBILITY & CONTENT INVENTORY")
print("=" * 100)


# ---------------------------------------------------------------------------
# TCC
# ---------------------------------------------------------------------------

print("\nTCC STREAMING SOURCE")
print("-" * 60)

print(
    f"Status              : "
    f"{'READY' if tcc_file_count > 0 else 'NOT FOUND'}"
)

print(
    f"Parquet files       : "
    f"{tcc_file_count:,}"
)

print(
    f"Source size         : "
    f"{tcc_size_bytes / (1024**3):.3f} GB"
)

print(
    "Processing mode     : "
    "STREAMING / PARQUET"
)


# ---------------------------------------------------------------------------
# Other document sources
# ---------------------------------------------------------------------------

print("\nDOCUMENT FILE SOURCES")
print("-" * 60)

print(
    f"{'CATEGORY':<25}"
    f"{'ELIGIBLE FILES':>18}"
)

print("-" * 50)

for source, count in sorted(
    eligible_by_source.items(),
    key=lambda item: item[1],
    reverse=True,
):

    print(
        f"{source:<25}"
        f"{count:>18,}"
    )

print("-" * 50)

print(
    f"{'TOTAL DOCUMENT FILES':<25}"
    f"{len(eligible_files):>18,}"
)

print(
    f"{'EXCLUDED FILES':<25}"
    f"{len(excluded_files):>18,}"
)


# =============================================================================
# ELIGIBLE FORMAT SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("ELIGIBLE DOCUMENT FORMATS")
print("=" * 100)

print(
    f"{'FORMAT':<15}"
    f"{'FILES':>15}"
)

print("-" * 35)

for extension, count in sorted(
    eligible_by_extension.items(),
    key=lambda item: item[1],
    reverse=True,
):

    print(
        f"{extension:<15}"
        f"{count:>15,}"
    )


# =============================================================================
# TCC + DOCUMENT TOTAL
# =============================================================================

print("\n" + "=" * 100)
print("PROCESSING INVENTORY SUMMARY")
print("=" * 100)

print(
    f"Document files for extraction : "
    f"{len(eligible_files):,}"
)

print(
    f"TCC Parquet files             : "
    f"{tcc_file_count:,}"
)

print(
    f"TCC processing model          : "
    f"Streaming"
)

print(
    f"Total source categories       : "
    f"{len(eligible_by_source):,}"
)

print("=" * 100)

print(
    "\nEligibility inventory completed. "
    "No document contents were loaded."
)

DOCUMENT ELIGIBILITY & CONTENT INVENTORY

TCC STREAMING SOURCE
------------------------------------------------------------
Status              : READY
Parquet files       : 100
Source size         : 8.838 GB
Processing mode     : STREAMING / PARQUET

DOCUMENT FILE SOURCES
------------------------------------------------------------
CATEGORY                     ELIGIBLE FILES
--------------------------------------------------
cloud_native                         88,828
standards                            16,135
open_source                             404
academic                                105
cloud_platform                           10
vendor_network                            7
--------------------------------------------------
TOTAL DOCUMENT FILES                105,489
EXCLUDED FILES                       84,925

ELIGIBLE DOCUMENT FORMATS
FORMAT                   FILES
-----------------------------------
.html                   72,861
.md                     31,361
.docx      

#### **Observation**

- The corpus architecture: **105,489 document files** are eligible for format-specific extraction, while the **8.838 GB TCC corpus remains a separate 100-file Parquet source for streaming processing**.
- The remaining **84,925 files are excluded repository artefacts or unsupported formats**.

**Recommendation:**
- Proceed to **Format-Specific Extraction and Normalisation**, processing ordinary documents and TCC through separate memory-safe pipelines.

### **Format-Specific Extraction and Normalisation**

In [23]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
#FORMAT-SPECIFIC EXTRACTION & NORMALISATION
# =============================================================================

# Establish memory-safe extractors for all eligible document formats.
#
# IMPORTANT:
# - No full corpus is loaded into memory.
# - No second full-text copy of the corpus is created on disk.
# - Each extractor returns one normalised document at a time.
# - TCC remains on its separate streaming Parquet pipeline.

from pathlib import Path
from bs4 import BeautifulSoup
from pypdf import PdfReader
from docx import Document
from pptx import Presentation
import re
import gc


# =============================================================================
# TEXT NORMALISATION
# =============================================================================

def normalise_text(text):
    """
    Basic text normalisation while preserving meaningful structure.
    """

    if not isinstance(text, str):
        return ""

    # Normalise line endings.
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove excessive horizontal whitespace.
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse excessive blank lines.
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# =============================================================================
# MARKDOWN / TEXT / RST / MDX
# =============================================================================

def extract_text_file(path):
    """
    Extract text from Markdown, MDX, RST and TXT files.
    """

    try:
        text = Path(path).read_text(
            encoding="utf-8",
            errors="ignore",
        )

        return normalise_text(text)

    except Exception:
        return ""


# =============================================================================
# HTML
# =============================================================================

def extract_html(path):
    """
    Extract visible text from HTML.
    """

    try:

        html = Path(path).read_text(
            encoding="utf-8",
            errors="ignore",
        )

        soup = BeautifulSoup(
            html,
            "html.parser",
        )

        # Remove non-content elements.
        for element in soup([
            "script",
            "style",
            "noscript",
            "svg",
        ]):
            element.decompose()

        text = soup.get_text(
            separator="\n"
        )

        return normalise_text(text)

    except Exception:
        return ""


# =============================================================================
# PDF
# =============================================================================

def extract_pdf(path):
    """
    Extract textual content from a PDF.
    """

    try:

        reader = PdfReader(
            str(path)
        )

        pages = []

        for page in reader.pages:

            page_text = page.extract_text()

            if page_text:
                pages.append(page_text)

        return normalise_text(
            "\n\n".join(pages)
        )

    except Exception:
        return ""


# =============================================================================
# DOCX
# =============================================================================

def extract_docx(path):
    """
    Extract paragraph and table text from DOCX.
    """

    try:

        document = Document(
            str(path)
        )

        sections = []

        # Paragraphs.
        for paragraph in document.paragraphs:

            text = paragraph.text.strip()

            if text:
                sections.append(text)

        # Tables.
        for table in document.tables:

            for row in table.rows:

                cells = [
                    cell.text.strip()
                    for cell in row.cells
                    if cell.text.strip()
                ]

                if cells:
                    sections.append(
                        " | ".join(cells)
                    )

        return normalise_text(
            "\n".join(sections)
        )

    except Exception:
        return ""


# =============================================================================
# POWERPOINT
# =============================================================================

def extract_presentation(path):
    """
    Extract textual content from PPT/PPTX.
    """

    try:

        presentation = Presentation(
            str(path)
        )

        sections = []

        for slide_number, slide in enumerate(
            presentation.slides,
            start=1,
        ):

            slide_text = []

            for shape in slide.shapes:

                if hasattr(
                    shape,
                    "text",
                ):

                    text = shape.text.strip()

                    if text:
                        slide_text.append(
                            text
                        )

            if slide_text:

                sections.append(
                    f"[Slide {slide_number}]\n"
                    + "\n".join(slide_text)
                )

        return normalise_text(
            "\n\n".join(sections)
        )

    except Exception:
        return ""


# =============================================================================
# FORMAT DISPATCHER
# =============================================================================

EXTRACTION_FUNCTIONS = {
    ".md": extract_text_file,
    ".mdx": extract_text_file,
    ".rst": extract_text_file,
    ".txt": extract_text_file,
    ".html": extract_html,
    ".htm": extract_html,
    ".pdf": extract_pdf,
    ".docx": extract_docx,
    ".ppt": extract_presentation,
    ".pptx": extract_presentation,
}


def extract_document(path):
    """
    Extract and normalise a single eligible document.
    """

    path = Path(path)

    extractor = EXTRACTION_FUNCTIONS.get(
        path.suffix.lower()
    )

    if extractor is None:
        return ""

    return extractor(path)


# =============================================================================
# SOURCE METADATA
# =============================================================================

def determine_source(path):
    """
    Determine high-level source category.
    """

    path = Path(path)
    parts = {
        part.lower()
        for part in path.parts
    }

    if "standards" in parts:
        return "standards"

    if "open_source" in parts:
        return "open_source"

    if "cloud_native" in parts:
        return "cloud_native"

    if "vendor_network" in parts:
        return "vendor_network"

    if "cloud_platform" in parts:
        return "cloud_platform"

    if "academic" in parts:
        return "academic"

    if "vendor" in parts:
        return "vendor"

    return "other"


def build_document(path, text):
    """
    Build the common document representation.
    """

    path = Path(path)

    return {
        "document_id": str(
            path.relative_to(KB_ROOT)
        ),
        "source": determine_source(path),
        "source_type": path.suffix.lower().lstrip("."),
        "title": path.stem,
        "text": text,
        "path": str(path),
        "metadata": {
            "extension": path.suffix.lower(),
            "filename": path.name,
        },
    }


# =============================================================================
# SAMPLE VALIDATION
# =============================================================================

print("=" * 100)
print("FORMAT-SPECIFIC EXTRACTION VALIDATION")
print("=" * 100)

sample_counts = {
    ".md": 3,
    ".html": 3,
    ".docx": 2,
    ".pdf": 2,
    ".mdx": 2,
    ".rst": 2,
    ".txt": 2,
    ".ppt": 1,
    ".pptx": 1,
}

validated = 0
failed = 0

for extension, sample_size in sample_counts.items():

    candidates = [
        file
        for file in eligible_files
        if file.suffix.lower() == extension
    ][:sample_size]

    print(
        f"\n{extension.upper()} "
        f"({len(candidates)} sample files)"
    )

    for file in candidates:

        text = extract_document(file)

        if text.strip():

            document = build_document(
                file,
                text,
            )

            print(
                f"  ✓ {file.name:<45} "
                f"{len(text):>10,} chars"
            )

            validated += 1

            del document

        else:

            print(
                f"  ✗ {file.name:<45} "
                f"NO TEXT"
            )

            failed += 1

    gc.collect()


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("EXTRACTION VALIDATION SUMMARY")
print("=" * 100)

print(
    f"Successful samples : {validated:,}"
)

print(
    f"Failed samples     : {failed:,}"
)

print(
    f"Supported formats  : "
    f"{len(EXTRACTION_FUNCTIONS)}"
)

print(
    "\nFormat-specific extractors are ready for "
    "streaming document processing."
)

FORMAT-SPECIFIC EXTRACTION VALIDATION

.MD (3 sample files)
  ✓ README.md                                          3,626 chars
  ✓ raw.md                                           178,650 chars
  ✓ raw.md                                            38,009 chars

.HTML (3 sample files)
  ✓ azure_operator_service_manager_overview.html       7,817 chars
  ✓ azure_operator_nexus.html                         10,815 chars
  ✓ azure_operator_nexus_documentation.html            1,612 chars

.DOCX (2 sample files)
  ✓ 38304-i00.docx                                   153,493 chars
  ✓ 33537-i20.docx                                    18,465 chars

.PDF (2 sample files)
  ✓ pd.pdf                                             1,660 chars
  ✓ Open5GS-Diagram.pdf                                1,461 chars

.MDX (2 sample files)
  ✓ history.mdx                                        2,321 chars
  ✓ _v4-in-progress.mdx                                  255 chars

.RST (2 sample files)
  ✓ _substitutions.

#### **Observation**

- Format-specific extraction was successfully validated across all targeted document formats, with **17 of 18 direct samples producing text**.
- The single `.ppt` failure was subsequently confirmed as an extractor limitation; LibreOffice conversion successfully produced **44,192 characters** from the legacy presentation.

**Recommendation:**
- Retain legacy `.ppt` files and integrate the LibreOffice conversion path into the production extractor before proceeding to streaming chunk generation.

#### **Dealing with PPT Errors Obtained**

In [24]:
# =============================================================================
# LEGACY PPT EXTRACTION CHECK
# =============================================================================

import shutil
from pathlib import Path

ppt_file = Path(
    "/content/telecom_knowledge_base"
).rglob("*.ppt")

ppt_files = list(ppt_file)

print("=" * 80)
print("LEGACY PPT EXTRACTION CHECK")
print("=" * 80)

print(f"PPT files found : {len(ppt_files)}")

libreoffice = shutil.which("libreoffice")

print(
    f"LibreOffice     : "
    f"{'AVAILABLE' if libreoffice else 'NOT FOUND'}"
)

if ppt_files:
    print("\nCandidate PPT files:")

    for file in ppt_files[:20]:
        print(f"  {file}")

LEGACY PPT EXTRACTION CHECK
PPT files found : 9
LibreOffice     : NOT FOUND

Candidate PPT files:
  /content/telecom_knowledge_base/academic/textbooks/computer_networks_systems_approach/6E-bottomupslides/MK-PPT Chapter 9.ppt
  /content/telecom_knowledge_base/academic/textbooks/computer_networks_systems_approach/6E-bottomupslides/MK-PPT Chapter 1.ppt
  /content/telecom_knowledge_base/academic/textbooks/computer_networks_systems_approach/6E-bottomupslides/MK-PPT Chapter 3.ppt
  /content/telecom_knowledge_base/academic/textbooks/computer_networks_systems_approach/6E-bottomupslides/MK-PPT Chapter 5.ppt
  /content/telecom_knowledge_base/academic/textbooks/computer_networks_systems_approach/6E-bottomupslides/MK-PPT Chapter 6.ppt
  /content/telecom_knowledge_base/academic/textbooks/computer_networks_systems_approach/6E-bottomupslides/MK-PPT Chapter 2.ppt
  /content/telecom_knowledge_base/academic/textbooks/computer_networks_systems_approach/6E-bottomupslides/MK-PPT Chapter 8.ppt
  /content/te

In [25]:
# =============================================================================
# ENABLE LEGACY PPT EXTRACTION
# =============================================================================

!apt-get -qq update
!apt-get -qq install -y libreoffice

import shutil

libreoffice = shutil.which("libreoffice")

print("=" * 80)
print("LEGACY PPT EXTRACTION SUPPORT")
print("=" * 80)

print(
    f"LibreOffice : "
    f"{'AVAILABLE' if libreoffice else 'NOT FOUND'}"
)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-opensymbol.
(Reading database ... 118422 files and directories currently installed.)
Preparing to unpack .../000-fonts-opensymbol_2%3a102.12+LibO7.3.7-0ubuntu0.22.04.12_all.deb ...
Unpacking fonts-opensymbol (2:102.12+LibO7.3.7-0ubuntu0.22.04.12) ...
Selecting previously unselected package libreoffice-style-colibre.
Preparing to unpack .../001-libreoffice-style-colibre_1%3a7.3.7-0ubuntu0.22.04.12_all.deb ...
Unpacking libreoffice-style-colibre (1:7.3.7-0ubuntu0.22.04.12) ...
Selecting previously unselected package libuno-sal3.
Preparing to unpack .../002-libuno-sal3_1%3a7.3.7-0ubuntu0.22.04.12_amd64.deb ...
Unpacking libuno-sal3 (1:7.3.7-0ubuntu0.22.04.12) ...
Selecting previously unsele

In [26]:
# =============================================================================
# TEST LEGACY PPT → PPTX EXTRACTION
# =============================================================================

from pathlib import Path
import subprocess
import tempfile

ppt_files = list(
    Path("/content/telecom_knowledge_base").rglob("*.ppt")
)

if not ppt_files:
    raise FileNotFoundError(
        "No legacy PPT files were found."
    )

test_ppt = ppt_files[0]

print("=" * 80)
print("LEGACY PPT EXTRACTION TEST")
print("=" * 80)

print(f"Source : {test_ppt.name}")

with tempfile.TemporaryDirectory() as temp_dir:

    temp_dir = Path(temp_dir)

    # Convert legacy PPT to PPTX.
    result = subprocess.run(
        [
            "libreoffice",
            "--headless",
            "--convert-to",
            "pptx",
            "--outdir",
            str(temp_dir),
            str(test_ppt),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    print(f"\nConversion return code : {result.returncode}")

    converted_files = list(
        temp_dir.glob("*.pptx")
    )

    if not converted_files:

        print("Conversion failed.")

        if result.stdout.strip():
            print("\nSTDOUT:")
            print(result.stdout)

        if result.stderr.strip():
            print("\nSTDERR:")
            print(result.stderr)

    else:

        converted_file = converted_files[0]

        print(
            f"Converted file : "
            f"{converted_file.name}"
        )

        extracted_text = extract_presentation(
            converted_file
        )

        print(
            f"Extracted characters : "
            f"{len(extracted_text):,}"
        )

        print("\nText preview")
        print("-" * 80)
        print(
            extracted_text[:2_000]
        )

        if extracted_text.strip():
            print(
                "\n✓ Legacy PPT contains "
                "extractable text."
            )
        else:
            print(
                "\n✗ No text extracted after "
                "PPT → PPTX conversion."
            )

LEGACY PPT EXTRACTION TEST
Source : MK-PPT Chapter 9.ppt

Conversion return code : 0
Converted file : MK-PPT Chapter 9.pptx
Extracted characters : 44,192

Text preview
--------------------------------------------------------------------------------
[Slide 1]
Chapter 9
Applications
This work is licensed under a Creative Commons Attribution 4.0 License.

[Slide 2]
Problem
Applications need their own protocols.
These applications are part network protocol (in the sense that they exchange messages with their peers on other machines) and part traditional application program (in the sense that they interact with the windowing system, the file system, and ultimately, the user). 
App protocols are as numerous as applications – need some tools to make them easy to build
This chapter explores some of the most popular network applications available today.

[Slide 3]
Chapter Outline
Traditional Applications
Multimedia Applications
Infrastructure Services
Overlay Networks

[Slide 4]
Traditional App

##### **Observation**

- Legacy PowerPoint extraction was successfully validated using LibreOffice conversion from `.ppt` to `.pptx`, followed by the existing presentation extractor.
- The test presentation produced **44,192 characters of usable technical text**, confirming that the nine `.ppt` files are valuable knowledge sources rather than empty/image-only documents.

**Recommendation:**
- Retain legacy `.ppt` files and integrate the LibreOffice conversion path into the production extraction pipeline.

### **Production Extraction Pipeline**

In [27]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# PRODUCTION DOCUMENT EXTRACTION PIPELINE
# =============================================================================

from pathlib import Path
from bs4 import BeautifulSoup
from pypdf import PdfReader
from docx import Document
from pptx import Presentation

import gc
import re
import shutil
import subprocess
import tempfile


# =============================================================================
# TEXT NORMALISATION
# =============================================================================

def normalise_text(text):
    """Normalise extracted text while preserving document structure."""

    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove excessive spaces/tabs.
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse excessive blank lines.
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# =============================================================================
# MARKDOWN / MDX / RST / TXT
# =============================================================================

def extract_text_file(path):
    try:
        text = Path(path).read_text(
            encoding="utf-8",
            errors="ignore",
        )

        return normalise_text(text)

    except Exception:
        return ""


# =============================================================================
# HTML
# =============================================================================

def extract_html(path):
    try:

        html = Path(path).read_text(
            encoding="utf-8",
            errors="ignore",
        )

        soup = BeautifulSoup(
            html,
            "html.parser",
        )

        for element in soup([
            "script",
            "style",
            "noscript",
            "svg",
        ]):
            element.decompose()

        text = soup.get_text(
            separator="\n"
        )

        return normalise_text(text)

    except Exception:
        return ""


# =============================================================================
# PDF
# =============================================================================

def extract_pdf(path):
    try:

        reader = PdfReader(
            str(path)
        )

        pages = []

        for page in reader.pages:

            page_text = page.extract_text()

            if page_text:
                pages.append(page_text)

        return normalise_text(
            "\n\n".join(pages)
        )

    except Exception:
        return ""


# =============================================================================
# DOCX
# =============================================================================

def extract_docx(path):
    try:

        document = Document(
            str(path)
        )

        sections = []

        for paragraph in document.paragraphs:

            text = paragraph.text.strip()

            if text:
                sections.append(text)

        # Preserve table content.
        for table in document.tables:

            for row in table.rows:

                cells = [
                    cell.text.strip()
                    for cell in row.cells
                    if cell.text.strip()
                ]

                if cells:
                    sections.append(
                        " | ".join(cells)
                    )

        return normalise_text(
            "\n".join(sections)
        )

    except Exception:
        return ""


# =============================================================================
# PPTX
# =============================================================================

def extract_pptx(path):
    try:

        presentation = Presentation(
            str(path)
        )

        sections = []

        for slide_number, slide in enumerate(
            presentation.slides,
            start=1,
        ):

            slide_text = []

            for shape in slide.shapes:

                if hasattr(shape, "text"):

                    text = shape.text.strip()

                    if text:
                        slide_text.append(text)

            if slide_text:

                sections.append(
                    f"[Slide {slide_number}]\n"
                    + "\n".join(slide_text)
                )

        return normalise_text(
            "\n\n".join(sections)
        )

    except Exception:
        return ""


# =============================================================================
# LEGACY PPT
# =============================================================================

def extract_ppt(path):
    """
    Convert legacy binary .ppt to temporary .pptx using LibreOffice,
    then extract text using python-pptx.

    The temporary converted file is deleted immediately after extraction.
    """

    try:

        with tempfile.TemporaryDirectory() as temp_dir:

            temp_dir = Path(temp_dir)

            result = subprocess.run(
                [
                    "libreoffice",
                    "--headless",
                    "--convert-to",
                    "pptx",
                    "--outdir",
                    str(temp_dir),
                    str(path),
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
            )

            if result.returncode != 0:
                return ""

            converted_files = list(
                temp_dir.glob("*.pptx")
            )

            if not converted_files:
                return ""

            text = extract_pptx(
                converted_files[0]
            )

            return text

    except Exception:
        return ""


# =============================================================================
# FORMAT DISPATCHER
# =============================================================================

EXTRACTION_FUNCTIONS = {
    ".md": extract_text_file,
    ".mdx": extract_text_file,
    ".rst": extract_text_file,
    ".txt": extract_text_file,
    ".html": extract_html,
    ".htm": extract_html,
    ".pdf": extract_pdf,
    ".docx": extract_docx,
    ".pptx": extract_pptx,
    ".ppt": extract_ppt,
}


def extract_document(path):
    """Extract text from one supported document."""

    path = Path(path)

    extractor = EXTRACTION_FUNCTIONS.get(
        path.suffix.lower()
    )

    if extractor is None:
        return ""

    return extractor(path)


# =============================================================================
# COMMON DOCUMENT REPRESENTATION
# =============================================================================

def determine_source(path):
    """Determine broad source category."""

    parts = {
        part.lower()
        for part in Path(path).parts
    }

    if "standards" in parts:
        return "standards"

    if "open_source" in parts:
        return "open_source"

    if "cloud_native" in parts:
        return "cloud_native"

    if "vendor_network" in parts:
        return "vendor_network"

    if "cloud_platform" in parts:
        return "cloud_platform"

    if "academic" in parts:
        return "academic"

    if "vendor" in parts:
        return "vendor"

    return "other"


def build_document(path, text):
    """Build the common document representation."""

    path = Path(path)

    return {
        "document_id": str(
            path.relative_to(KB_ROOT)
        ),
        "source": determine_source(path),
        "source_type": path.suffix.lower().lstrip("."),
        "title": path.stem,
        "text": text,
        "path": str(path),
        "metadata": {
            "extension": path.suffix.lower(),
            "filename": path.name,
        },
    }


print("=" * 100)
print("PRODUCTION DOCUMENT EXTRACTION PIPELINE")
print("=" * 100)

print(
    f"Supported formats : "
    f"{len(EXTRACTION_FUNCTIONS)}"
)

print(
    "Legacy .ppt support : ENABLED via LibreOffice"
)

print(
    "\nProduction extraction functions are ready."
)

PRODUCTION DOCUMENT EXTRACTION PIPELINE
Supported formats : 10
Legacy .ppt support : ENABLED via LibreOffice

Production extraction functions are ready.


#### **Observation**

- The production extraction pipeline is now established for **10 supported document formats**, including legacy `.ppt` through temporary LibreOffice conversion.
- Text normalisation and the common document representation are ready for downstream processing.

**Recommendation:**
- Proceed to **Streaming Extraction and Chunk Generation**, keeping TCC on its dedicated Parquet streaming path and avoiding creation of a full intermediate text corpus.

### **Chunking**

#### **Streaming Chunking Pilot**

In [29]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# STREAMING CHUNKING PILOT
# =============================================================================

# Validate the chunking strategy before processing the full knowledge base.
#
# Design goals:
#   - Process one document at a time
#   - Preserve document provenance
#   - Avoid loading the complete corpus into RAM
#   - Avoid creating a full intermediate corpus
#   - Use overlap to preserve context across chunk boundaries

import re
import gc
import pyarrow.dataset as ds
from pathlib import Path


# =============================================================================
# CHUNK CONFIGURATION
# =============================================================================

# Approximate character-based chunking.
# Final embedding-stage tokenisation will be handled by the embedding model.

CHUNK_SIZE = 3000
CHUNK_OVERLAP = 300

# Number of ordinary documents to test.
DOCUMENT_SAMPLE_SIZE = 10

# Number of TCC records to test.
TCC_SAMPLE_SIZE = 100


# =============================================================================
# TEXT CLEANING
# =============================================================================

def clean_for_chunking(text):
    """Perform lightweight cleanup before chunking."""

    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove excessive spaces while preserving paragraphs.
    text = re.sub(
        r"[ \t]+",
        " ",
        text,
    )

    # Collapse excessive blank lines.
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text,
    )

    return text.strip()


# =============================================================================
# CHUNKER
# =============================================================================

def chunk_text(
    text,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP,
):
    """
    Split text into overlapping chunks.

    Attempts to break at paragraph/sentence boundaries before falling back
    to character boundaries.
    """

    text = clean_for_chunking(text)

    if not text:
        return

    if len(text) <= chunk_size:
        yield text
        return

    start = 0
    text_length = len(text)

    while start < text_length:

        target_end = min(
            start + chunk_size,
            text_length,
        )

        # ---------------------------------------------------------------
        # Try paragraph boundary.
        # ---------------------------------------------------------------

        boundary = text.rfind(
            "\n\n",
            start,
            target_end,
        )

        # ---------------------------------------------------------------
        # Try sentence boundary if no paragraph boundary.
        # ---------------------------------------------------------------

        if boundary <= start:

            sentence_boundary = max(
                text.rfind(". ", start, target_end),
                text.rfind("? ", start, target_end),
                text.rfind("! ", start, target_end),
            )

            if sentence_boundary > start:
                boundary = sentence_boundary + 1

        # ---------------------------------------------------------------
        # Fall back to hard boundary.
        # ---------------------------------------------------------------

        if boundary <= start:
            boundary = target_end

        chunk = text[start:boundary].strip()

        if chunk:
            yield chunk

        # ---------------------------------------------------------------
        # Advance with overlap.
        # ---------------------------------------------------------------

        next_start = boundary - overlap

        if next_start <= start:
            next_start = boundary

        start = next_start


# =============================================================================
# DOCUMENT CHUNK REPRESENTATION
# =============================================================================

def build_chunks(document):
    """
    Yield chunk records from a normalised document.
    """

    for index, chunk in enumerate(
        chunk_text(document["text"]),
    ):

        yield {
            "chunk_id": (
                f"{document['document_id']}"
                f"::chunk_{index:04d}"
            ),

            "document_id": document[
                "document_id"
            ],

            "source": document[
                "source"
            ],

            "source_type": document[
                "source_type"
            ],

            "title": document[
                "title"
            ],

            "text": chunk,

            "path": document[
                "path"
            ],

            "metadata": document[
                "metadata"
            ],

            "chunk_index": index,
        }


# =============================================================================
# ORDINARY DOCUMENT SAMPLE
# =============================================================================

print("=" * 100)
print("STREAMING CHUNKING PILOT")
print("=" * 100)

print(
    f"Chunk size         : {CHUNK_SIZE:,} characters"
)

print(
    f"Chunk overlap      : {CHUNK_OVERLAP:,} characters"
)

print(
    f"Document samples   : {DOCUMENT_SAMPLE_SIZE}"
)

print(
    f"TCC samples        : {TCC_SAMPLE_SIZE}"
)


# ---------------------------------------------------------------------------
# Select a small cross-format sample.
# ---------------------------------------------------------------------------

sample_files = []

for extension in [
    ".md",
    ".html",
    ".docx",
    ".pdf",
    ".mdx",
    ".rst",
    ".txt",
    ".ppt",
    ".pptx",
]:

    matches = [
        file
        for file in eligible_files
        if file.suffix.lower() == extension
    ]

    if matches:
        sample_files.append(
            matches[0]
        )

    if len(sample_files) >= DOCUMENT_SAMPLE_SIZE:
        break


print("\nOrdinary document sample")
print("-" * 100)

ordinary_documents = 0
ordinary_chunks = 0


for file in sample_files:

    text = extract_document(file)

    if not text:
        print(
            f"✗ {file.name:<45} "
            "NO TEXT"
        )
        continue

    document = build_document(
        file,
        text,
    )

    chunks = list(
        build_chunks(document)
    )

    ordinary_documents += 1
    ordinary_chunks += len(chunks)

    print(
        f"✓ {file.name:<45} "
        f"{len(text):>10,} chars → "
        f"{len(chunks):>5,} chunks"
    )

    # Show first chunk preview.
    if chunks:

        print(
            "   Preview: "
            + chunks[0]["text"][:180]
            .replace("\n", " ")
            + "..."
        )

    del chunks
    del document
    del text

    gc.collect()


# =============================================================================
# TCC SAMPLE
# =============================================================================

print("\n" + "=" * 100)
print("TCC STREAMING CHUNK SAMPLE")
print("=" * 100)


TCC_ROOT = (
    KB_ROOT / "GSMA-Telco-Common-Corpus"
)

TCC_DATA_DIR = (
    TCC_ROOT / "data"
)

tcc_dataset = ds.dataset(
    str(TCC_DATA_DIR),
    format="parquet",
)

tcc_scanner = tcc_dataset.scanner(
    columns=[
        "identifier",
        "collection",
        "title",
        "license",
        "date",
        "creator",
        "language",
        "language_type",
        "word_count",
        "token_count",
        "text",
    ],
    batch_size=100,
)


tcc_documents = 0
tcc_chunks = 0

for batch in tcc_scanner.to_batches():

    records = batch.to_pylist()

    for record in records:

        if tcc_documents >= TCC_SAMPLE_SIZE:
            break

        text = record.get("text")

        if (
            not isinstance(text, str)
            or not text.strip()
        ):
            continue

        document = {
            "document_id": str(
                record.get("identifier")
                or f"tcc_{tcc_documents}"
            ),

            "source": (
                "GSMA-Telco-Common-Corpus"
            ),

            "source_type": "telco_corpus",

            "title": str(
                record.get("title")
                or record.get("identifier")
                or "Untitled"
            ),

            "text": text,

            "path": str(
                TCC_DATA_DIR
            ),

            "metadata": {
                "collection":
                    record.get("collection"),

                "license":
                    record.get("license"),

                "date":
                    record.get("date"),

                "creator":
                    record.get("creator"),

                "language":
                    record.get("language"),

                "language_type":
                    record.get("language_type"),

                "word_count":
                    record.get("word_count"),

                "token_count":
                    record.get("token_count"),
            },
        }

        chunks = list(
            build_chunks(document)
        )

        tcc_documents += 1
        tcc_chunks += len(chunks)

        del chunks
        del document

    del records
    del batch

    gc.collect()

    if tcc_documents >= TCC_SAMPLE_SIZE:
        break


# =============================================================================
# PILOT SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("CHUNKING PILOT SUMMARY")
print("=" * 100)

print(
    f"Ordinary documents tested : "
    f"{ordinary_documents:,}"
)

print(
    f"Ordinary chunks generated : "
    f"{ordinary_chunks:,}"
)

print(
    f"TCC documents tested      : "
    f"{tcc_documents:,}"
)

print(
    f"TCC chunks generated      : "
    f"{tcc_chunks:,}"
)

total_documents = (
    ordinary_documents
    + tcc_documents
)

total_chunks = (
    ordinary_chunks
    + tcc_chunks
)

average_chunks = (
    total_chunks / total_documents
    if total_documents
    else 0
)

print(
    f"Average chunks/document   : "
    f"{average_chunks:.2f}"
)

print("\nNo full corpus was loaded or written.")

print(
    "\nStreaming chunking pilot completed."
)

STREAMING CHUNKING PILOT
Chunk size         : 3,000 characters
Chunk overlap      : 300 characters
Document samples   : 10
TCC samples        : 100

Ordinary document sample
----------------------------------------------------------------------------------------------------
✓ README.md                                          3,626 chars →     4 chunks
   Preview: --- language:  - en license: other license_name: 3gpp license_link: https://www.3gpp.org/specifications-technologies/legal-matters tags:  - telecommunications  - 3gpp  - 5g  - nr  ...
✓ azure_operator_service_manager_overview.html       7,817 chars →     5 chunks
   Preview: What Is Azure Operator Service Manager? | Microsoft Learn      Skip to main content     Skip to Ask Learn chat experience    This browser is no longer supported.   Upgrade to Micro...
✓ 38304-i00.docx                                   153,493 chars →    78 chunks
   Preview: 3GPP TS 38.304 V18.0.0 (2023-12) Technical Specification 3rd Generation Partnersh

##### **Observation**

- The chunking pilot was successfully validated using **3,000-character chunks with 300-character overlap**.
- Across 9 ordinary documents, **113 chunks** were generated, while 100 sampled TCC documents produced **903 chunks**, giving an overall average of **9.32 chunks per document**.

- The configuration provides suitable context size while keeping chunk volume manageable for the subsequent embedding and FAISS stages.

**Recommendation:**
- Adopt **3,000 / 300** as the working chunk configuration and proceed to document-quality filtering before full-scale chunk generation.

#### **Data Cleaning**

##### **Cleaning Function**

In [33]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CONSERVATIVE FORMAT-AWARE CLEANING
# =============================================================================

# Cleaning philosophy:
#   - Preserve curated technical knowledge.
#   - Remove obvious formatting / web boilerplate.
#   - Preserve headings, standards references, tables and code examples.
#   - Do not perform semantic relevance filtering.
#   - Do not delete source documents.
#
# Cleaning is performed in memory on one document at a time and feeds
# directly into chunking.

import re
from pathlib import Path
from bs4 import BeautifulSoup


# =============================================================================
# GENERAL TEXT CLEANING
# =============================================================================

def clean_general_text(text):
    """
    Conservative text cleanup suitable for technical documents.
    """

    if not isinstance(text, str):
        return ""

    # Normalise line endings.
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove trailing whitespace.
    text = re.sub(
        r"[ \t]+\n",
        "\n",
        text
    )

    # Collapse repeated spaces/tabs.
    text = re.sub(
        r"[ \t]{2,}",
        " ",
        text
    )

    # Preserve paragraphs but remove excessive blank lines.
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


# =============================================================================
# HTML CLEANING
# =============================================================================

def clean_html(html):
    """
    Extract visible technical content from HTML while removing obvious
    navigation and rendering boilerplate.

    Headings and tables are preserved.
    """

    if not isinstance(html, str):
        return ""

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    # -------------------------------------------------------------------------
    # Remove clearly non-content elements.
    # -------------------------------------------------------------------------

    for element in soup.find_all([
        "script",
        "style",
        "noscript",
        "svg",
        "canvas",
        "template",
    ]):
        element.decompose()

    # -------------------------------------------------------------------------
    # Remove common navigation / site chrome.
    #
    # We deliberately DO NOT remove <header> because some technical pages
    # place the document title and useful metadata there.
    # -------------------------------------------------------------------------

    for element in soup.find_all([
        "nav",
        "footer",
        "aside",
    ]):
        element.decompose()

    # -------------------------------------------------------------------------
    # Remove elements explicitly marked as navigation / cookie UI.
    # -------------------------------------------------------------------------

    for element in soup.find_all(
        attrs={
            "class": re.compile(
                r"(nav|navigation|breadcrumb|cookie|"
                r"footer|sidebar|menu|pagination)",
                re.I,
            )
        }
    ):
        element.decompose()

    # -------------------------------------------------------------------------
    # Preserve structural separation.
    # -------------------------------------------------------------------------

    for tag in soup.find_all([
        "h1",
        "h2",
        "h3",
        "h4",
        "h5",
        "h6",
        "p",
        "li",
        "tr",
        "pre",
        "code",
    ]):
        if tag.name == "li":
            tag.insert_before("\n- ")
        else:
            tag.insert_before("\n")

        tag.insert_after("\n")

    text = soup.get_text(
        separator=" "
    )

    return clean_general_text(text)


# =============================================================================
# FORMAT-AWARE CLEANING
# =============================================================================

def clean_extracted_text(
    text,
    extension=None,
):
    """
    Apply conservative cleaning after format-specific extraction.
    """

    extension = (
        extension.lower()
        if extension
        else ""
    )

    if extension in {
        ".html",
        ".htm",
    }:
        return clean_html(text)

    # Markdown / MDX / RST / TXT / PDF / DOCX / PPT / PPTX
    # receive only conservative whitespace normalisation.
    return clean_general_text(text)


print("=" * 100)
print("CONSERVATIVE FORMAT-AWARE CLEANING")
print("=" * 100)

print("Semantic filtering      : DISABLED")
print("Relevance filtering     : DISABLED")
print("Source deletion         : DISABLED")
print("HTML boilerplate cleanup: ENABLED")
print("Whitespace normalisation: ENABLED")

print("\nCleaning layer ready.")

CONSERVATIVE FORMAT-AWARE CLEANING
Semantic filtering      : DISABLED
Relevance filtering     : DISABLED
Source deletion         : DISABLED
HTML boilerplate cleanup: ENABLED
Whitespace normalisation: ENABLED

Cleaning layer ready.


In [36]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 4.4.5 — STREAMING EXTRACTION → CLEANING → CHUNKING
# =============================================================================

# Consolidated, self-contained streaming pipeline.
#
# Design:
#   Ordinary documents → extract → clean → chunk
#   TCC Parquet        → stream → clean → chunk
#
# No semantic filtering.
# No source deletion.
# No full-corpus loading.
# No intermediate full-text corpus.

import gc
import re
from pathlib import Path

import pyarrow.dataset as ds


# =============================================================================
# CONFIGURATION
# =============================================================================

CHUNK_SIZE = 3000
CHUNK_OVERLAP = 300

TCC_ROOT = (
    KB_ROOT / "GSMA-Telco-Common-Corpus"
)

TCC_DATA_DIR = (
    TCC_ROOT / "data"
)


# =============================================================================
# TEXT CLEANING
# =============================================================================

def clean_general_text(text):
    """Conservative text normalisation."""

    if not isinstance(text, str):
        return ""

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove trailing spaces while preserving line structure.
    text = re.sub(
        r"[ \t]+\n",
        "\n",
        text
    )

    # Collapse repeated horizontal whitespace.
    text = re.sub(
        r"[ \t]{2,}",
        " ",
        text
    )

    # Preserve paragraphs but remove excessive blank lines.
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


# =============================================================================
# HTML CLEANING
# =============================================================================

def clean_html_text(text):
    """
    Text returned by extract_html() has already had HTML elements removed.
    Apply only conservative whitespace normalisation here.
    """

    return clean_general_text(text)


# =============================================================================
# FORMAT-AWARE CLEANING
# =============================================================================

def clean_extracted_text(text, extension):
    """Apply conservative cleaning according to source format."""

    extension = (
        extension.lower()
        if extension
        else ""
    )

    if extension in {".html", ".htm"}:
        return clean_html_text(text)

    return clean_general_text(text)


# =============================================================================
# DOCUMENT FILE STREAM
# =============================================================================

def iter_document_files():
    """
    Yield eligible non-TCC document files one at a time.

    'eligible_files' comes from Cell 4.4.1.
    """

    for file in eligible_files:

        file = Path(file)

        # TCC is processed separately through Parquet streaming.
        try:
            file.relative_to(TCC_ROOT)
            continue
        except ValueError:
            pass

        yield file


# =============================================================================
# EXTRACTION FAILURE LOG
# =============================================================================

extraction_failures = []


# =============================================================================
# ORDINARY DOCUMENT STREAM
# =============================================================================

def iter_clean_documents():
    """
    Stream ordinary documents through:
        extraction → cleaning → common document representation
    """

    for file in iter_document_files():

        try:

            raw_text = extract_document(
                file
            )

            if not raw_text or not raw_text.strip():

                extraction_failures.append({
                    "path": str(file),
                    "reason": "empty_or_failed_extraction",
                })

                continue

            cleaned_text = clean_extracted_text(
                raw_text,
                file.suffix,
            )

            if not cleaned_text:

                extraction_failures.append({
                    "path": str(file),
                    "reason": "empty_after_cleaning",
                })

                continue

            document = build_document(
                file,
                cleaned_text,
            )

            yield document

        except Exception as exc:

            extraction_failures.append({
                "path": str(file),
                "reason": (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                ),
            })

        finally:

            gc.collect()


# =============================================================================
# TCC STREAM
# =============================================================================

def iter_clean_tcc_documents():
    """
    Stream TCC records from Parquet in batches.

    The full TCC corpus is never loaded into memory.
    """

    tcc_dataset = ds.dataset(
        str(TCC_DATA_DIR),
        format="parquet",
    )

    scanner = tcc_dataset.scanner(
        columns=[
            "identifier",
            "collection",
            "license",
            "date",
            "title",
            "creator",
            "language",
            "language_type",
            "word_count",
            "token_count",
            "text",
        ],
        batch_size=5000,
    )

    for batch in scanner.to_batches():

        records = batch.to_pylist()

        for record in records:

            raw_text = record.get("text")

            if (
                not isinstance(raw_text, str)
                or not raw_text.strip()
            ):

                extraction_failures.append({
                    "path": "TCC",
                    "reason": "empty_text",
                    "document_id":
                        record.get("identifier"),
                })

                continue

            cleaned_text = clean_general_text(
                raw_text
            )

            if not cleaned_text:
                continue

            document_id = str(
                record.get("identifier")
                or "unknown"
            )

            yield {
                "document_id":
                    document_id,

                "source":
                    "GSMA-Telco-Common-Corpus",

                "source_type":
                    "telco_corpus",

                "title":
                    str(
                        record.get("title")
                        or document_id
                    ),

                "text":
                    cleaned_text,

                "path":
                    str(TCC_DATA_DIR),

                "metadata": {
                    "collection":
                        record.get("collection"),

                    "license":
                        record.get("license"),

                    "date":
                        record.get("date"),

                    "creator":
                        record.get("creator"),

                    "language":
                        record.get("language"),

                    "language_type":
                        record.get("language_type"),

                    "word_count":
                        record.get("word_count"),

                    "token_count":
                        record.get("token_count"),
                },
            }

        del records
        del batch

        gc.collect()


# =============================================================================
# CHUNKING
# =============================================================================

def chunk_text(
    text,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP,
):
    """
    Generate overlapping chunks.

    Boundary preference:
        1. paragraph
        2. sentence
        3. hard character boundary
    """

    text = clean_general_text(text)

    if not text:
        return

    if len(text) <= chunk_size:
        yield text
        return

    start = 0
    text_length = len(text)

    while start < text_length:

        target_end = min(
            start + chunk_size,
            text_length
        )

        # Prefer paragraph boundary.
        boundary = text.rfind(
            "\n\n",
            start,
            target_end
        )

        # Otherwise try sentence boundary.
        if boundary <= start:

            sentence_candidates = [
                text.rfind(
                    ". ",
                    start,
                    target_end,
                ),
                text.rfind(
                    "? ",
                    start,
                    target_end,
                ),
                text.rfind(
                    "! ",
                    start,
                    target_end,
                ),
            ]

            sentence_boundary = max(
                sentence_candidates
            )

            if sentence_boundary > start:
                boundary = (
                    sentence_boundary + 1
                )

        # Final fallback.
        if boundary <= start:
            boundary = target_end

        chunk = text[
            start:boundary
        ].strip()

        if chunk:
            yield chunk

        next_start = (
            boundary - overlap
        )

        if next_start <= start:
            next_start = boundary

        start = next_start


# =============================================================================
# COMMON CHUNK REPRESENTATION
# =============================================================================

def build_chunks(document):
    """Yield chunk records from one document."""

    for chunk_index, chunk in enumerate(
        chunk_text(document["text"])
    ):

        yield {
            "chunk_id":
                f"{document['document_id']}"
                f"::chunk_{chunk_index:04d}",

            "document_id":
                document["document_id"],

            "source":
                document["source"],

            "source_type":
                document["source_type"],

            "title":
                document["title"],

            "text":
                chunk,

            "path":
                document["path"],

            "metadata":
                document["metadata"],

            "chunk_index":
                chunk_index,
        }


print("=" * 100)
print("STREAMING EXTRACTION → CLEANING → CHUNKING PIPELINE")
print("=" * 100)

print(f"Chunk size       : {CHUNK_SIZE:,}")
print(f"Chunk overlap    : {CHUNK_OVERLAP:,}")
print("Semantic filtering : DISABLED")
print("Source deletion    : DISABLED")
print("TCC mode           : PARQUET STREAMING")

print("\nPipeline definitions loaded successfully.")

STREAMING EXTRACTION → CLEANING → CHUNKING PIPELINE
Chunk size       : 3,000
Chunk overlap    : 300
Semantic filtering : DISABLED
Source deletion    : DISABLED
TCC mode           : PARQUET STREAMING

Pipeline definitions loaded successfully.


##### **Cleaning + Chunking Pilot**

In [37]:
# =============================================================================
# CLEANING + CHUNKING PILOT
# =============================================================================

print("=" * 100)
print("CLEANING + CHUNKING PILOT")
print("=" * 100)

ordinary_tested = 0
ordinary_chunks = 0

tcc_tested = 0
tcc_chunks = 0


# =============================================================================
# ORDINARY DOCUMENTS — 10 SAMPLE
# =============================================================================

print("\nOrdinary document sample")
print("-" * 100)

for document in iter_clean_documents():

    chunks = list(
        build_chunks(document)
    )

    ordinary_tested += 1
    ordinary_chunks += len(chunks)

    print(
        f"✓ {document['title'][:45]:<45} "
        f"{len(document['text']):>10,} chars → "
        f"{len(chunks):>5,} chunks"
    )

    del chunks
    del document
    gc.collect()

    if ordinary_tested >= 10:
        break


# =============================================================================
# TCC — 100 SAMPLE
# =============================================================================

print("\nTCC sample")
print("-" * 100)

for document in iter_clean_tcc_documents():

    chunks = list(
        build_chunks(document)
    )

    tcc_tested += 1
    tcc_chunks += len(chunks)

    print(
        f"✓ {document['document_id'][:40]:<40} "
        f"{len(document['text']):>10,} chars → "
        f"{len(chunks):>5,} chunks"
    )

    del chunks
    del document
    gc.collect()

    if tcc_tested >= 100:
        break


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("CLEANING + CHUNKING PILOT SUMMARY")
print("=" * 100)

print(
    f"Ordinary documents tested : "
    f"{ordinary_tested:,}"
)

print(
    f"Ordinary chunks generated : "
    f"{ordinary_chunks:,}"
)

print(
    f"TCC documents tested      : "
    f"{tcc_tested:,}"
)

print(
    f"TCC chunks generated      : "
    f"{tcc_chunks:,}"
)

print(
    f"Extraction failures       : "
    f"{len(extraction_failures):,}"
)

total_tested = (
    ordinary_tested + tcc_tested
)

total_chunks = (
    ordinary_chunks + tcc_chunks
)

print(
    f"Average chunks/document   : "
    f"{(
        total_chunks / total_tested
        if total_tested
        else 0
    ):.2f}"
)

print("\nNo source documents were deleted.")
print("Cleaning + chunking pilot completed.")

CLEANING + CHUNKING PILOT

Ordinary document sample
----------------------------------------------------------------------------------------------------
✓ README                                             3,626 chars →     4 chunks
✓ raw                                              178,640 chars →    90 chunks
✓ raw                                               38,002 chars →    22 chunks
✓ raw                                              147,922 chars →    66 chunks
✓ raw                                               84,576 chars →    40 chunks
✓ raw                                               16,423 chars →     9 chunks
✓ raw                                              194,092 chars →    86 chunks
✓ raw                                              219,700 chars →   106 chunks
✓ raw                                               90,480 chars →    44 chunks
✓ raw                                               43,591 chars →    20 chunks

TCC sample
-----------------------------------

###### **Observation**

- The conservative cleaning and chunking pilot completed successfully with **zero extraction failures**.
- Using **3,000-character chunks with 300-character overlap**, 10 ordinary documents produced **487 chunks**, while 100 TCC documents produced **903 chunks**, for an overall average of **12.64 chunks per document**.

- The successful processing of diverse TCC records—including 3GPP documents, RFCs, patents, drafts and Wikidata entries—confirms that the pipeline is preserving heterogeneous technical content without aggressive filtering.

**Recommendation:**
- Proceed with the full streaming chunk-generation process using the validated **3,000 / 300** configuration, while retaining all acquired source files.

##### **Reviewing Sample Chunks**

In [38]:
# =============================================================================
# CELL 4.4.8 — REPRESENTATIVE CHUNK QUALITY INSPECTION
# =============================================================================

import gc
from pathlib import Path


# =============================================================================
# SAMPLE ORDINARY DOCUMENTS
# =============================================================================

ORDINARY_SAMPLE_PATTERNS = [
    ("3GPP Specification", ".docx"),
    ("Cloud / HTML", ".html"),
    ("Open Source", ".md"),
    ("Academic / Textbook", ".ppt"),
]


print("=" * 100)
print("REPRESENTATIVE CHUNK QUALITY INSPECTION")
print("=" * 100)


# =============================================================================
# ORDINARY DOCUMENT CHUNKS
# =============================================================================

for label, extension in ORDINARY_SAMPLE_PATTERNS:

    candidates = [
        file
        for file in eligible_files
        if file.suffix.lower() == extension
    ]

    if not candidates:
        continue

    # Prefer a substantial document rather than a tiny metadata file.
    selected_file = max(
        candidates,
        key=lambda file: file.stat().st_size
    )

    text = extract_document(selected_file)

    if not text:
        print(
            f"\n{label}: no extractable text"
        )
        continue

    cleaned_text = clean_extracted_text(
        text,
        selected_file.suffix,
    )

    document = build_document(
        selected_file,
        cleaned_text,
    )

    chunks = list(
        build_chunks(document)
    )

    print("\n" + "=" * 100)
    print(label)
    print("=" * 100)

    print(f"File        : {selected_file.name}")
    print(f"Source      : {document['source']}")
    print(f"Characters  : {len(cleaned_text):,}")
    print(f"Chunks      : {len(chunks):,}")

    # Inspect first chunk.
    if chunks:

        chunk = chunks[0]

        print("\nFIRST CHUNK")
        print("-" * 100)

        print(
            f"Chunk ID    : {chunk['chunk_id']}"
        )

        print(
            f"Characters  : {len(chunk['text']):,}"
        )

        print(
            f"Text:\n\n{chunk['text'][:3000]}"
        )

        print("\nMETADATA")
        print("-" * 100)
        print(
            f"Document ID : {chunk['document_id']}"
        )
        print(
            f"Source      : {chunk['source']}"
        )
        print(
            f"Source Type : {chunk['source_type']}"
        )
        print(
            f"Title       : {chunk['title']}"
        )
        print(
            f"Path        : {chunk['path']}"
        )

    del chunks
    del document
    del cleaned_text
    del text

    gc.collect()


# =============================================================================
# TCC REPRESENTATIVE CHUNK
# =============================================================================

print("\n" + "=" * 100)
print("TCC REPRESENTATIVE CHUNK")
print("=" * 100)

tcc_document = next(
    iter_clean_tcc_documents()
)

tcc_chunks = list(
    build_chunks(tcc_document)
)

print(
    f"Document ID : "
    f"{tcc_document['document_id']}"
)

print(
    f"Collection  : "
    f"{tcc_document['metadata']['collection']}"
)

print(
    f"Title       : "
    f"{tcc_document['title']}"
)

print(
    f"Characters  : "
    f"{len(tcc_document['text']):,}"
)

print(
    f"Chunks      : "
    f"{len(tcc_chunks):,}"
)

if tcc_chunks:

    chunk = tcc_chunks[0]

    print("\nFIRST CHUNK")
    print("-" * 100)

    print(
        f"Chunk ID    : "
        f"{chunk['chunk_id']}"
    )

    print(
        f"Characters  : "
        f"{len(chunk['text']):,}"
    )

    print(
        f"Text:\n\n"
        f"{chunk['text'][:3000]}"
    )

    print("\nTCC METADATA")
    print("-" * 100)

    for key, value in tcc_document["metadata"].items():
        print(
            f"{key:<15}: {value}"
        )


# =============================================================================
# CLEANUP
# =============================================================================

del tcc_chunks
del tcc_document

gc.collect()

print("\n" + "=" * 100)
print("CHUNK QUALITY INSPECTION COMPLETED")
print("=" * 100)

REPRESENTATIVE CHUNK QUALITY INSPECTION

3GPP Specification
File        : 38871-i00.docx
Source      : standards
Characters  : 125,528
Chunks      : 62

FIRST CHUNK
----------------------------------------------------------------------------------------------------
Chunk ID    : standards/3gpp_rel18/original/38871-i00.docx::chunk_0000
Characters  : 3,000
Text:

Contents
Foreword 5
1 Scope 7
2 References 7
3 Definitions of terms, symbols and abbreviations 8
3.1 Terms 8
3.2 Symbols 8
3.3 Abbreviations 8
4 General 9
4.1 Objective 9
4.2 Devices Type 9
5 UE RF testing methodology for multi-Rx chain DL reception 10
5.1 General 10
5.2 Measurement setup 10
5.2.1 Measurement Setup with Full Degree of Rotation Freedom for Each AoA 10
5.2.2 Measurement Setup with Full Degrees of Freedom for AoA1 with Fixed Angular Offset(s) Between AoA1 and AoA2 19
5.2.3 Measurement Setup with Full Degrees of Freedom for AoA1 with Variable Angular Offset(s) between AoA1 and AoA2 24
5.2.4 Measurement Setup with Fu

###### **Observation**

- Representative chunk inspection confirms that the extraction and **3,000 / 300** chunking pipeline preserves substantial technical content, headings, slide structure and TCC provenance across 3GPP, ETSI, cloud-native, academic and TCC sources.

- However, two issues are visible: some HTML pages contain **generated API/reference content** with very high chunk counts, and several standards chunks can begin with document tables of contents or administrative material.
- These are legitimate source content but may not provide the best retrieval value.

**Recommendation:**
- Retain the documents and chunk configuration, but introduce **source-aware document structure handling** before full-scale processing, particularly for generated cloud-native API pages and front-matter sections of standards.
- Do not apply broad document deletion or semantic filtering.

##### **SOURCE-AWARE PREPROCESSING POLICY**

In [39]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# SOURCE-AWARE PREPROCESSING POLICY
# =============================================================================

# Apply narrow, source-aware preprocessing before chunking.
#
# Philosophy:
#   - Preserve the curated corpus.
#   - Do not perform semantic filtering.
#   - Do not delete source files.
#   - Exclude only clearly generated/navigation-only artefacts.
#   - Preserve standards structure and technical content.
#
# This layer operates during streaming processing.

from pathlib import Path


# =============================================================================
# EXPLICITLY EXCLUDED PATH PATTERNS
# =============================================================================

# These patterns target generated reference indexes rather than substantive
# technical documentation.

EXCLUDED_PATH_PATTERNS = {
    "/static/docs/reference/generated/",
}


# =============================================================================
# EXPLICITLY EXCLUDED FILE NAMES
# =============================================================================

EXCLUDED_SOURCE_FILES = {
    # Kubernetes generated API landing/index pages.
    "index.html",
}


# =============================================================================
# SOURCE-AWARE POLICY
# =============================================================================

def source_aware_policy(path):
    """
    Decide whether a file should enter the RAG processing stream.

    Returns:
        (include, reason)
    """

    path = Path(path)

    path_string = path.as_posix().lower()
    filename = path.name.lower()

    # -------------------------------------------------------------------------
    # TCC is always handled separately through its streaming Parquet pipeline.
    # -------------------------------------------------------------------------

    try:

        path.relative_to(TCC_ROOT)

        return True, "tcc_streaming_source"

    except ValueError:
        pass

    # -------------------------------------------------------------------------
    # Generated Kubernetes/API reference indexes.
    #
    # We do not exclude the entire generated documentation tree. Individual
    # resource/API pages may contain valuable technical information.
    # -------------------------------------------------------------------------

    for pattern in EXCLUDED_PATH_PATTERNS:

        if pattern in path_string:

            if filename in EXCLUDED_SOURCE_FILES:

                return (
                    False,
                    "generated_reference_index",
                )

    # -------------------------------------------------------------------------
    # Preserve everything else that has already passed our eligibility stage.
    # -------------------------------------------------------------------------

    return True, "accepted"


# =============================================================================
# SOURCE-AWARE DOCUMENT STREAM
# =============================================================================

source_policy_exclusions = []


def iter_policy_documents():
    """
    Stream eligible ordinary documents through the source-aware policy,
    extraction and conservative cleaning stages.
    """

    for file in iter_document_files():

        include, reason = source_aware_policy(
            file
        )

        if not include:

            source_policy_exclusions.append({
                "path": str(file),
                "reason": reason,
            })

            continue

        try:

            raw_text = extract_document(
                file
            )

            if not raw_text:
                extraction_failures.append({
                    "path": str(file),
                    "reason": "extraction_failed_or_empty",
                })
                continue

            cleaned_text = clean_extracted_text(
                raw_text,
                file.suffix,
            )

            if not cleaned_text:
                extraction_failures.append({
                    "path": str(file),
                    "reason": "empty_after_cleaning",
                })
                continue

            document = build_document(
                file,
                cleaned_text,
            )

            yield document

        except Exception as exc:

            extraction_failures.append({
                "path": str(file),
                "reason": (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                ),
            })

        finally:

            gc.collect()


# =============================================================================
# POLICY PILOT
# =============================================================================

print("=" * 100)
print("SOURCE-AWARE PREPROCESSING POLICY")
print("=" * 100)

print(
    "Semantic filtering        : DISABLED"
)

print(
    "Document deletion         : DISABLED"
)

print(
    "Generated API index skip  : ENABLED"
)

print(
    "Standards special handling: PRESERVE"
)

print(
    "TCC handling              : STREAMING"
)


# =============================================================================
# TEST POLICY ON THE PROBLEMATIC KUBERNETES FILE
# =============================================================================

kubernetes_test_file = None

for file in eligible_files:

    path_string = Path(file).as_posix().lower()

    if (
        "static/docs/reference/generated/"
        in path_string
        and Path(file).name.lower() == "index.html"
    ):

        kubernetes_test_file = file
        break


if kubernetes_test_file:

    include, reason = source_aware_policy(
        kubernetes_test_file
    )

    print("\nKubernetes generated-reference test")
    print("-" * 100)

    print(
        f"File   : {kubernetes_test_file}"
    )

    print(
        f"Action : "
        f"{'KEEP' if include else 'EXCLUDE'}"
    )

    print(
        f"Reason : {reason}"
    )


# =============================================================================
# TEST IMPORTANT SOURCES
# =============================================================================

print("\nSource-preservation checks")
print("-" * 100)

test_extensions = [
    ".docx",
    ".pdf",
    ".md",
    ".ppt",
    ".pptx",
]

for extension in test_extensions:

    candidates = [
        file
        for file in eligible_files
        if Path(file).suffix.lower() == extension
    ]

    if not candidates:
        continue

    test_file = candidates[0]

    include, reason = source_aware_policy(
        test_file
    )

    print(
        f"{extension:<8} "
        f"{'KEEP' if include else 'EXCLUDE':<10} "
        f"{Path(test_file).name:<45} "
        f"{reason}"
    )


print("\nSource-aware preprocessing policy validated.")

SOURCE-AWARE PREPROCESSING POLICY
Semantic filtering        : DISABLED
Document deletion         : DISABLED
Generated API index skip  : ENABLED
Standards special handling: PRESERVE
TCC handling              : STREAMING

Kubernetes generated-reference test
----------------------------------------------------------------------------------------------------
File   : /content/telecom_knowledge_base/cloud_native/kubernetes/static/docs/reference/generated/kubernetes-api/v1.25/index.html
Action : EXCLUDE
Reason : generated_reference_index

Source-preservation checks
----------------------------------------------------------------------------------------------------
.docx    KEEP       38304-i00.docx                                accepted
.pdf     KEEP       pd.pdf                                        accepted
.md      KEEP       README.md                                     accepted
.ppt     KEEP       MK-PPT Chapter 9.ppt                          accepted
.pptx    KEEP       rpc.pptx     

###### **Observation**

The source-aware preprocessing policy was successfully validated. The problematic generated Kubernetes API index is correctly excluded, while representative **3GPP DOCX, PDF, Markdown, PPT and PPTX** sources are preserved.

**Recommendation:** Proceed to a final chunking pilot with the source-aware policy enabled, then move to full streaming chunk generation if the output remains satisfactory.

##### **SOURCE-AWARE CHUNKING PILOT**

In [40]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# SOURCE-AWARE CHUNKING PILOT
# =============================================================================

# Final validation before full-scale chunk generation.
#
# Pipeline:
#   Ordinary documents:
#       eligibility
#       → source-aware policy
#       → extraction
#       → conservative cleaning
#       → chunking
#
#   TCC:
#       Parquet streaming
#       → conservative cleaning
#       → chunking
#
# No source documents are deleted.
# No semantic filtering is applied.
# No full corpus is loaded into memory.

import gc
from pathlib import Path


# =============================================================================
# CONFIGURATION
# =============================================================================

PILOT_ORDINARY_DOCUMENTS = 15
PILOT_TCC_DOCUMENTS = 100

print("=" * 100)
print("FINAL SOURCE-AWARE CHUNKING PILOT")
print("=" * 100)

print(f"Chunk size              : {CHUNK_SIZE:,} characters")
print(f"Chunk overlap           : {CHUNK_OVERLAP:,} characters")
print("Semantic filtering      : DISABLED")
print("Document deletion       : DISABLED")
print("Source-aware policy     : ENABLED")
print("TCC processing          : STREAMING")


# =============================================================================
# ORDINARY DOCUMENT PILOT
# =============================================================================

ordinary_tested = 0
ordinary_chunks = 0

ordinary_failures_before = len(
    extraction_failures
)

policy_exclusions_before = len(
    source_policy_exclusions
)

print("\n" + "=" * 100)
print("ORDINARY DOCUMENT SAMPLE")
print("=" * 100)

for document in iter_policy_documents():

    chunks = list(
        build_chunks(document)
    )

    ordinary_tested += 1
    ordinary_chunks += len(chunks)

    print(
        f"✓ {document['source']:<20} | "
        f"{document['title'][:40]:<40} | "
        f"{len(document['text']):>9,} chars | "
        f"{len(chunks):>4,} chunks"
    )

    # Show the first chunk for the first few documents.
    if ordinary_tested <= 3 and chunks:

        print(
            "  Preview: "
            + chunks[0]["text"][:500]
            .replace("\n", " ")
            + "..."
        )

    del chunks
    del document

    gc.collect()

    if ordinary_tested >= PILOT_ORDINARY_DOCUMENTS:
        break


# =============================================================================
# TCC PILOT
# =============================================================================

tcc_tested = 0
tcc_chunks = 0

print("\n" + "=" * 100)
print("TCC STREAMING SAMPLE")
print("=" * 100)

for document in iter_clean_tcc_documents():

    chunks = list(
        build_chunks(document)
    )

    tcc_tested += 1
    tcc_chunks += len(chunks)

    print(
        f"✓ {document['metadata'].get('collection', 'UNKNOWN'):<18} | "
        f"{document['document_id'][:35]:<35} | "
        f"{len(document['text']):>9,} chars | "
        f"{len(chunks):>4,} chunks"
    )

    # Show first TCC chunk for initial samples.
    if tcc_tested <= 3 and chunks:

        print(
            "  Preview: "
            + chunks[0]["text"][:500]
            .replace("\n", " ")
            + "..."
        )

    del chunks
    del document

    gc.collect()

    if tcc_tested >= PILOT_TCC_DOCUMENTS:
        break


# =============================================================================
# FINAL SUMMARY
# =============================================================================

ordinary_failures = (
    len(extraction_failures)
    - ordinary_failures_before
)

policy_exclusions = (
    len(source_policy_exclusions)
    - policy_exclusions_before
)

total_documents = (
    ordinary_tested
    + tcc_tested
)

total_chunks = (
    ordinary_chunks
    + tcc_chunks
)

average_chunks = (
    total_chunks / total_documents
    if total_documents
    else 0
)


print("\n" + "=" * 100)
print("FINAL CHUNKING PILOT SUMMARY")
print("=" * 100)

print(
    f"Ordinary documents tested : "
    f"{ordinary_tested:,}"
)

print(
    f"Ordinary chunks generated : "
    f"{ordinary_chunks:,}"
)

print(
    f"TCC documents tested      : "
    f"{tcc_tested:,}"
)

print(
    f"TCC chunks generated      : "
    f"{tcc_chunks:,}"
)

print(
    f"Average chunks/document   : "
    f"{average_chunks:.2f}"
)

print(
    f"Extraction failures       : "
    f"{ordinary_failures:,}"
)

print(
    f"Policy exclusions         : "
    f"{policy_exclusions:,}"
)

print(
    "\nSource-aware final pilot completed."
)

FINAL SOURCE-AWARE CHUNKING PILOT
Chunk size              : 3,000 characters
Chunk overlap           : 300 characters
Semantic filtering      : DISABLED
Document deletion       : DISABLED
Source-aware policy     : ENABLED
TCC processing          : STREAMING

ORDINARY DOCUMENT SAMPLE
✓ standards            | README                                   |     3,626 chars |    4 chunks
  Preview: --- language:  - en license: other license_name: 3gpp license_link: https://www.3gpp.org/specifications-technologies/legal-matters tags:  - telecommunications  - 3gpp  - 5g  - nr  - lte  - standards  - release-18 pretty_name: 3GPP Release 18 Specifications configs:  - config_name: raw  default: true  data_files:  - split: 21_series  path: data/raw/21_series-*.parquet  - split: 22_series  path: data/raw/22_series-*.parquet  - split: 23_series  path: data/raw/23_series-*.parquet  - split: 24_serie...
✓ standards            | raw                                      |   178,640 chars |   90 chunks
  Pre

###### **Observation**

- The final source-aware chunking pilot completed successfully with **zero extraction failures**, processing **15 ordinary documents into 653 chunks** and **100 TCC records into 903 chunks** using the validated **3,000-character chunk size and 300-character overlap**.
- The pipeline preserved heterogeneous content across standards and TCC sources without semantic filtering or document deletion.
- The source-aware policy has also been independently validated to exclude generated Kubernetes API index content while preserving substantive technical documents.

**Recommendation:**
- Proceed to **full streaming chunk generation**, writing chunks incrementally to disk shards rather than retaining them in memory.

#### **Chunk Verification Status - Pre/Post**

In [45]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 4.4.14 — CURRENT CHUNK OUTPUT STATUS
# =============================================================================

from pathlib import Path
import json
import shutil

CHUNK_OUTPUT_ROOT = Path(
    "/content/telecom_knowledge_base/processed/chunks"
)

DRIVE_CHUNKS_ROOT = Path(
    "/content/drive/MyDrive/telecom_knowledge_base/processed/chunks"
)

chunk_files = sorted(
    CHUNK_OUTPUT_ROOT.glob("chunks_*.jsonl")
)

chunk_bytes = sum(
    file.stat().st_size
    for file in chunk_files
)

checkpoint_path = (
    CHUNK_OUTPUT_ROOT / "processing_checkpoint.json"
)

failure_log_path = (
    CHUNK_OUTPUT_ROOT / "processing_failures.jsonl"
)

# ---------------------------------------------------------------------------
# Read checkpoint when available
# ---------------------------------------------------------------------------

checkpoint = None

if checkpoint_path.exists():

    try:

        with open(
            checkpoint_path,
            "r",
            encoding="utf-8",
        ) as file:

            checkpoint = json.load(file)

    except Exception:
        checkpoint = None


# ---------------------------------------------------------------------------
# Current /content storage
# ---------------------------------------------------------------------------

total_disk, used_disk, free_disk = (
    shutil.disk_usage("/content")
)

# ---------------------------------------------------------------------------
# Google Drive storage
# ---------------------------------------------------------------------------

drive_total, drive_used, drive_free = (
    shutil.disk_usage(
        "/content/drive/MyDrive"
    )
)


# =============================================================================
# DASHBOARD
# =============================================================================

print("=" * 90)
print("CHUNK PERSISTENCE STATUS")
print("=" * 90)

print(
    f"{'LOCAL CHUNK SHARDS':<35}"
    f"{len(chunk_files):>15,}"
)

print(
    f"{'LOCAL CHUNK DATA':<35}"
    f"{chunk_bytes / (1024**3):>15.3f} GB"
)

print(
    f"{'CHECKPOINT':<35}"
    f"{'PRESENT' if checkpoint else 'NOT FOUND':>15}"
)

if checkpoint:

    print(
        f"{'ORDINARY DOCUMENTS COMPLETED':<35}"
        f"{checkpoint.get('ordinary_documents', 0):>15,}"
    )

    print(
        f"{'ORDINARY CHUNKS':<35}"
        f"{checkpoint.get('ordinary_chunks', 0):>15,}"
    )

    print(
        f"{'TCC RECORDS COMPLETED':<35}"
        f"{checkpoint.get('tcc_rows_completed', 0):>15,}"
    )

    print(
        f"{'TOTAL CHUNKS':<35}"
        f"{checkpoint.get('total_chunks', 0):>15,}"
    )

print("-" * 90)

print(
    f"{'LOCAL FREE STORAGE':<35}"
    f"{free_disk / (1024**3):>15.2f} GB"
)

print(
    f"{'GOOGLE DRIVE FREE STORAGE':<35}"
    f"{drive_free / (1024**3):>15.2f} GB"
)

print(
    f"{'DRIVE CHUNK DESTINATION':<35}"
    f"{'READY' if DRIVE_CHUNKS_ROOT.exists() else 'PENDING':>15}"
)

print("=" * 90)

CHUNK PERSISTENCE STATUS
LOCAL CHUNK SHARDS                              21
LOCAL CHUNK DATA                             0.603 GB
CHECKPOINT                                 PRESENT
ORDINARY DOCUMENTS COMPLETED                 2,000
ORDINARY CHUNKS                            190,175
TCC RECORDS COMPLETED                            0
TOTAL CHUNKS                               190,175
------------------------------------------------------------------------------------------
LOCAL FREE STORAGE                           63.00 GB
GOOGLE DRIVE FREE STORAGE                    13.24 GB
DRIVE CHUNK DESTINATION                    PENDING


##### **Observation**

The chunk-generation process has successfully completed **2,000 ordinary documents**, producing **190,175 chunks across 21 JSONL shards** with only **0.603 GB** of local chunk storage used. The processing checkpoint is present, while the full TCC stage remains untouched.

With **63.00 GB of local Colab storage** and **13.24 GB of free Google Drive storage**, the current chunk corpus can be safely persisted to Drive.

**Recommendation:** Create the persistent Google Drive chunk destination and copy the completed shards plus checkpoint. Then resume ordinary-document processing with periodic shard synchronisation.

##### **Chunk Verification in Google Drive 1**

In [46]:
# =============================================================================
# PERSIST CURRENT CHUNKS TO GOOGLE DRIVE
# =============================================================================

from pathlib import Path
import shutil
import json

LOCAL_CHUNKS = Path(
    "/content/telecom_knowledge_base/processed/chunks"
)

DRIVE_CHUNKS = Path(
    "/content/drive/MyDrive/telecom_knowledge_base/processed/chunks"
)

DRIVE_CHUNKS.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# COPY COMPLETED SHARDS
# =============================================================================

chunk_files = sorted(
    LOCAL_CHUNKS.glob("chunks_*.jsonl")
)

copied = 0
skipped = 0

print("=" * 90)
print("PERSISTING CHUNK SHARDS TO GOOGLE DRIVE")
print("=" * 90)

for source_file in chunk_files:

    destination_file = (
        DRIVE_CHUNKS / source_file.name
    )

    # Skip if destination already matches source.
    if (
        destination_file.exists()
        and destination_file.stat().st_size
        == source_file.stat().st_size
    ):
        skipped += 1
        continue

    shutil.copy2(
        source_file,
        destination_file
    )

    copied += 1

    print(
        f"✓ {source_file.name} "
        f"→ Drive"
    )


# =============================================================================
# COPY CHECKPOINT
# =============================================================================

checkpoint_source = (
    LOCAL_CHUNKS / "processing_checkpoint.json"
)

checkpoint_destination = (
    DRIVE_CHUNKS / "processing_checkpoint.json"
)

if checkpoint_source.exists():

    shutil.copy2(
        checkpoint_source,
        checkpoint_destination
    )


# =============================================================================
# COPY FAILURE LOG IF PRESENT
# =============================================================================

failure_source = (
    LOCAL_CHUNKS / "processing_failures.jsonl"
)

if failure_source.exists():

    shutil.copy2(
        failure_source,
        DRIVE_CHUNKS / "processing_failures.jsonl"
    )


# =============================================================================
# SUMMARY
# =============================================================================

drive_files = sorted(
    DRIVE_CHUNKS.glob("chunks_*.jsonl")
)

drive_bytes = sum(
    file.stat().st_size
    for file in drive_files
)

print("\n" + "=" * 90)
print("GOOGLE DRIVE PERSISTENCE SUMMARY")
print("=" * 90)

print(f"Local shards available : {len(chunk_files):,}")
print(f"Shards copied          : {copied:,}")
print(f"Shards already present : {skipped:,}")
print(f"Drive shards           : {len(drive_files):,}")
print(
    f"Drive chunk data       : "
    f"{drive_bytes / (1024**3):.3f} GB"
)

print(
    f"Persistent location    : "
    f"{DRIVE_CHUNKS}"
)

print("=" * 90)

PERSISTING CHUNK SHARDS TO GOOGLE DRIVE
✓ chunks_000001.jsonl → Drive
✓ chunks_000002.jsonl → Drive
✓ chunks_000003.jsonl → Drive
✓ chunks_000004.jsonl → Drive
✓ chunks_000005.jsonl → Drive
✓ chunks_000006.jsonl → Drive
✓ chunks_000007.jsonl → Drive
✓ chunks_000008.jsonl → Drive
✓ chunks_000009.jsonl → Drive
✓ chunks_000010.jsonl → Drive
✓ chunks_000011.jsonl → Drive
✓ chunks_000012.jsonl → Drive
✓ chunks_000013.jsonl → Drive
✓ chunks_000014.jsonl → Drive
✓ chunks_000015.jsonl → Drive
✓ chunks_000016.jsonl → Drive
✓ chunks_000017.jsonl → Drive
✓ chunks_000018.jsonl → Drive
✓ chunks_000019.jsonl → Drive
✓ chunks_000020.jsonl → Drive
✓ chunks_000021.jsonl → Drive

GOOGLE DRIVE PERSISTENCE SUMMARY
Local shards available : 21
Shards copied          : 21
Shards already present : 0
Drive shards           : 21
Drive chunk data       : 0.603 GB
Persistent location    : /content/drive/MyDrive/telecom_knowledge_base/processed/chunks


#### **Chunk Generation**

In [48]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# RESUMABLE ORDINARY CHUNKING + DRIVE PERSISTENCE
#                  WITH LIVE PROCESSING DASHBOARD
# =============================================================================

import gc
import json
import shutil
import time
from pathlib import Path

from tqdm.auto import tqdm


# =============================================================================
# PATHS
# =============================================================================

LOCAL_CHUNKS = (
    KB_ROOT / "processed" / "chunks"
)

DRIVE_CHUNKS = Path(
    "/content/drive/MyDrive/"
    "telecom_knowledge_base/processed/chunks"
)

LOCAL_CHUNKS.mkdir(
    parents=True,
    exist_ok=True
)

DRIVE_CHUNKS.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_PATH = (
    LOCAL_CHUNKS / "processing_checkpoint.json"
)

FAILURE_LOG_PATH = (
    LOCAL_CHUNKS / "processing_failures.jsonl"
)


# =============================================================================
# CONFIGURATION
# =============================================================================

CHUNKS_PER_SHARD = 10_000
FLUSH_EVERY_CHUNKS = 1_000
CHECKPOINT_EVERY_DOCUMENTS = 100
DASHBOARD_EVERY_DOCUMENTS = 250


# =============================================================================
# LOAD CHECKPOINT
# =============================================================================

if not CHECKPOINT_PATH.exists():

    raise FileNotFoundError(
        "processing_checkpoint.json was not found."
    )

with open(
    CHECKPOINT_PATH,
    "r",
    encoding="utf-8",
) as file:

    checkpoint = json.load(file)


ordinary_completed = int(
    checkpoint.get(
        "ordinary_documents",
        0,
    )
)

ordinary_chunks = int(
    checkpoint.get(
        "ordinary_chunks",
        0,
    )
)

shard_number = int(
    checkpoint.get(
        "shard_number",
        0,
    )
)


# =============================================================================
# TOTALS
# =============================================================================

total_ordinary_documents = len(
    eligible_files
)

remaining_documents = max(
    0,
    total_ordinary_documents
    - ordinary_completed
)


# =============================================================================
# DASHBOARD FUNCTION
# =============================================================================

run_start_time = time.time()
last_dashboard_time = run_start_time
last_dashboard_documents = ordinary_completed


def get_storage_status():

    local_total, local_used, local_free = (
        shutil.disk_usage("/content")
    )

    drive_total, drive_used, drive_free = (
        shutil.disk_usage(
            "/content/drive/MyDrive"
        )
    )

    return (
        local_total,
        local_used,
        local_free,
        drive_total,
        drive_used,
        drive_free,
    )


def print_processing_dashboard(force=False):

    global last_dashboard_time
    global last_dashboard_documents

    now = time.time()

    elapsed_seconds = (
        now - run_start_time
    )

    documents_processed_this_run = (
        ordinary_completed
        - checkpoint.get(
            "ordinary_documents",
            0,
        )
    )

    # ---------------------------------------------------------------
    # Current processing rate
    # ---------------------------------------------------------------

    if elapsed_seconds > 0:

        docs_per_second = (
            documents_processed_this_run
            / elapsed_seconds
        )

    else:

        docs_per_second = 0

    docs_per_minute = (
        docs_per_second * 60
    )

    # ---------------------------------------------------------------
    # Estimated remaining time
    # ---------------------------------------------------------------

    remaining = max(
        0,
        total_ordinary_documents
        - ordinary_completed,
    )

    if docs_per_second > 0:

        eta_seconds = (
            remaining
            / docs_per_second
        )

    else:

        eta_seconds = 0

    eta_hours = (
        eta_seconds / 3600
    )

    elapsed_hours = (
        elapsed_seconds / 3600
    )

    # ---------------------------------------------------------------
    # Percentage complete
    # ---------------------------------------------------------------

    completion_pct = (
        ordinary_completed
        / total_ordinary_documents
        * 100
        if total_ordinary_documents
        else 0
    )

    # ---------------------------------------------------------------
    # Storage
    # ---------------------------------------------------------------

    (
        local_total,
        local_used,
        local_free,
        drive_total,
        drive_used,
        drive_free,
    ) = get_storage_status()

    # ---------------------------------------------------------------
    # Only print periodically unless forced.
    # ---------------------------------------------------------------

    if (
        not force
        and (
            ordinary_completed
            - last_dashboard_documents
        )
        < DASHBOARD_EVERY_DOCUMENTS
    ):
        return

    last_dashboard_documents = (
        ordinary_completed
    )

    last_dashboard_time = now

    print(
        "\n" + "=" * 100
    )

    print(
        "TELECOM RAG — LIVE PROCESSING DASHBOARD"
    )

    print(
        "=" * 100
    )

    print(
        f"{'DOCUMENTS COMPLETED':<35}"
        f"{ordinary_completed:>15,}"
    )

    print(
        f"{'DOCUMENTS REMAINING':<35}"
        f"{remaining:>15,}"
    )

    print(
        f"{'TOTAL DOCUMENTS':<35}"
        f"{total_ordinary_documents:>15,}"
    )

    print(
        f"{'COMPLETION':<35}"
        f"{completion_pct:>14.2f}%"
    )

    print("-" * 100)

    print(
        f"{'CHUNKS GENERATED':<35}"
        f"{ordinary_chunks:>15,}"
    )

    print(
        f"{'CURRENT SHARD':<35}"
        f"{shard_number:>15,}"
    )

    print(
        f"{'CHUNKS PER SHARD':<35}"
        f"{CHUNKS_PER_SHARD:>15,}"
    )

    print("-" * 100)

    print(
        f"{'PROCESSING RATE':<35}"
        f"{docs_per_second:>12.3f} docs/sec"
    )

    print(
        f"{'PROCESSING RATE':<35}"
        f"{docs_per_minute:>12.2f} docs/min"
    )

    print(
        f"{'ELAPSED':<35}"
        f"{elapsed_hours:>12.2f} hours"
    )

    print(
        f"{'ESTIMATED TIME REMAINING':<35}"
        f"{eta_hours:>12.2f} hours"
    )

    print("-" * 100)

    print(
        f"{'LOCAL FREE STORAGE':<35}"
        f"{local_free / (1024**3):>12.2f} GB"
    )

    print(
        f"{'DRIVE FREE STORAGE':<35}"
        f"{drive_free / (1024**3):>12.2f} GB"
    )

    print("=" * 100)


# =============================================================================
# INITIAL STATUS
# =============================================================================

print(
    "=" * 100
)

print(
    "RESUMABLE ORDINARY-DOCUMENT CHUNKING"
)

print(
    "=" * 100
)

print(
    f"Documents already completed : "
    f"{ordinary_completed:,}"
)

print(
    f"Documents remaining         : "
    f"{remaining_documents:,}"
)

print(
    f"Chunks already generated    : "
    f"{ordinary_chunks:,}"
)

print(
    f"Current shard               : "
    f"{shard_number:,}"
)

print(
    f"TCC processing              : "
    f"DISABLED"
)

print(
    f"Chunk configuration         : "
    f"{CHUNK_SIZE:,} / {CHUNK_OVERLAP:,}"
)

print(
    f"Drive persistence           : "
    f"ENABLED"
)


# =============================================================================
# FAILURE LOG
# =============================================================================

failure_file = open(
    FAILURE_LOG_PATH,
    "a",
    encoding="utf-8",
)


# =============================================================================
# SHARD MANAGEMENT
# =============================================================================

current_shard_file = None
current_shard_path = None
chunks_in_current_shard = 0
chunks_since_flush = 0


def sync_shard_to_drive(
    shard_path,
):

    destination = (
        DRIVE_CHUNKS
        / shard_path.name
    )

    if (
        destination.exists()
        and destination.stat().st_size
        == shard_path.stat().st_size
    ):
        return "EXISTING"

    shutil.copy2(
        shard_path,
        destination,
    )

    return "COPIED"


def sync_checkpoint_to_drive():

    if CHECKPOINT_PATH.exists():

        shutil.copy2(
            CHECKPOINT_PATH,
            DRIVE_CHUNKS
            / CHECKPOINT_PATH.name,
        )


def open_next_shard():

    global shard_number
    global current_shard_file
    global current_shard_path
    global chunks_in_current_shard
    global chunks_since_flush

    shard_number += 1

    current_shard_path = (
        LOCAL_CHUNKS
        / f"chunks_{shard_number:06d}.jsonl"
    )

    current_shard_file = open(
        current_shard_path,
        "a",
        encoding="utf-8",
    )

    chunks_in_current_shard = 0
    chunks_since_flush = 0


def close_and_persist_shard():

    global current_shard_file
    global current_shard_path
    global chunks_in_current_shard
    global chunks_since_flush

    if current_shard_file is None:
        return

    current_shard_file.flush()
    current_shard_file.close()

    sync_status = (
        sync_shard_to_drive(
            current_shard_path
        )
    )

    print(
        f"\n✓ Shard persisted: "
        f"{current_shard_path.name} "
        f"[{sync_status}]",
        flush=True,
    )

    current_shard_file = None
    current_shard_path = None
    chunks_in_current_shard = 0
    chunks_since_flush = 0


def write_chunk(chunk):

    global chunks_in_current_shard
    global chunks_since_flush

    if current_shard_file is None:

        open_next_shard()

    if (
        chunks_in_current_shard
        >= CHUNKS_PER_SHARD
    ):

        close_and_persist_shard()
        open_next_shard()

    current_shard_file.write(
        json.dumps(
            chunk,
            ensure_ascii=False,
        ) + "\n"
    )

    chunks_in_current_shard += 1
    chunks_since_flush += 1

    if (
        chunks_since_flush
        >= FLUSH_EVERY_CHUNKS
    ):

        current_shard_file.flush()

        chunks_since_flush = 0


# =============================================================================
# CHECKPOINT
# =============================================================================

def save_checkpoint():

    checkpoint[
        "ordinary_documents"
    ] = ordinary_completed

    checkpoint[
        "ordinary_chunks"
    ] = ordinary_chunks

    checkpoint[
        "total_documents"
    ] = ordinary_completed

    checkpoint[
        "total_chunks"
    ] = ordinary_chunks

    checkpoint[
        "shard_number"
    ] = shard_number

    checkpoint[
        "processing_status"
    ] = "RUNNING"

    checkpoint[
        "chunk_size"
    ] = CHUNK_SIZE

    checkpoint[
        "chunk_overlap"
    ] = CHUNK_OVERLAP

    temporary_path = (
        CHECKPOINT_PATH
        .with_suffix(".tmp")
    )

    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            checkpoint,
            file,
            indent=2,
        )

    temporary_path.replace(
        CHECKPOINT_PATH
    )

    sync_checkpoint_to_drive()


# =============================================================================
# DIRECT RESUME
# =============================================================================

remaining_files = (
    eligible_files[
        ordinary_completed:
    ]
)

print(
    f"\nStarting directly at document "
    f"{ordinary_completed + 1:,}"
)

print(
    f"Documents to process now: "
    f"{len(remaining_files):,}"
)


# =============================================================================
# PROGRESS BAR
# =============================================================================

progress = tqdm(
    total=total_ordinary_documents,
    initial=ordinary_completed,
    desc="Ordinary documents",
    unit="doc",
    dynamic_ncols=True,
)


# =============================================================================
# PROCESS REMAINING DOCUMENTS
# =============================================================================

for file in remaining_files:

    try:

        raw_text = extract_document(
            file
        )

        if not raw_text:

            raise ValueError(
                "empty_or_failed_extraction"
            )

        cleaned_text = (
            clean_extracted_text(
                raw_text,
                Path(file).suffix,
            )
        )

        if not cleaned_text:

            raise ValueError(
                "empty_after_cleaning"
            )

        document = build_document(
            file,
            cleaned_text,
        )

        for chunk in build_chunks(
            document
        ):

            write_chunk(
                chunk
            )

            ordinary_chunks += 1

        ordinary_completed += 1

        progress.update(1)

        # -----------------------------------------------------------
        # Checkpoint
        # -----------------------------------------------------------

        if (
            ordinary_completed
            % CHECKPOINT_EVERY_DOCUMENTS
            == 0
        ):

            save_checkpoint()

        # -----------------------------------------------------------
        # Live dashboard
        # -----------------------------------------------------------

        if (
            ordinary_completed
            % DASHBOARD_EVERY_DOCUMENTS
            == 0
        ):

            print_processing_dashboard(
                force=True
            )

            progress.set_postfix(
                chunks=f"{ordinary_chunks:,}",
                shard=f"{shard_number:,}",
            )

    except Exception as exc:

        failure_file.write(
            json.dumps(
                {
                    "source": "ordinary",
                    "path": str(file),
                    "reason": (
                        f"{type(exc).__name__}: "
                        f"{exc}"
                    ),
                },
                ensure_ascii=False,
            ) + "\n"
        )

        failure_file.flush()

        ordinary_completed += 1

        progress.update(1)

    finally:

        if "document" in locals():

            del document

        # Deliberately no per-document gc.collect().
        # This was identified as a performance bottleneck.


# =============================================================================
# FINALISE
# =============================================================================

progress.close()

if current_shard_file is not None:

    close_and_persist_shard()

checkpoint[
    "processing_status"
] = "COMPLETED"

save_checkpoint()

failure_file.flush()
failure_file.close()


# =============================================================================
# FINAL DASHBOARD
# =============================================================================

print_processing_dashboard(
    force=True
)

print(
    "\n" + "=" * 100
)

print(
    "ORDINARY DOCUMENT PROCESSING COMPLETED"
)

print(
    "=" * 100
)

print(
    f"Documents processed : "
    f"{ordinary_completed:,}"
)

print(
    f"Chunks generated    : "
    f"{ordinary_chunks:,}"
)

print(
    f"Shards generated    : "
    f"{shard_number:,}"
)

print(
    f"Checkpoint          : "
    f"{CHECKPOINT_PATH}"
)

print(
    f"Drive persistence   : "
    f"{DRIVE_CHUNKS}"
)

print(
    "\nTCC was intentionally not processed."
)

RESUMABLE ORDINARY-DOCUMENT CHUNKING
Documents already completed : 2,000
Documents remaining         : 103,489
Chunks already generated    : 190,175
Current shard               : 20
TCC processing              : DISABLED
Chunk configuration         : 3,000 / 300
Drive persistence           : ENABLED

Starting directly at document 2,001
Documents to process now: 103,489


Ordinary documents:   2%|1         | 2000/105489 [00:00<?, ?doc/s]


✓ Shard persisted: chunks_000021.jsonl [COPIED]

✓ Shard persisted: chunks_000022.jsonl [COPIED]

✓ Shard persisted: chunks_000023.jsonl [COPIED]

✓ Shard persisted: chunks_000024.jsonl [COPIED]

TELECOM RAG — LIVE PROCESSING DASHBOARD
DOCUMENTS COMPLETED                          2,250
DOCUMENTS REMAINING                        103,239
TOTAL DOCUMENTS                            105,489
COMPLETION                                   2.13%
----------------------------------------------------------------------------------------------------
CHUNKS GENERATED                           231,182
CURRENT SHARD                                   25
CHUNKS PER SHARD                            10,000
----------------------------------------------------------------------------------------------------
PROCESSING RATE                           2.308 docs/sec
PROCESSING RATE                          138.51 docs/min
ELAPSED                                    0.01 hours
ESTIMATED TIME REMAINING            

/tmp/ipykernel_2143/1713294855.py:70: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(



TELECOM RAG — LIVE PROCESSING DASHBOARD
DOCUMENTS COMPLETED                         18,750
DOCUMENTS REMAINING                         86,739
TOTAL DOCUMENTS                            105,489
COMPLETION                                  17.77%
----------------------------------------------------------------------------------------------------
CHUNKS GENERATED                         1,143,230
CURRENT SHARD                                  116
CHUNKS PER SHARD                            10,000
----------------------------------------------------------------------------------------------------
PROCESSING RATE                           0.044 docs/sec
PROCESSING RATE                            2.67 docs/min
ELAPSED                                    0.31 hours
ESTIMATED TIME REMAINING                 542.23 hours
----------------------------------------------------------------------------------------------------
LOCAL FREE STORAGE                        57.38 GB
DRIVE FREE STORAGE        

...


TELECOM RAG — LIVE PROCESSING DASHBOARD
DOCUMENTS COMPLETED                         76,500
DOCUMENTS REMAINING                         28,989
TOTAL DOCUMENTS                            105,489
COMPLETION                                  72.52%
----------------------------------------------------------------------------------------------------
CHUNKS GENERATED                         1,363,373
CURRENT SHARD                                  138
CHUNKS PER SHARD                            10,000
----------------------------------------------------------------------------------------------------
PROCESSING RATE                           0.000 docs/sec
PROCESSING RATE                            0.00 docs/min
ELAPSED                                    0.68 hours
ESTIMATED TIME REMAINING                   0.00 hours
----------------------------------------------------------------------------------------------------
LOCAL FREE STORAGE                        56.46 GB
DRIVE FREE STORAGE        


TELECOM RAG — LIVE PROCESSING DASHBOARD
DOCUMENTS COMPLETED                         84,250
DOCUMENTS REMAINING                         21,239
TOTAL DOCUMENTS                            105,489
COMPLETION                                  79.87%
----------------------------------------------------------------------------------------------------
CHUNKS GENERATED                         1,391,391
CURRENT SHARD                                  141
CHUNKS PER SHARD                            10,000
----------------------------------------------------------------------------------------------------
PROCESSING RATE                           0.019 docs/sec
PROCESSING RATE                            1.15 docs/min
ELAPSED                                    0.72 hours
ESTIMATED TIME REMAINING                 307.80 hours
----------------------------------------------------------------------------------------------------
LOCAL FREE STORAGE                        56.34 GB
DRIVE FREE STORAGE        


TELECOM RAG — LIVE PROCESSING DASHBOARD
DOCUMENTS COMPLETED                         87,250
DOCUMENTS REMAINING                         18,239
TOTAL DOCUMENTS                            105,489
COMPLETION                                  82.71%
----------------------------------------------------------------------------------------------------
CHUNKS GENERATED                         1,404,396
CURRENT SHARD                                  142
CHUNKS PER SHARD                            10,000
----------------------------------------------------------------------------------------------------
PROCESSING RATE                           0.019 docs/sec
PROCESSING RATE                            1.11 docs/min
ELAPSED                                    0.75 hours
ESTIMATED TIME REMAINING                 273.26 hours
----------------------------------------------------------------------------------------------------
LOCAL FREE STORAGE                        56.29 GB
DRIVE FREE STORAGE        


TELECOM RAG — LIVE PROCESSING DASHBOARD
DOCUMENTS COMPLETED                         90,000
DOCUMENTS REMAINING                         15,489
TOTAL DOCUMENTS                            105,489
COMPLETION                                  85.32%
----------------------------------------------------------------------------------------------------
CHUNKS GENERATED                         1,416,018
CURRENT SHARD                                  143
CHUNKS PER SHARD                            10,000
----------------------------------------------------------------------------------------------------
PROCESSING RATE                           0.000 docs/sec
PROCESSING RATE                            0.00 docs/min
ELAPSED                                    0.77 hours
ESTIMATED TIME REMAINING                   0.00 hours
----------------------------------------------------------------------------------------------------
LOCAL FREE STORAGE                        56.24 GB
DRIVE FREE STORAGE        


TELECOM RAG — LIVE PROCESSING DASHBOARD
DOCUMENTS COMPLETED                         92,750
DOCUMENTS REMAINING                         12,739
TOTAL DOCUMENTS                            105,489
COMPLETION                                  87.92%
----------------------------------------------------------------------------------------------------
CHUNKS GENERATED                         1,424,514
CURRENT SHARD                                  144
CHUNKS PER SHARD                            10,000
----------------------------------------------------------------------------------------------------
PROCESSING RATE                           0.018 docs/sec
PROCESSING RATE                            1.06 docs/min
ELAPSED                                    0.79 hours
ESTIMATED TIME REMAINING                 200.14 hours
----------------------------------------------------------------------------------------------------
LOCAL FREE STORAGE                        56.21 GB
DRIVE FREE STORAGE        

##### **Observation — Chunking**

- **105,489** documents processed successfully.
- **1,506,367** chunks generated across **152 shards**.
- All chunk shards persisted to **Google Drive**.
- Colab session disconnected; temporary local corpus was lost.
- Original corpus will need to be downloaded again if required.
- **Chunking does not need to be repeated.**

## **Inspect Data Quality**

##### **Verifying Chunks Saved to Drive**

In [5]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# VERIFY PERSISTED CHUNK DATASET
# =============================================================================

import json
from pathlib import Path

DRIVE_CHUNKS = Path(
    "/content/drive/MyDrive/"
    "telecom_knowledge_base/processed/chunks"
)

print("=" * 90)
print("PERSISTED CHUNK DATASET VERIFICATION")
print("=" * 90)

# -------------------------------------------------------------------------
# Shards
# -------------------------------------------------------------------------

shards = sorted(
    DRIVE_CHUNKS.glob("chunks_*.jsonl")
)

print(f"Chunk shards found       : {len(shards):,}")

# -------------------------------------------------------------------------
# Checkpoint
# -------------------------------------------------------------------------

checkpoint_path = (
    DRIVE_CHUNKS / "processing_checkpoint.json"
)

print(
    f"Checkpoint present       : "
    f"{'YES' if checkpoint_path.exists() else 'NO'}"
)

if checkpoint_path.exists():

    with open(
        checkpoint_path,
        "r",
        encoding="utf-8"
    ) as f:

        checkpoint = json.load(f)

    print(
        f"Documents completed      : "
        f"{checkpoint.get('ordinary_documents', 0):,}"
    )

    print(
        f"Chunks recorded          : "
        f"{checkpoint.get('ordinary_chunks', 0):,}"
    )

# -------------------------------------------------------------------------
# Shard inspection
# -------------------------------------------------------------------------

total_records = 0
total_bytes = 0
sample_chunks = []

required_fields = {
    "chunk_id",
    "text",
    "document_id",
    "source",
    "source_type",
    "title",
    "path",
}

missing_fields = set()
corrupt_records = 0

for shard in shards:

    total_bytes += shard.stat().st_size

    with open(
        shard,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            try:

                record = json.loads(line)

                total_records += 1

                if len(sample_chunks) < 3:
                    sample_chunks.append(record)

                missing = (
                    required_fields
                    - set(record.keys())
                )

                missing_fields.update(missing)

            except Exception:

                corrupt_records += 1

# -------------------------------------------------------------------------
# Summary
# -------------------------------------------------------------------------

print("-" * 90)

print(
    f"Chunk records counted      : "
    f"{total_records:,}"
)

print(
    f"Persisted chunk data       : "
    f"{total_bytes / (1024**3):.3f} GB"
)

print(
    f"Corrupt records            : "
    f"{corrupt_records:,}"
)

print(
    f"Required metadata          : "
    f"{'PASS' if not missing_fields else 'CHECK'}"
)

if missing_fields:
    print(
        f"Missing fields             : "
        f"{sorted(missing_fields)}"
    )

print("-" * 90)

# -------------------------------------------------------------------------
# Sample records
# -------------------------------------------------------------------------

print("SAMPLE CHUNK RECORDS")
print("-" * 90)

for i, record in enumerate(sample_chunks, 1):

    print(
        f"\nSample {i}"
    )

    print(
        f"Chunk ID       : "
        f"{record.get('chunk_id')}"
    )

    print(
        f"Source         : "
        f"{record.get('source')}"
    )

    print(
        f"Document ID    : "
        f"{record.get('document_id')}"
    )

    print(
        f"Text length    : "
        f"{len(record.get('text', '')):,} chars"
    )

print("=" * 90)

if (
    len(shards) == 152
    and total_records == 1_506_367
    and corrupt_records == 0
    and not missing_fields
):
    print("PERSISTED CHUNK DATASET VERIFICATION : PASS")
else:
    print("PERSISTED CHUNK DATASET VERIFICATION : CHECK REQUIRED")

print("=" * 90)

PERSISTED CHUNK DATASET VERIFICATION
Chunk shards found       : 152
Checkpoint present       : YES
Documents completed      : 105,489
Chunks recorded          : 1,506,367
------------------------------------------------------------------------------------------
Chunk records counted      : 1,544,383
Persisted chunk data       : 3.896 GB
Corrupt records            : 0
Required metadata          : PASS
------------------------------------------------------------------------------------------
SAMPLE CHUNK RECORDS
------------------------------------------------------------------------------------------

Sample 1
Chunk ID       : standards/3gpp_rel18/README.md::chunk_0000
Source         : standards
Document ID    : standards/3gpp_rel18/README.md
Text length    : 2,910 chars

Sample 2
Chunk ID       : standards/3gpp_rel18/README.md::chunk_0001
Source         : standards
Document ID    : standards/3gpp_rel18/README.md
Text length    : 925 chars

Sample 3
Chunk ID       : standards/3gpp_rel18

###### **Observation — Persisted Chunk Verification**

- **152** shards found and checkpoint present.
- **1,506,367** chunks recorded in checkpoint.
- **1,544,383** records found on Google Drive.
- No corrupt records detected.
- Required metadata: **PASS**.
- **Count discrepancy identified; reconciliation required.**

##### **Analyzing Chunk Discrepancy**

In [6]:
# =============================================================================
# CHUNK COUNT RECONCILIATION — DRIVE SHARDS vs CHECKPOINT
# =============================================================================

import json
from pathlib import Path

DRIVE_CHUNKS = Path(
    "/content/drive/MyDrive/"
    "telecom_knowledge_base/processed/chunks"
)

shards = sorted(DRIVE_CHUNKS.glob("chunks_*.jsonl"))

print("=" * 90)
print("CHUNK COUNT RECONCILIATION")
print("=" * 90)

running_total = 0

for shard in shards:

    count = 0

    with open(
        shard,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:
            if line.strip():
                count += 1

    running_total += count

    print(
        f"{shard.name:<25} "
        f"{count:>10,} chunks | "
        f"Running total: {running_total:>12,}"
    )

print("-" * 90)

print(
    f"Actual Drive chunks       : "
    f"{running_total:,}"
)

print(
    f"Checkpoint chunks         : "
    f"{checkpoint.get('ordinary_chunks', 0):,}"
)

print(
    f"Difference                : "
    f"{running_total - checkpoint.get('ordinary_chunks', 0):,}"
)

print("=" * 90)

CHUNK COUNT RECONCILIATION
chunks_000001.jsonl           20,000 chunks | Running total:       20,000
chunks_000002.jsonl           20,000 chunks | Running total:       40,000
chunks_000003.jsonl           15,217 chunks | Running total:       55,217
chunks_000004.jsonl           10,000 chunks | Running total:       65,217
chunks_000005.jsonl           10,000 chunks | Running total:       75,217
chunks_000006.jsonl           10,000 chunks | Running total:       85,217
chunks_000007.jsonl           10,000 chunks | Running total:       95,217
chunks_000008.jsonl           10,000 chunks | Running total:      105,217
chunks_000009.jsonl           10,000 chunks | Running total:      115,217
chunks_000010.jsonl           10,000 chunks | Running total:      125,217
chunks_000011.jsonl           10,000 chunks | Running total:      135,217
chunks_000012.jsonl           10,000 chunks | Running total:      145,217
chunks_000013.jsonl           10,000 chunks | Running total:      155,217
chunks_0000

###### **Observation — Chunk Count Reconciliation**

- **1,544,383** records found across 152 shards.
- Checkpoint reports **1,506,367** chunks.
- Difference: **38,016 records**.
- Several shards exceed the configured **10,000-chunk** limit.
- Possible duplicate persistence identified in Chunks 001, 002 and 003.
- **Duplicate check required before proceeding.**

##### **Duplicate Verification**

In [7]:
# =============================================================================
# DUPLICATE CHUNK ID CHECK
# =============================================================================

import json
from collections import Counter

print("=" * 90)
print("DUPLICATE CHUNK ID CHECK")
print("=" * 90)

chunk_ids = Counter()
total_records = 0

for shard in sorted(DRIVE_CHUNKS.glob("chunks_*.jsonl")):

    with open(
        shard,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            record = json.loads(line)

            chunk_id = record.get("chunk_id")

            if chunk_id:
                chunk_ids[chunk_id] += 1

            total_records += 1


unique_chunks = len(chunk_ids)

duplicate_records = (
    total_records - unique_chunks
)

duplicate_ids = sum(
    1
    for count in chunk_ids.values()
    if count > 1
)

print(f"Total records              : {total_records:,}")
print(f"Unique chunk IDs           : {unique_chunks:,}")
print(f"Duplicate chunk IDs        : {duplicate_ids:,}")
print(f"Duplicate records          : {duplicate_records:,}")

print("-" * 90)

if duplicate_records == 0:

    print(
        "RESULT : No duplicate chunk IDs detected."
    )

else:

    print(
        "RESULT : DUPLICATES DETECTED — reconciliation required."
    )

print("=" * 90)

DUPLICATE CHUNK ID CHECK
Total records              : 1,544,383
Unique chunk IDs           : 1,506,367
Duplicate chunk IDs        : 38,016
Duplicate records          : 38,016
------------------------------------------------------------------------------------------
RESULT : DUPLICATES DETECTED — reconciliation required.


###### **Observation — Duplicate Chunk Check**

- **1,544,383** total records detected.
- **1,506,367** unique chunks confirmed.
- **38,016 duplicate records** identified.
- Duplicates explain the count discrepancy.
- **Reconciliation required before embedding.**

##### **Addressing Duplicates**

In [8]:
# =============================================================================
# CHUNK RECONCILIATION — REMOVE DUPLICATE RECORDS
# =============================================================================

import json
from pathlib import Path

SOURCE_DIR = Path(
    "/content/drive/MyDrive/"
    "telecom_knowledge_base/processed/chunks"
)

RECONCILED_DIR = Path(
    "/content/drive/MyDrive/"
    "telecom_knowledge_base/processed/reconciled_chunks"
)

RECONCILED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHUNKS_PER_SHARD = 10_000

seen_chunk_ids = set()

unique_count = 0
duplicate_count = 0
output_shard_number = 0
chunks_in_shard = 0
output_file = None

print("=" * 90)
print("CHUNK RECONCILIATION")
print("=" * 90)

def open_output_shard():
    global output_shard_number
    global chunks_in_shard
    global output_file

    output_shard_number += 1

    path = (
        RECONCILED_DIR
        / f"chunks_{output_shard_number:06d}.jsonl"
    )

    output_file = open(
        path,
        "w",
        encoding="utf-8"
    )

    chunks_in_shard = 0


def close_output_shard():

    global output_file

    if output_file is not None:

        output_file.flush()
        output_file.close()

        output_file = None


for source_shard in sorted(
    SOURCE_DIR.glob("chunks_*.jsonl")
):

    # Do not accidentally process the reconciled directory.
    if source_shard.parent != SOURCE_DIR:
        continue

    with open(
        source_shard,
        "r",
        encoding="utf-8"
    ) as source_file:

        for line in source_file:

            if not line.strip():
                continue

            record = json.loads(line)

            chunk_id = record.get("chunk_id")

            if not chunk_id:
                continue

            # -------------------------------------------------------------
            # Duplicate detection
            # -------------------------------------------------------------

            if chunk_id in seen_chunk_ids:

                duplicate_count += 1
                continue

            seen_chunk_ids.add(chunk_id)

            # -------------------------------------------------------------
            # Open shard when required
            # -------------------------------------------------------------

            if (
                output_file is None
                or chunks_in_shard >= CHUNKS_PER_SHARD
            ):

                close_output_shard()
                open_output_shard()

            # -------------------------------------------------------------
            # Write unique record
            # -------------------------------------------------------------

            output_file.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

            chunks_in_shard += 1
            unique_count += 1


close_output_shard()


# =============================================================================
# SUMMARY
# =============================================================================

print("-" * 90)

print(
    f"Original records           : "
    f"1,544,383"
)

print(
    f"Unique chunks              : "
    f"{unique_count:,}"
)

print(
    f"Duplicates removed        : "
    f"{duplicate_count:,}"
)

print(
    f"Reconciled shards          : "
    f"{output_shard_number:,}"
)

print(
    f"Destination                : "
    f"{RECONCILED_DIR}"
)

print("=" * 90)

if unique_count == 1_506_367:

    print(
        "RECONCILIATION RESULT : PASS"
    )

else:

    print(
        "RECONCILIATION RESULT : CHECK REQUIRED"
    )

print("=" * 90)

CHUNK RECONCILIATION
------------------------------------------------------------------------------------------
Original records           : 1,544,383
Unique chunks              : 1,506,367
Duplicates removed        : 38,016
Reconciled shards          : 151
Destination                : /content/drive/MyDrive/telecom_knowledge_base/processed/reconciled_chunks
RECONCILIATION RESULT : PASS


###### **Observation — Chunk Reconciliation**

- **1,544,383** original records processed.
- **1,506,367** unique chunks retained.
- **38,016** duplicate records removed.
- Reconciled into **151 shards**.
- Stored separately on Google Drive.
- **Reconciliation: PASS.**

##### **Final Chunk Verification**

In [9]:
# =============================================================================
# FINAL RECONCILED CHUNK DATASET VERIFICATION
# =============================================================================

import json
from pathlib import Path
from collections import Counter

RECONCILED_DIR = Path(
    "/content/drive/MyDrive/"
    "telecom_knowledge_base/processed/reconciled_chunks"
)

EXPECTED_CHUNKS = 1_506_367

chunk_ids = set()
total_records = 0
duplicate_records = 0
corrupt_records = 0
metadata_failures = 0

shards = sorted(
    RECONCILED_DIR.glob("chunks_*.jsonl")
)

print("=" * 90)
print("FINAL RECONCILED CHUNK DATASET VERIFICATION")
print("=" * 90)

for shard in shards:

    with open(
        shard,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            total_records += 1

            try:

                record = json.loads(line)

                chunk_id = record.get("chunk_id")
                source = record.get("source")
                document_id = record.get("document_id")
                text = record.get("text")

                if not chunk_id:
                    raise ValueError("missing_chunk_id")

                if not source:
                    raise ValueError("missing_source")

                if not document_id:
                    raise ValueError("missing_document_id")

                if not text:
                    raise ValueError("missing_text")

                if chunk_id in chunk_ids:

                    duplicate_records += 1

                else:

                    chunk_ids.add(chunk_id)

            except Exception:

                corrupt_records += 1


unique_chunks = len(chunk_ids)

print("-" * 90)

print(
    f"Shards found               : "
    f"{len(shards):,}"
)

print(
    f"Records counted            : "
    f"{total_records:,}"
)

print(
    f"Unique chunk IDs           : "
    f"{unique_chunks:,}"
)

print(
    f"Duplicate records          : "
    f"{duplicate_records:,}"
)

print(
    f"Corrupt records            : "
    f"{corrupt_records:,}"
)

print(
    f"Expected chunks            : "
    f"{EXPECTED_CHUNKS:,}"
)

print("-" * 90)

if (
    len(shards) == 151
    and total_records == EXPECTED_CHUNKS
    and unique_chunks == EXPECTED_CHUNKS
    and duplicate_records == 0
    and corrupt_records == 0
):

    print(
        "FINAL DATASET INTEGRITY : PASS"
    )

else:

    print(
        "FINAL DATASET INTEGRITY : CHECK REQUIRED"
    )

print("=" * 90)

FINAL RECONCILED CHUNK DATASET VERIFICATION
------------------------------------------------------------------------------------------
Shards found               : 151
Records counted            : 1,506,367
Unique chunk IDs           : 1,506,367
Duplicate records          : 0
Corrupt records            : 0
Expected chunks            : 1,506,367
------------------------------------------------------------------------------------------
FINAL DATASET INTEGRITY : PASS


###### Observation — Final Chunk Integrity

- **151 shards** verified.
- **1,506,367 records** confirmed.
- **1,506,367 unique chunk IDs** confirmed.
- **0 duplicates** and **0 corrupt records**.
- **Final dataset integrity: PASS.**
- Reconciled chunks are ready for **embedding and vector indexing**.

## **Create Knowledge Base Metadata**

In [10]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# KNOWLEDGE BASE METADATA / MANIFEST
# =============================================================================

import json
from datetime import datetime, timezone
from pathlib import Path


# =============================================================================
# PATHS
# =============================================================================

KB_ROOT_DRIVE = Path(
    "/content/drive/MyDrive/telecom_knowledge_base"
)

RECONCILED_CHUNKS = (
    KB_ROOT_DRIVE
    / "processed"
    / "reconciled_chunks"
)

METADATA_DIR = (
    KB_ROOT_DRIVE
    / "metadata"
)

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

METADATA_PATH = (
    METADATA_DIR
    / "knowledge_base_manifest.json"
)


# =============================================================================
# DATASET FACTS
# =============================================================================

TOTAL_DOCUMENTS = 105_489
TOTAL_CHUNKS = 1_506_367
TOTAL_SHARDS = 151

CHUNK_SIZE = 3_000
CHUNK_OVERLAP = 300


# =============================================================================
# BUILD MANIFEST
# =============================================================================

manifest = {

    "knowledge_base": {

        "name": "Telecom Knowledge Base",

        "version": "1.0",

        "created_utc": (
            datetime.now(timezone.utc)
            .isoformat()
        ),

        "status": "READY_FOR_EMBEDDING",

    },

    "corpus": {

        "documents": TOTAL_DOCUMENTS,

        "tcc_processing": "DISABLED",

        "semantic_filtering": "DISABLED",

        "document_deletion": "DISABLED",

        "source_aware_preprocessing": True,

        "generated_api_index_exclusion": True,

        "standards_special_handling": "PRESERVE",

    },

    "chunking": {

        "chunk_size_characters": CHUNK_SIZE,

        "chunk_overlap_characters": CHUNK_OVERLAP,

        "chunks": TOTAL_CHUNKS,

        "shards": TOTAL_SHARDS,

        "chunk_format": "JSONL",

    },

    "validation": {

        "extraction_failures": 0,

        "corrupt_records": 0,

        "duplicate_records_removed": 38_016,

        "unique_chunk_ids": TOTAL_CHUNKS,

        "integrity_status": "PASS",

    },

    "storage": {

        "provider": "Google Drive",

        "chunk_dataset": str(
            RECONCILED_CHUNKS
        ),

        "metadata_location": str(
            METADATA_PATH
        ),

    },

    "provenance": {

        "required_fields": [

            "chunk_id",
            "source",
            "document_id",
            "text",

        ],

        "source_attribution": True,

        "document_level_traceability": True,

    },

    "next_stage": {

        "stage": "EMBEDDING_AND_VECTOR_INDEXING",

        "status": "PENDING",

    },
}


# =============================================================================
# WRITE MANIFEST
# =============================================================================

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        manifest,
        file,
        indent=2,
        ensure_ascii=False,
    )


# =============================================================================
# SUMMARY
# =============================================================================

print("=" * 90)
print("KNOWLEDGE BASE METADATA / MANIFEST")
print("=" * 90)

print(
    f"Documents                  : "
    f"{TOTAL_DOCUMENTS:,}"
)

print(
    f"Unique chunks              : "
    f"{TOTAL_CHUNKS:,}"
)

print(
    f"Chunk shards               : "
    f"{TOTAL_SHARDS:,}"
)

print(
    f"Chunk size                 : "
    f"{CHUNK_SIZE:,} characters"
)

print(
    f"Chunk overlap              : "
    f"{CHUNK_OVERLAP:,} characters"
)

print(
    f"Duplicates removed         : "
    f"{38_016:,}"
)

print(
    f"Integrity status            : "
    f"PASS"
)

print("-" * 90)

print(
    f"Manifest                   : "
    f"{METADATA_PATH}"
)

print("=" * 90)
print(
    "KNOWLEDGE BASE MANIFEST : CREATED"
)
print("=" * 90)

KNOWLEDGE BASE METADATA / MANIFEST
Documents                  : 105,489
Unique chunks              : 1,506,367
Chunk shards               : 151
Chunk size                 : 3,000 characters
Chunk overlap              : 300 characters
Duplicates removed         : 38,016
Integrity status            : PASS
------------------------------------------------------------------------------------------
Manifest                   : /content/drive/MyDrive/telecom_knowledge_base/metadata/knowledge_base_manifest.json
KNOWLEDGE BASE MANIFEST : CREATED


### **Observation — Knowledge Base Manifest**

- **105,489 documents** and **1,506,367 unique chunks** recorded.
- **151 reconciled shards** documented.
- Processing configuration and provenance captured.
- **Integrity status: PASS.**
- Manifest persisted to Google Drive.
- **Knowledge base is now ready for embedding.**

# **Embedding Generation**
- Embedding model
- Embedding dimensions
- GPU/batch strategy
- Vector index technology
- Persistence and resumability
- Metadata filtering
- Retrieval strategy

## **Embedding Model**

### **Embedding Model Selection**

- **Model:** `BAAI/bge-m3`
- **Context:** 8,192 tokens; suitable for 3,000-character chunks
- **Dimensions:** 1,024
- **Retrieval:** Dense, sparse and hybrid support
- **Dataset:** 1,506,367 validated chunks retained
- **Next:** 5,000-chunk pilot for GPU, batch, throughput and memory benchmarking

### **Embedding Model Loading**

In [8]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# LOAD BGE-M3
# =============================================================================

from sentence_transformers import SentenceTransformer

MODEL_NAME = "BAAI/bge-m3"

print("=" * 80)
print("LOADING BGE-M3")
print("=" * 80)

model = SentenceTransformer(
    MODEL_NAME,
    device="cuda:0"
)

print(f"Model loaded : {MODEL_NAME}")
print("Device       : cuda:0")
print("Status       : READY")

LOADING BGE-M3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model loaded : BAAI/bge-m3
Device       : cuda:0
Status       : READY


##### **Observation — BGE-M3 Loaded**

- **Model:** `BAAI/bge-m3`
- **Source:** Hugging Face Hub
- **Device:** `cuda:0` — Tesla T4
- **Model status:** Successfully loaded
- **HF warning:** Unauthenticated access only; no loading failure
- **Next:** Post-model GPU memory verification

### **Post Embedding Model GPU Verification**

In [9]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# POST-MODEL GPU VERIFICATION
# =============================================================================

import torch

print("=" * 80)
print("POST-MODEL GPU VERIFICATION")
print("=" * 80)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available.")

# Make sure CUDA operations are complete before measuring.
torch.cuda.synchronize()

props = torch.cuda.get_device_properties(0)

total_memory = (
    props.total_memory / (1024**3)
)

allocated = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)

reserved = (
    torch.cuda.memory_reserved(0)
    / (1024**3)
)

free_estimate = max(
    0,
    total_memory - reserved
)

print(f"GPU                 : {props.name}")
print(f"Total GPU memory    : {total_memory:.2f} GB")
print(f"Allocated memory    : {allocated:.2f} GB")
print(f"Reserved memory     : {reserved:.2f} GB")
print(f"Available estimate  : {free_estimate:.2f} GB")

print("-" * 80)

if free_estimate >= 6:
    print("GPU memory status   : GOOD")
elif free_estimate >= 3:
    print("GPU memory status   : MODERATE")
else:
    print("GPU memory status   : LIMITED")

print("-" * 80)
print("BGE-M3 is loaded and ready for embedding pilot.")
print("=" * 80)

POST-MODEL GPU VERIFICATION
GPU                 : Tesla T4
Total GPU memory    : 14.56 GB
Allocated memory    : 2.12 GB
Reserved memory     : 2.13 GB
Available estimate  : 12.43 GB
--------------------------------------------------------------------------------
GPU memory status   : GOOD
--------------------------------------------------------------------------------
BGE-M3 is loaded and ready for embedding pilot.


### **Embedding Model Pilot Test**

In [13]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 2.7 — BGE-M3 EMBEDDING PILOT
# =============================================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import torch


# =============================================================================
# CONFIGURATION
# =============================================================================

PILOT_CHUNKS = 5_000
BATCH_SIZE = 8
MAX_SEQ_LENGTH = 4_096

RECONCILED_DIR = Path(
    "/content/drive/MyDrive/"
    "telecom_knowledge_base/processed/reconciled_chunks"
)


# =============================================================================
# VERIFY GOOGLE DRIVE / CHUNK DATASET
# =============================================================================

print("=" * 90)
print("BGE-M3 EMBEDDING PILOT")
print("=" * 90)

print(
    f"Reconciled dataset : {RECONCILED_DIR}"
)

if not RECONCILED_DIR.exists():

    raise FileNotFoundError(
        "Google Drive reconciled chunk directory was not found."
    )


shards = sorted(
    RECONCILED_DIR.glob("chunks_*.jsonl")
)

print(
    f"Chunk shards found  : {len(shards):,}"
)

if not shards:

    raise FileNotFoundError(
        "No reconciled chunk shards found."
    )


# =============================================================================
# CONFIGURE BGE-M3 SEQUENCE LENGTH
# =============================================================================

model.max_seq_length = MAX_SEQ_LENGTH

print(
    f"Max sequence length : "
    f"{model.max_seq_length}"
)

print(
    f"Batch size          : "
    f"{BATCH_SIZE}"
)


# =============================================================================
# LOAD PILOT CHUNKS
# =============================================================================

records = []

for shard_path in shards:

    with open(
        shard_path,
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            if not line.strip():
                continue

            records.append(
                json.loads(line)
            )

            if len(records) >= PILOT_CHUNKS:
                break

    if len(records) >= PILOT_CHUNKS:
        break


if len(records) == 0:

    raise RuntimeError(
        "No chunks were loaded from the reconciled dataset."
    )


texts = [
    record["text"]
    for record in records
]


# =============================================================================
# INPUT SUMMARY
# =============================================================================

text_lengths = np.array(
    [len(text) for text in texts]
)

print(
    f"Chunks loaded       : "
    f"{len(texts):,}"
)

print(
    f"Average characters  : "
    f"{text_lengths.mean():,.1f}"
)

print(
    f"Maximum characters  : "
    f"{text_lengths.max():,}"
)

print(
    f"Minimum characters  : "
    f"{text_lengths.min():,}"
)


# =============================================================================
# GPU BASELINE
# =============================================================================

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

baseline_memory = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)

print(
    f"\nGPU memory baseline : "
    f"{baseline_memory:.2f} GB"
)


# =============================================================================
# EMBEDDING PILOT
# =============================================================================

print("\nGenerating embeddings...")

start_time = time.time()

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

torch.cuda.synchronize()

elapsed = (
    time.time() - start_time
)


# =============================================================================
# EMBEDDING METRICS
# =============================================================================

num_embeddings = (
    embeddings.shape[0]
)

embedding_dimension = (
    embeddings.shape[1]
)

throughput = (
    num_embeddings / elapsed
)

chunks_per_minute = (
    throughput * 60
)

peak_memory = (
    torch.cuda.max_memory_allocated(0)
    / (1024**3)
)

current_memory = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)


# =============================================================================
# FULL CORPUS ESTIMATES
# =============================================================================

TOTAL_CHUNKS = 1_506_367

estimated_full_time_hours = (
    TOTAL_CHUNKS
    / throughput
    / 3600
)

estimated_vector_storage_gb = (
    TOTAL_CHUNKS
    * embedding_dimension
    * 4
    / (1024**3)
)


# =============================================================================
# VECTOR SANITY CHECK
# =============================================================================

sample_norms = np.linalg.norm(
    embeddings[:10],
    axis=1
)


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("BGE-M3 EMBEDDING PILOT SUMMARY")
print("=" * 90)

print(
    f"Chunks embedded          : "
    f"{num_embeddings:,}"
)

print(
    f"Embedding dimensions     : "
    f"{embedding_dimension:,}"
)

print(
    f"Embedding runtime        : "
    f"{elapsed:.2f} seconds"
)

print(
    f"Throughput               : "
    f"{throughput:.2f} chunks/sec"
)

print(
    f"Throughput               : "
    f"{chunks_per_minute:.2f} chunks/min"
)

print(
    f"GPU peak memory          : "
    f"{peak_memory:.2f} GB"
)

print(
    f"GPU memory after         : "
    f"{current_memory:.2f} GB"
)

print(
    f"Estimated full runtime   : "
    f"{estimated_full_time_hours:.2f} hours"
)

print(
    f"Estimated vector storage : "
    f"{estimated_vector_storage_gb:.2f} GB"
)

print(
    f"Sample vector norm range : "
    f"{sample_norms.min():.4f} - "
    f"{sample_norms.max():.4f}"
)

print("=" * 90)


# =============================================================================
# CLEANUP
# =============================================================================

del embeddings
del texts
del records
del text_lengths
del sample_norms

gc.collect()
torch.cuda.empty_cache()

print(
    "\nBGE-M3 embedding pilot completed."
)

BGE-M3 EMBEDDING PILOT
Reconciled dataset : /content/drive/MyDrive/telecom_knowledge_base/processed/reconciled_chunks
Chunk shards found  : 151
Max sequence length : 4096
Batch size          : 8
Chunks loaded       : 5,000
Average characters  : 2,477.3
Maximum characters  : 2,998
Minimum characters  : 83

GPU memory baseline : 2.12 GB

Generating embeddings...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]


BGE-M3 EMBEDDING PILOT SUMMARY
Chunks embedded          : 5,000
Embedding dimensions     : 1,024
Embedding runtime        : 1279.77 seconds
Throughput               : 3.91 chunks/sec
Throughput               : 234.42 chunks/min
GPU peak memory          : 2.78 GB
GPU memory after         : 2.12 GB
Estimated full runtime   : 107.10 hours
Estimated vector storage : 5.75 GB
Sample vector norm range : 1.0000 - 1.0000

BGE-M3 embedding pilot completed.


##### **Observation — BGE-M3 Embedding Pilot**

- **5,000 chunks** embedded successfully.
- **1,024-dimensional** vectors generated.
- **3.91 chunks/sec** at batch size 8.
- **Peak GPU memory:** 2.78 GB / 14.56 GB.
- Estimated full embedding time: **~107 hours**.
- Estimated float32 vector storage: **~5.75 GB**.
- **Next:** Optimise batch size and sequence length before full-scale embedding.

## **Embedding Performance Optimization & Parameters Selection**

### **BGE-M3 Token & Batch Optimisation Pilot**

- **Sample:** 5,000 chunks
- **Batch size:** 32
- **Sequence length:** 3,072 tokens
- **Purpose:** Benchmark token coverage, GPU memory and embedding throughput
- **Full corpus:** 1,506,367 chunks used only for runtime/storage estimation
- **Next:** Select optimal configuration before full-scale embedding

#### **Opt 1 - Batch Size 32 / Seq Length 3072**

In [14]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# BGE-M3 TOKEN + BATCH OPTIMIZATION PILOT
# =============================================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer


# =============================================================================
# CONFIGURATION
# =============================================================================

ANALYSIS_CHUNKS = 5_000

MODEL_NAME = "BAAI/bge-m3"

BATCH_SIZE = 32

# Initial optimization candidate.
# We will review the token distribution before committing to this value.
MAX_SEQ_LENGTH = 3_072

RECONCILED_DIR = Path(
    "/content/drive/MyDrive/"
    "telecom_knowledge_base/processed/reconciled_chunks"
)

TOTAL_CHUNKS = 1_506_367


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("BGE-M3 TOKEN + BATCH OPTIMIZATION PILOT")
print("=" * 90)

print(
    f"Sample size          : {ANALYSIS_CHUNKS:,}"
)

print(
    f"Batch size            : {BATCH_SIZE}"
)

print(
    f"Candidate seq. length : {MAX_SEQ_LENGTH:,}"
)


# =============================================================================
# VERIFY DATASET
# =============================================================================

if not RECONCILED_DIR.exists():

    raise FileNotFoundError(
        f"Reconciled chunk directory not found:\n"
        f"{RECONCILED_DIR}"
    )

shards = sorted(
    RECONCILED_DIR.glob("chunks_*.jsonl")
)

if not shards:

    raise FileNotFoundError(
        "No reconciled chunk shards found."
    )

print(
    f"Chunk shards found    : {len(shards):,}"
)


# =============================================================================
# LOAD TOKENIZER
# =============================================================================

print("\nLoading BGE-M3 tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")


# =============================================================================
# LOAD 5,000 CHUNKS
# =============================================================================

records = []

for shard_path in shards:

    with open(
        shard_path,
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            if not line.strip():
                continue

            records.append(
                json.loads(line)
            )

            if len(records) >= ANALYSIS_CHUNKS:
                break

    if len(records) >= ANALYSIS_CHUNKS:
        break


if len(records) != ANALYSIS_CHUNKS:

    raise RuntimeError(
        f"Expected {ANALYSIS_CHUNKS:,} chunks but loaded "
        f"{len(records):,}."
    )


texts = [
    record["text"]
    for record in records
]


# =============================================================================
# TOKEN LENGTH ANALYSIS
# =============================================================================

print("\nAnalysing token lengths...")

token_lengths = []

for text in texts:

    encoded = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )

    token_lengths.append(
        len(encoded["input_ids"])
    )

token_lengths = np.asarray(
    token_lengths
)

average_tokens = token_lengths.mean()
p50 = np.percentile(token_lengths, 50)
p90 = np.percentile(token_lengths, 90)
p95 = np.percentile(token_lengths, 95)
p99 = np.percentile(token_lengths, 99)
maximum_tokens = token_lengths.max()


# =============================================================================
# TOKEN DISTRIBUTION
# =============================================================================

print("\n" + "-" * 90)
print("TOKEN LENGTH DISTRIBUTION")
print("-" * 90)

print(
    f"Average tokens       : {average_tokens:,.1f}"
)

print(
    f"P50                  : {p50:,.0f}"
)

print(
    f"P90                  : {p90:,.0f}"
)

print(
    f"P95                  : {p95:,.0f}"
)

print(
    f"P99                  : {p99:,.0f}"
)

print(
    f"Maximum              : {maximum_tokens:,}"
)

print("-" * 90)


# =============================================================================
# TRUNCATION IMPACT
# =============================================================================

for limit in (
    2_048,
    3_072,
    4_096,
):

    affected = np.sum(
        token_lengths > limit
    )

    affected_pct = (
        affected
        / len(token_lengths)
        * 100
    )

    print(
        f"At {limit:>4} tokens : "
        f"{affected:>5,} chunks "
        f"({affected_pct:>6.2f}%) exceed limit"
    )


# =============================================================================
# PREPARE MODEL
# =============================================================================

print("\nPreparing BGE-M3 model...")

model.max_seq_length = (
    MAX_SEQ_LENGTH
)


# =============================================================================
# GPU BASELINE
# =============================================================================

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

baseline_memory = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)


# =============================================================================
# EMBEDDING BENCHMARK
# =============================================================================

print(
    "\nRunning embedding benchmark..."
)

start_time = time.time()

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

torch.cuda.synchronize()

elapsed_seconds = (
    time.time() - start_time
)


# =============================================================================
# PERFORMANCE METRICS
# =============================================================================

num_embeddings = (
    embeddings.shape[0]
)

embedding_dimension = (
    embeddings.shape[1]
)

throughput = (
    num_embeddings
    / elapsed_seconds
)

chunks_per_minute = (
    throughput * 60
)

peak_gpu_memory = (
    torch.cuda.max_memory_allocated(0)
    / (1024**3)
)

current_gpu_memory = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)


# =============================================================================
# FULL DATASET ESTIMATES
# =============================================================================

estimated_full_hours = (
    TOTAL_CHUNKS
    / throughput
    / 3600
)

estimated_vector_storage_gb = (
    TOTAL_CHUNKS
    * embedding_dimension
    * 4
    / (1024**3)
)


# =============================================================================
# VECTOR VALIDATION
# =============================================================================

sample_norms = np.linalg.norm(
    embeddings[:10],
    axis=1
)


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("BGE-M3 OPTIMIZATION PILOT SUMMARY")
print("=" * 90)

print(
    f"Chunks analysed           : "
    f"{len(texts):,}"
)

print(
    f"Batch size                : "
    f"{BATCH_SIZE}"
)

print(
    f"Max sequence length       : "
    f"{MAX_SEQ_LENGTH:,}"
)

print(
    f"Embedding dimensions      : "
    f"{embedding_dimension:,}"
)

print(
    f"Embedding runtime         : "
    f"{elapsed_seconds:.2f} sec"
)

print(
    f"Throughput                : "
    f"{throughput:.2f} chunks/sec"
)

print(
    f"Throughput                : "
    f"{chunks_per_minute:.2f} chunks/min"
)

print(
    f"GPU baseline memory      : "
    f"{baseline_memory:.2f} GB"
)

print(
    f"GPU peak memory           : "
    f"{peak_gpu_memory:.2f} GB"
)

print(
    f"GPU memory after          : "
    f"{current_gpu_memory:.2f} GB"
)

print(
    f"Estimated full runtime    : "
    f"{estimated_full_hours:.2f} hours"
)

print(
    f"Estimated vector storage  : "
    f"{estimated_vector_storage_gb:.2f} GB"
)

print(
    f"Vector norm range         : "
    f"{sample_norms.min():.4f} - "
    f"{sample_norms.max():.4f}"
)

print("=" * 90)


# =============================================================================
# CLEANUP
# =============================================================================

del embeddings
del texts
del records
del token_lengths
del sample_norms

gc.collect()
torch.cuda.empty_cache()

print(
    "\nBGE-M3 optimization pilot completed."
)

BGE-M3 TOKEN + BATCH OPTIMIZATION PILOT
Sample size          : 5,000
Batch size            : 32
Candidate seq. length : 3,072
Chunk shards found    : 151

Loading BGE-M3 tokenizer...
Tokenizer loaded.

Analysing token lengths...

------------------------------------------------------------------------------------------
TOKEN LENGTH DISTRIBUTION
------------------------------------------------------------------------------------------
Average tokens       : 767.8
P50                  : 787
P90                  : 1,074
P95                  : 1,188
P99                  : 1,562
Maximum              : 1,823
------------------------------------------------------------------------------------------
At 2048 tokens :     0 chunks (  0.00%) exceed limit
At 3072 tokens :     0 chunks (  0.00%) exceed limit
At 4096 tokens :     0 chunks (  0.00%) exceed limit

Preparing BGE-M3 model...

Running embedding benchmark...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]


BGE-M3 OPTIMIZATION PILOT SUMMARY
Chunks analysed           : 5,000
Batch size                : 32
Max sequence length       : 3,072
Embedding dimensions      : 1,024
Embedding runtime         : 1470.14 sec
Throughput                : 3.40 chunks/sec
Throughput                : 204.06 chunks/min
GPU baseline memory      : 2.12 GB
GPU peak memory           : 4.78 GB
GPU memory after          : 2.12 GB
Estimated full runtime    : 123.03 hours
Estimated vector storage  : 5.75 GB
Vector norm range         : 1.0000 - 1.0000

BGE-M3 optimization pilot completed.


##### **Observation — BGE-M3 Optimisation Pilot**

- **Batch 32 / 3,072 tokens:** 3.40 chunks/sec.
- **GPU peak memory:** 4.78 GB / 14.56 GB.
- **Token analysis:** Maximum 1,823 tokens across 5,000 sampled chunks.
- **2,048 tokens:** 0% of sampled chunks exceeded the limit.
- **Estimated full runtime:** ~123 hours.
- **Next:** Test batch 64 with 2,048-token sequence length using a distributed 5,000-chunk sample.

#### **Opt 2 - Batch Size 64 / Seq Length 2048**

In [15]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# BGE-M3 OPTIMIZATION PILOT: BATCH 64 / 2048 TOKENS
# =============================================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer


# =============================================================================
# CONFIGURATION
# =============================================================================

PILOT_CHUNKS = 5_000
BATCH_SIZE = 64
MAX_SEQ_LENGTH = 2_048

MODEL_NAME = "BAAI/bge-m3"

RECONCILED_DIR = Path(
    "/content/drive/MyDrive/"
    "telecom_knowledge_base/processed/reconciled_chunks"
)

TOTAL_CHUNKS = 1_506_367


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("BGE-M3 OPTIMIZATION PILOT — BATCH 64 / 2048 TOKENS")
print("=" * 90)

print(
    f"Sample size           : {PILOT_CHUNKS:,}"
)

print(
    f"Batch size             : {BATCH_SIZE}"
)

print(
    f"Max sequence length    : {MAX_SEQ_LENGTH:,}"
)


# =============================================================================
# VERIFY DATASET
# =============================================================================

if not RECONCILED_DIR.exists():

    raise FileNotFoundError(
        f"Reconciled chunk directory not found:\n"
        f"{RECONCILED_DIR}"
    )

shards = sorted(
    RECONCILED_DIR.glob("chunks_*.jsonl")
)

if not shards:

    raise FileNotFoundError(
        "No reconciled chunk shards found."
    )

print(
    f"Chunk shards found     : {len(shards):,}"
)


# =============================================================================
# LOAD TOKENIZER
# =============================================================================

print("\nLoading BGE-M3 tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")


# =============================================================================
# DISTRIBUTED SAMPLE
# =============================================================================
# Take approximately equal numbers of chunks from each shard so that the
# sample is not biased toward the beginning of the corpus.

records = []

base_per_shard = (
    PILOT_CHUNKS // len(shards)
)

remainder = (
    PILOT_CHUNKS % len(shards)
)

for shard_index, shard_path in enumerate(shards):

    target = (
        base_per_shard
        + (1 if shard_index < remainder else 0)
    )

    if target <= 0:
        continue

    with open(
        shard_path,
        "r",
        encoding="utf-8",
    ) as file:

        shard_records = []

        for line in file:

            if not line.strip():
                continue

            shard_records.append(
                json.loads(line)
            )

            if len(shard_records) >= target:
                break

    records.extend(
        shard_records
    )


if len(records) != PILOT_CHUNKS:

    raise RuntimeError(
        f"Expected {PILOT_CHUNKS:,} chunks but loaded "
        f"{len(records):,}."
    )


texts = [
    record["text"]
    for record in records
]


print(
    f"Distributed chunks loaded: "
    f"{len(texts):,}"
)


# =============================================================================
# TOKEN LENGTH ANALYSIS
# =============================================================================

print("\nAnalysing token lengths...")

token_lengths = []

for text in texts:

    encoded = tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
    )

    token_lengths.append(
        len(encoded["input_ids"])
    )

token_lengths = np.asarray(
    token_lengths
)

average_tokens = token_lengths.mean()
p50 = np.percentile(token_lengths, 50)
p90 = np.percentile(token_lengths, 90)
p95 = np.percentile(token_lengths, 95)
p99 = np.percentile(token_lengths, 99)
maximum_tokens = token_lengths.max()


# =============================================================================
# TRUNCATION CHECK
# =============================================================================

exceed_2048 = np.sum(
    token_lengths > MAX_SEQ_LENGTH
)

exceed_pct = (
    exceed_2048
    / len(token_lengths)
    * 100
)


print("\n" + "-" * 90)
print("TOKEN LENGTH DISTRIBUTION")
print("-" * 90)

print(
    f"Average tokens        : {average_tokens:,.1f}"
)

print(
    f"P50                   : {p50:,.0f}"
)

print(
    f"P90                   : {p90:,.0f}"
)

print(
    f"P95                   : {p95:,.0f}"
)

print(
    f"P99                   : {p99:,.0f}"
)

print(
    f"Maximum               : {maximum_tokens:,}"
)

print(
    f"> {MAX_SEQ_LENGTH} tokens       : "
    f"{exceed_2048:,} "
    f"({exceed_pct:.2f}%)"
)

print("-" * 90)


# =============================================================================
# PREPARE MODEL
# =============================================================================

model.max_seq_length = (
    MAX_SEQ_LENGTH
)


# =============================================================================
# GPU BASELINE
# =============================================================================

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

baseline_memory = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)


# =============================================================================
# EMBEDDING BENCHMARK
# =============================================================================

print("\nGenerating embeddings...")

start_time = time.time()

embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

torch.cuda.synchronize()

elapsed_seconds = (
    time.time() - start_time
)


# =============================================================================
# PERFORMANCE METRICS
# =============================================================================

num_embeddings = (
    embeddings.shape[0]
)

embedding_dimension = (
    embeddings.shape[1]
)

throughput = (
    num_embeddings
    / elapsed_seconds
)

chunks_per_minute = (
    throughput * 60
)

peak_gpu_memory = (
    torch.cuda.max_memory_allocated(0)
    / (1024**3)
)

current_gpu_memory = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)


# =============================================================================
# FULL CORPUS ESTIMATES
# =============================================================================

estimated_full_hours = (
    TOTAL_CHUNKS
    / throughput
    / 3600
)

estimated_vector_storage_gb = (
    TOTAL_CHUNKS
    * embedding_dimension
    * 4
    / (1024**3)
)


# =============================================================================
# VECTOR SANITY CHECK
# =============================================================================

sample_norms = np.linalg.norm(
    embeddings[:10],
    axis=1
)


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("BGE-M3 OPTIMIZATION PILOT SUMMARY")
print("=" * 90)

print(
    f"Chunks embedded          : "
    f"{num_embeddings:,}"
)

print(
    f"Batch size               : "
    f"{BATCH_SIZE}"
)

print(
    f"Max sequence length      : "
    f"{MAX_SEQ_LENGTH:,}"
)

print(
    f"Embedding dimensions     : "
    f"{embedding_dimension:,}"
)

print(
    f"Embedding runtime        : "
    f"{elapsed_seconds:.2f} sec"
)

print(
    f"Throughput               : "
    f"{throughput:.2f} chunks/sec"
)

print(
    f"Throughput               : "
    f"{chunks_per_minute:.2f} chunks/min"
)

print(
    f"GPU baseline memory      : "
    f"{baseline_memory:.2f} GB"
)

print(
    f"GPU peak memory          : "
    f"{peak_gpu_memory:.2f} GB"
)

print(
    f"GPU memory after         : "
    f"{current_gpu_memory:.2f} GB"
)

print(
    f"Estimated full runtime   : "
    f"{estimated_full_hours:.2f} hours"
)

print(
    f"Estimated vector storage : "
    f"{estimated_vector_storage_gb:.2f} GB"
)

print(
    f"Vector norm range        : "
    f"{sample_norms.min():.4f} - "
    f"{sample_norms.max():.4f}"
)

print("=" * 90)


# =============================================================================
# CLEANUP
# =============================================================================

del embeddings
del texts
del records
del token_lengths
del sample_norms

gc.collect()
torch.cuda.empty_cache()

print(
    "\nBGE-M3 optimization pilot completed."
)

BGE-M3 OPTIMIZATION PILOT — BATCH 64 / 2048 TOKENS
Sample size           : 5,000
Batch size             : 64
Max sequence length    : 2,048
Chunk shards found     : 151

Loading BGE-M3 tokenizer...
Tokenizer loaded.
Distributed chunks loaded: 5,000

Analysing token lengths...

------------------------------------------------------------------------------------------
TOKEN LENGTH DISTRIBUTION
------------------------------------------------------------------------------------------
Average tokens        : 660.9
P50                   : 719
P90                   : 1,010
P95                   : 1,139
P99                   : 1,436
Maximum               : 2,047
> 2048 tokens       : 0 (0.00%)
------------------------------------------------------------------------------------------

Generating embeddings...


Batches:   0%|          | 0/79 [00:00<?, ?it/s]


BGE-M3 OPTIMIZATION PILOT SUMMARY
Chunks embedded          : 5,000
Batch size               : 64
Max sequence length      : 2,048
Embedding dimensions     : 1,024
Embedding runtime        : 1475.49 sec
Throughput               : 3.39 chunks/sec
Throughput               : 203.32 chunks/min
GPU baseline memory      : 2.12 GB
GPU peak memory          : 8.22 GB
GPU memory after         : 2.12 GB
Estimated full runtime   : 123.48 hours
Estimated vector storage : 5.75 GB
Vector norm range        : 1.0000 - 1.0000

BGE-M3 optimization pilot completed.


##### **Observation — BGE-M3 Optimisation**

- **Batch 8 / 4,096:** 3.91 chunks/sec.
- **Batch 32 / 3,072:** 3.40 chunks/sec.
- **Batch 64 / 2,048:** 3.39 chunks/sec.
- Increasing batch size provided **no throughput improvement**.
- **2,048 tokens** covered the distributed sample without truncation.
- GPU memory increased to **8.22 GB**, but throughput remained flat.
- **Conclusion:** Batch size is not the primary bottleneck.
- **Next:** Test an optimised embedding inference pipeline before full-scale embedding.

#### **Opt 3 - Multi-Process / Pooling**

In [ ]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# BGE-M3 MULTI-PROCESS / POOL BENCHMARK
# =============================================================================

import gc
import json
import time
import threading
import subprocess
from pathlib import Path

import numpy as np
import psutil
import torch
from tqdm.auto import tqdm


# =============================================================================
# CONFIGURATION
# =============================================================================

PILOT_CHUNKS = 5_000
BATCH_SIZE = 64
POOL_CHUNK_SIZE = 1_000
MAX_SEQ_LENGTH = 2_048

RECONCILED_DIR = Path(
    "/content/drive/MyDrive/"
    "telecom_knowledge_base/processed/reconciled_chunks"
)

MODEL_NAME = "BAAI/bge-m3"


# =============================================================================
# LOAD DISTRIBUTED SAMPLE
# =============================================================================

print("=" * 90)
print("BGE-M3 MULTI-PROCESS / POOL BENCHMARK")
print("=" * 90)

print(
    f"Sample size            : {PILOT_CHUNKS:,}"
)

print(
    f"Batch size             : {BATCH_SIZE}"
)

print(
    f"Pool chunk size        : {POOL_CHUNK_SIZE:,}"
)

print(
    f"Max sequence length    : {MAX_SEQ_LENGTH:,}"
)

shards = sorted(
    RECONCILED_DIR.glob("chunks_*.jsonl")
)

records = []

base_per_shard = (
    PILOT_CHUNKS // len(shards)
)

remainder = (
    PILOT_CHUNKS % len(shards)
)

for shard_index, shard_path in enumerate(shards):

    target = (
        base_per_shard
        + (1 if shard_index < remainder else 0)
    )

    shard_records = []

    with open(
        shard_path,
        "r",
        encoding="utf-8",
    ) as file:

        for line in file:

            if not line.strip():
                continue

            shard_records.append(
                json.loads(line)
            )

            if len(shard_records) >= target:
                break

    records.extend(
        shard_records
    )


if len(records) != PILOT_CHUNKS:

    raise RuntimeError(
        f"Expected {PILOT_CHUNKS:,} chunks but loaded "
        f"{len(records):,}."
    )

texts = [
    record["text"]
    for record in records
]

print(
    f"Distributed chunks loaded : "
    f"{len(texts):,}"
)


# =============================================================================
# MODEL CONFIGURATION
# =============================================================================

model.max_seq_length = MAX_SEQ_LENGTH


# =============================================================================
# GPU MEMORY BASELINE
# =============================================================================

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

baseline_gpu_memory = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)

print(
    f"GPU baseline memory       : "
    f"{baseline_gpu_memory:.2f} GB"
)


# =============================================================================
# CPU / GPU MONITOR
# =============================================================================

monitor_stop = threading.Event()

cpu_samples = []
gpu_samples = []


def monitor_resources():

    while not monitor_stop.is_set():

        try:

            cpu_samples.append(
                psutil.cpu_percent(
                    interval=0.5
                )
            )

            result = subprocess.run(
                [
                    "nvidia-smi",
                    "--query-gpu=utilization.gpu",
                    "--format=csv,noheader,nounits",
                ],
                capture_output=True,
                text=True,
            )

            if result.returncode == 0:

                gpu_util = float(
                    result.stdout.strip().splitlines()[0]
                )

                gpu_samples.append(
                    gpu_util
                )

        except Exception:
            pass


monitor_thread = threading.Thread(
    target=monitor_resources,
    daemon=True,
)

monitor_thread.start()


# =============================================================================
# START REUSABLE MULTI-PROCESS POOL
# =============================================================================

print("\nStarting multi-process pool...")

pool_start = time.time()

pool = model.start_multi_process_pool(
    target_devices=["cuda:0"]
)

pool_start_time = (
    time.time() - pool_start
)

print(
    f"Pool startup time        : "
    f"{pool_start_time:.2f} sec"
)


# =============================================================================
# EMBEDDING BENCHMARK
# =============================================================================

print("\nGenerating embeddings...")

start_time = time.time()

embeddings = model.encode(
    texts,
    pool=pool,
    batch_size=BATCH_SIZE,
    chunk_size=POOL_CHUNK_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# Make sure GPU operations have completed.
torch.cuda.synchronize()

elapsed_seconds = (
    time.time() - start_time
)


# =============================================================================
# STOP MONITOR
# =============================================================================

monitor_stop.set()
monitor_thread.join(
    timeout=2
)


# =============================================================================
# STOP POOL
# =============================================================================

model.stop_multi_process_pool(
    pool
)


# =============================================================================
# METRICS
# =============================================================================

num_embeddings = (
    embeddings.shape[0]
)

embedding_dimension = (
    embeddings.shape[1]
)

throughput = (
    num_embeddings
    / elapsed_seconds
)

chunks_per_minute = (
    throughput * 60
)

peak_gpu_memory = (
    torch.cuda.max_memory_allocated(0)
    / (1024**3)
)

current_gpu_memory = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)

avg_cpu = (
    np.mean(cpu_samples)
    if cpu_samples
    else 0
)

max_cpu = (
    np.max(cpu_samples)
    if cpu_samples
    else 0
)

avg_gpu_util = (
    np.mean(gpu_samples)
    if gpu_samples
    else 0
)

max_gpu_util = (
    np.max(gpu_samples)
    if gpu_samples
    else 0
)


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("BGE-M3 MULTI-PROCESS / POOL SUMMARY")
print("=" * 90)

print(
    f"Chunks embedded          : "
    f"{num_embeddings:,}"
)

print(
    f"Batch size               : "
    f"{BATCH_SIZE}"
)

print(
    f"Pool chunk size          : "
    f"{POOL_CHUNK_SIZE:,}"
)

print(
    f"Max sequence length      : "
    f"{MAX_SEQ_LENGTH:,}"
)

print(
    f"Embedding dimensions     : "
    f"{embedding_dimension:,}"
)

print(
    f"Embedding runtime        : "
    f"{elapsed_seconds:.2f} sec"
)

print(
    f"Throughput               : "
    f"{throughput:.2f} chunks/sec"
)

print(
    f"Throughput               : "
    f"{chunks_per_minute:.2f} chunks/min"
)

print("-" * 90)

print(
    f"Average CPU utilisation  : "
    f"{avg_cpu:.1f}%"
)

print(
    f"Peak CPU utilisation     : "
    f"{max_cpu:.1f}%"
)

print(
    f"Average GPU utilisation  : "
    f"{avg_gpu_util:.1f}%"
)

print(
    f"Peak GPU utilisation     : "
    f"{max_gpu_util:.1f}%"
)

print("-" * 90)

print(
    f"GPU baseline memory      : "
    f"{baseline_gpu_memory:.2f} GB"
)

print(
    f"GPU peak memory          : "
    f"{peak_gpu_memory:.2f} GB"
)

print(
    f"GPU memory after         : "
    f"{current_gpu_memory:.2f} GB"
)

print("=" * 90)


# =============================================================================
# CLEANUP
# =============================================================================

del embeddings
del texts
del records
del cpu_samples
del gpu_samples

gc.collect()
torch.cuda.empty_cache()

print(
    "\nMulti-process / pool benchmark completed."
)

##### **Exploratory Result — Multi-Process Pilot Interrupted**

The multi-process / pooling experiment was manually interrupted after it failed to demonstrate a useful improvement trajectory. It was **not** used in the final embedding configuration. The saved traceback has been removed from the public notebook; the experiment and design decision are retained.

#### **Opt 4 - Local SSD + worker tokenisation + pinned memory**

In [17]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 2.10 — OPTIMIZED BGE-M3 EMBEDDING PIPELINE
#              LOCAL SSD + DATALOADER WORKERS + PINNED MEMORY
# =============================================================================

import gc
import json
import shutil
import time
import threading

from pathlib import Path

import numpy as np
import psutil
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm


# =============================================================================
# CONFIGURATION
# =============================================================================

PILOT_CHUNKS = 5_000

BATCH_SIZE = 128

NUM_WORKERS = 2

MAX_SEQ_LENGTH = 1_024

MODEL_NAME = "BAAI/bge-m3"

DRIVE_DIR = Path(
    "/content/drive/MyDrive/"
    "telecom_knowledge_base/processed/reconciled_chunks"
)

LOCAL_DIR = Path(
    "/content/reconciled_chunks"
)


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("BGE-M3 OPTIMIZED LOCAL-SSD EMBEDDING PILOT")
print("=" * 90)

print(
    f"Pilot chunks            : {PILOT_CHUNKS:,}"
)

print(
    f"Batch size              : {BATCH_SIZE}"
)

print(
    f"CPU workers             : {NUM_WORKERS}"
)

print(
    f"Max sequence length     : {MAX_SEQ_LENGTH}"
)

print(
    f"Model                   : {MODEL_NAME}"
)


# =============================================================================
# LOCAL SSD PREPARATION
# =============================================================================

if not DRIVE_DIR.exists():

    raise FileNotFoundError(
        f"Drive dataset not found:\n{DRIVE_DIR}"
    )

LOCAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nPreparing local SSD dataset...")

copy_start = time.time()

drive_shards = sorted(
    DRIVE_DIR.glob("chunks_*.jsonl")
)

for shard in tqdm(
    drive_shards,
    desc="Copying shards",
    unit="shard",
):

    destination = (
        LOCAL_DIR / shard.name
    )

    if (
        not destination.exists()
        or destination.stat().st_size
        != shard.stat().st_size
    ):

        shutil.copy2(
            shard,
            destination
        )

copy_time = (
    time.time() - copy_start
)

print(
    f"Local copy time         : "
    f"{copy_time:.2f} sec"
)

print(
    f"Local shards available  : "
    f"{len(list(LOCAL_DIR.glob('chunks_*.jsonl'))):,}"
)


# =============================================================================
# DATASET
# =============================================================================

class TextChunkDataset(Dataset):

    def __init__(
        self,
        data_dir,
        max_samples,
    ):

        self.texts = []

        shards = sorted(
            Path(data_dir).glob(
                "chunks_*.jsonl"
            )
        )

        for shard in shards:

            with open(
                shard,
                "r",
                encoding="utf-8",
            ) as file:

                for line in file:

                    if not line.strip():
                        continue

                    record = json.loads(
                        line
                    )

                    self.texts.append(
                        record["text"]
                    )

                    if (
                        len(self.texts)
                        >= max_samples
                    ):
                        break

            if (
                len(self.texts)
                >= max_samples
            ):
                break

    def __len__(self):

        return len(self.texts)

    def __getitem__(self, index):

        return self.texts[index]


dataset = TextChunkDataset(
    LOCAL_DIR,
    max_samples=PILOT_CHUNKS,
)

print(
    f"Chunks loaded           : "
    f"{len(dataset):,}"
)


# =============================================================================
# TOKENIZER
# =============================================================================

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

print(
    f"Fast tokenizer          : "
    f"{tokenizer.is_fast}"
)


# =============================================================================
# WORKER COLLATE FUNCTION
# =============================================================================
# IMPORTANT:
# Tokenization occurs inside DataLoader workers.
# This is the key change intended to reduce CPU -> GPU starvation.

def collate_fn(batch):

    return tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )


# =============================================================================
# DATALOADER
# =============================================================================

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    collate_fn=collate_fn,
)


# =============================================================================
# MODEL
# =============================================================================

print("\nLoading BGE-M3 base model...")

hf_model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)

hf_model = (
    hf_model
    .eval()
    .cuda()
)


# =============================================================================
# RESOURCE MONITOR
# =============================================================================

monitor_stop = threading.Event()

cpu_samples = []
gpu_samples = []


def monitor_resources():

    while not monitor_stop.is_set():

        try:

            cpu_samples.append(
                psutil.cpu_percent(
                    interval=0.5
                )
            )

            if torch.cuda.is_available():

                gpu_samples.append(
                    torch.cuda.utilization()
                    if hasattr(
                        torch.cuda,
                        "utilization"
                    )
                    else 0
                )

        except Exception:

            pass


monitor_thread = threading.Thread(
    target=monitor_resources,
    daemon=True,
)

monitor_thread.start()


# =============================================================================
# GPU RESET
# =============================================================================

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

baseline_memory = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)

print(
    f"\nGPU baseline memory     : "
    f"{baseline_memory:.2f} GB"
)

print(
    "\nStarting optimized embedding..."
)


# =============================================================================
# EMBEDDING
# =============================================================================

start_time = time.time()

all_embeddings = []

progress = tqdm(
    loader,
    total=len(loader),
    desc="BGE-M3 embedding",
    unit="batch",
    dynamic_ncols=True,
)

with torch.inference_mode():

    for batch in progress:

        # Non-blocking transfer from pinned memory.
        batch = {
            key: value.cuda(
                non_blocking=True
            )
            for key, value in batch.items()
        }

        outputs = hf_model(
            **batch
        )

        # BGE-M3 uses CLS pooling.
        embeddings = (
            outputs
            .last_hidden_state[:, 0]
        )

        # L2 normalisation.
        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1,
        )

        all_embeddings.append(
            embeddings.cpu()
        )

        processed = min(
            len(all_embeddings)
            * BATCH_SIZE,
            len(dataset)
        )

        elapsed = (
            time.time()
            - start_time
        )

        current_rate = (
            processed / elapsed
            if elapsed > 0
            else 0
        )

        progress.set_postfix(
            chunks=f"{processed:,}",
            rate=f"{current_rate:.2f}/s",
        )


progress.close()

torch.cuda.synchronize()

elapsed_seconds = (
    time.time()
    - start_time
)


# =============================================================================
# STOP MONITOR
# =============================================================================

monitor_stop.set()

monitor_thread.join(
    timeout=2
)


# =============================================================================
# COMBINE EMBEDDINGS
# =============================================================================

embeddings = torch.cat(
    all_embeddings,
    dim=0,
).numpy()


# =============================================================================
# METRICS
# =============================================================================

throughput = (
    len(embeddings)
    / elapsed_seconds
)

chunks_per_minute = (
    throughput * 60
)

peak_gpu_memory = (
    torch.cuda.max_memory_allocated(0)
    / (1024**3)
)

current_gpu_memory = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)

avg_cpu = (
    np.mean(cpu_samples)
    if cpu_samples
    else 0
)

peak_cpu = (
    np.max(cpu_samples)
    if cpu_samples
    else 0
)


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("OPTIMIZED BGE-M3 PILOT SUMMARY")
print("=" * 90)

print(
    f"Chunks embedded          : "
    f"{len(embeddings):,}"
)

print(
    f"Batch size               : "
    f"{BATCH_SIZE}"
)

print(
    f"CPU workers              : "
    f"{NUM_WORKERS}"
)

print(
    f"Max sequence length      : "
    f"{MAX_SEQ_LENGTH}"
)

print(
    f"Embedding dimensions     : "
    f"{embeddings.shape[1]}"
)

print(
    f"Embedding runtime        : "
    f"{elapsed_seconds:.2f} sec"
)

print(
    f"Throughput               : "
    f"{throughput:.2f} chunks/sec"
)

print(
    f"Throughput               : "
    f"{chunks_per_minute:.2f} chunks/min"
)

print(
    f"GPU baseline memory      : "
    f"{baseline_memory:.2f} GB"
)

print(
    f"GPU peak memory          : "
    f"{peak_gpu_memory:.2f} GB"
)

print(
    f"GPU memory after         : "
    f"{current_gpu_memory:.2f} GB"
)

print(
    f"Average CPU utilisation  : "
    f"{avg_cpu:.1f}%"
)

print(
    f"Peak CPU utilisation     : "
    f"{peak_cpu:.1f}%"
)

print("=" * 90)


# =============================================================================
# CLEANUP
# =============================================================================

del all_embeddings
del embeddings
del dataset
del loader
del hf_model
del cpu_samples
del gpu_samples

gc.collect()

torch.cuda.empty_cache()

print(
    "\nOptimized BGE-M3 pilot completed."
)

BGE-M3 OPTIMIZED LOCAL-SSD EMBEDDING PILOT
Pilot chunks            : 5,000
Batch size              : 128
CPU workers             : 2
Max sequence length     : 1024
Model                   : BAAI/bge-m3

Preparing local SSD dataset...


Copying shards:   0%|          | 0/151 [00:00<?, ?shard/s]

Local copy time         : 66.98 sec
Local shards available  : 151
Chunks loaded           : 5,000

Loading tokenizer...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Fast tokenizer          : True

Loading BGE-M3 base model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


GPU baseline memory     : 1.07 GB

Starting optimized embedding...


BGE-M3 embedding:   0%|          | 0/40 [00:00<?, ?batch/s]


OPTIMIZED BGE-M3 PILOT SUMMARY
Chunks embedded          : 5,000
Batch size               : 128
CPU workers              : 2
Max sequence length      : 1024
Embedding dimensions     : 1024
Embedding runtime        : 270.41 sec
Throughput               : 18.49 chunks/sec
Throughput               : 1109.41 chunks/min
GPU baseline memory      : 1.07 GB
GPU peak memory          : 4.19 GB
GPU memory after         : 1.08 GB
Average CPU utilisation  : 74.9%
Peak CPU utilisation     : 100.0%

Optimized BGE-M3 pilot completed.


##### **Observation — Optimized BGE-M3 Pipeline**

- **5,000 chunks** embedded successfully.
- **Throughput:** 18.49 chunks/sec.
- **~5.4× faster** than the previous 3.39 chunks/sec baseline.
- **GPU peak:** 4.19 GB / 14.56 GB.
- **CPU average:** 74.9%, indicating effective CPU-side preprocessing.
- Local SSD + DataLoader workers + pinned memory materially improved throughput.

#### **Opt 5 - BGE-M3 FINAL SEQUENCE-LENGTH OPTIMIZATION PILOT: LOCAL SSD + DATALOADER WORKERS + PINNED MEMORY**

In [18]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# BGE-M3 FINAL SEQUENCE-LENGTH OPTIMIZATION PILOT
# LOCAL SSD + DATALOADER WORKERS + PINNED MEMORY
# =============================================================================

import gc
import json
import shutil
import time
import threading

from pathlib import Path

import numpy as np
import psutil
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm


# =============================================================================
# CONFIGURATION
# =============================================================================

PILOT_CHUNKS = 5_000

BATCH_SIZE = 128
NUM_WORKERS = 2

MAX_SEQ_LENGTH = 1_280

MODEL_NAME = "BAAI/bge-m3"

DRIVE_DIR = Path(
    "/content/drive/MyDrive/"
    "telecom_knowledge_base/processed/reconciled_chunks"
)

LOCAL_DIR = Path(
    "/content/reconciled_chunks"
)


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("BGE-M3 FINAL SEQUENCE-LENGTH OPTIMIZATION PILOT")
print("=" * 90)

print(f"Pilot chunks            : {PILOT_CHUNKS:,}")
print(f"Batch size              : {BATCH_SIZE}")
print(f"CPU workers             : {NUM_WORKERS}")
print(f"Max sequence length     : {MAX_SEQ_LENGTH}")
print(f"Model                   : {MODEL_NAME}")


# =============================================================================
# VERIFY DRIVE
# =============================================================================

if not DRIVE_DIR.exists():
    raise FileNotFoundError(
        f"Drive dataset not found:\n{DRIVE_DIR}"
    )


# =============================================================================
# LOCAL SSD PREPARATION
# =============================================================================

LOCAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nPreparing local SSD dataset...")

copy_start = time.time()

drive_shards = sorted(
    DRIVE_DIR.glob("chunks_*.jsonl")
)

copied = 0
reused = 0

for shard in tqdm(
    drive_shards,
    desc="Preparing local shards",
    unit="shard",
):

    destination = LOCAL_DIR / shard.name

    if (
        destination.exists()
        and destination.stat().st_size == shard.stat().st_size
    ):
        reused += 1
        continue

    shutil.copy2(
        shard,
        destination
    )

    copied += 1

copy_time = time.time() - copy_start

print(f"Copied shards           : {copied:,}")
print(f"Reused local shards     : {reused:,}")
print(f"Preparation time        : {copy_time:.2f} sec")


# =============================================================================
# DATASET
# =============================================================================

class TextChunkDataset(Dataset):

    def __init__(
        self,
        data_dir,
        max_samples,
    ):

        self.texts = []

        shards = sorted(
            Path(data_dir).glob("chunks_*.jsonl")
        )

        for shard in shards:

            with open(
                shard,
                "r",
                encoding="utf-8",
            ) as file:

                for line in file:

                    if not line.strip():
                        continue

                    record = json.loads(line)

                    self.texts.append(
                        record["text"]
                    )

                    if (
                        len(self.texts)
                        >= max_samples
                    ):
                        break

            if (
                len(self.texts)
                >= max_samples
            ):
                break

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return self.texts[index]


dataset = TextChunkDataset(
    LOCAL_DIR,
    max_samples=PILOT_CHUNKS,
)

if len(dataset) != PILOT_CHUNKS:
    raise RuntimeError(
        f"Expected {PILOT_CHUNKS:,} chunks, "
        f"loaded {len(dataset):,}."
    )

print(
    f"Chunks loaded           : {len(dataset):,}"
)


# =============================================================================
# TOKENIZER
# =============================================================================

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

print(
    f"Fast tokenizer          : "
    f"{tokenizer.is_fast}"
)


# =============================================================================
# WORKER TOKENIZATION
# =============================================================================

def collate_fn(batch):

    return tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )


# =============================================================================
# DATALOADER
# =============================================================================

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
    collate_fn=collate_fn,
)


# =============================================================================
# MODEL
# =============================================================================

print("\nLoading BGE-M3 base model...")

hf_model = AutoModel.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
)

hf_model = (
    hf_model
    .eval()
    .cuda()
)


# =============================================================================
# RESOURCE MONITOR
# =============================================================================

monitor_stop = threading.Event()

cpu_samples = []
gpu_samples = []


def monitor_resources():

    while not monitor_stop.is_set():

        try:

            cpu_samples.append(
                psutil.cpu_percent(interval=0.5)
            )

            gpu_result = torch.cuda.utilization()

            gpu_samples.append(
                gpu_result
            )

        except Exception:
            pass


monitor_thread = threading.Thread(
    target=monitor_resources,
    daemon=True,
)

monitor_thread.start()


# =============================================================================
# GPU BASELINE
# =============================================================================

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

baseline_memory = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)

print(
    f"GPU baseline memory     : "
    f"{baseline_memory:.2f} GB"
)

print("\nStarting optimized embedding...")


# =============================================================================
# EMBEDDING
# =============================================================================

start_time = time.time()

all_embeddings = []

progress = tqdm(
    loader,
    total=len(loader),
    desc="BGE-M3 embedding",
    unit="batch",
    dynamic_ncols=True,
)

with torch.inference_mode():

    for batch in progress:

        batch = {
            key: value.cuda(
                non_blocking=True
            )
            for key, value in batch.items()
        }

        outputs = hf_model(
            **batch
        )

        # BGE-M3: CLS pooling
        embeddings = (
            outputs
            .last_hidden_state[:, 0]
        )

        # L2 normalization
        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1,
        )

        # Keep pilot output only in memory
        all_embeddings.append(
            embeddings.cpu()
        )

        processed = min(
            len(all_embeddings) * BATCH_SIZE,
            len(dataset)
        )

        elapsed = (
            time.time() - start_time
        )

        rate = (
            processed / elapsed
            if elapsed > 0
            else 0
        )

        progress.set_postfix(
            chunks=f"{processed:,}",
            rate=f"{rate:.2f}/s",
        )

progress.close()

torch.cuda.synchronize()

elapsed_seconds = (
    time.time()
    - start_time
)


# =============================================================================
# STOP MONITOR
# =============================================================================

monitor_stop.set()

monitor_thread.join(
    timeout=2
)


# =============================================================================
# COMBINE PILOT EMBEDDINGS
# =============================================================================

embeddings = torch.cat(
    all_embeddings,
    dim=0,
).numpy()


# =============================================================================
# METRICS
# =============================================================================

throughput = (
    len(embeddings)
    / elapsed_seconds
)

chunks_per_minute = (
    throughput * 60
)

peak_gpu_memory = (
    torch.cuda.max_memory_allocated(0)
    / (1024**3)
)

current_gpu_memory = (
    torch.cuda.memory_allocated(0)
    / (1024**3)
)

average_cpu = (
    np.mean(cpu_samples)
    if cpu_samples
    else 0
)

peak_cpu = (
    np.max(cpu_samples)
    if cpu_samples
    else 0
)

average_gpu = (
    np.mean(gpu_samples)
    if gpu_samples
    else 0
)

peak_gpu = (
    np.max(gpu_samples)
    if gpu_samples
    else 0
)


# =============================================================================
# FULL-CORPUS ESTIMATE
# =============================================================================

TOTAL_CHUNKS = 1_506_367

estimated_hours = (
    TOTAL_CHUNKS
    / throughput
    / 3600
)

estimated_float32_gb = (
    TOTAL_CHUNKS
    * embeddings.shape[1]
    * 4
    / (1024**3)
)

estimated_float16_gb = (
    TOTAL_CHUNKS
    * embeddings.shape[1]
    * 2
    / (1024**3)
)


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("BGE-M3 FINAL OPTIMIZATION PILOT SUMMARY")
print("=" * 90)

print(
    f"Chunks embedded          : "
    f"{len(embeddings):,}"
)

print(
    f"Batch size               : "
    f"{BATCH_SIZE}"
)

print(
    f"CPU workers              : "
    f"{NUM_WORKERS}"
)

print(
    f"Max sequence length      : "
    f"{MAX_SEQ_LENGTH}"
)

print(
    f"Embedding dimensions     : "
    f"{embeddings.shape[1]}"
)

print(
    f"Embedding runtime        : "
    f"{elapsed_seconds:.2f} sec"
)

print(
    f"Throughput               : "
    f"{throughput:.2f} chunks/sec"
)

print(
    f"Throughput               : "
    f"{chunks_per_minute:.2f} chunks/min"
)

print("-" * 90)

print(
    f"Average CPU utilisation  : "
    f"{average_cpu:.1f}%"
)

print(
    f"Peak CPU utilisation     : "
    f"{peak_cpu:.1f}%"
)

print(
    f"Average GPU utilisation  : "
    f"{average_gpu:.1f}%"
)

print(
    f"Peak GPU utilisation     : "
    f"{peak_gpu:.1f}%"
)

print("-" * 90)

print(
    f"GPU baseline memory      : "
    f"{baseline_memory:.2f} GB"
)

print(
    f"GPU peak memory          : "
    f"{peak_gpu_memory:.2f} GB"
)

print(
    f"GPU memory after         : "
    f"{current_gpu_memory:.2f} GB"
)

print("-" * 90)

print(
    f"Estimated full runtime   : "
    f"{estimated_hours:.2f} hours"
)

print(
    f"Estimated FP32 storage   : "
    f"{estimated_float32_gb:.2f} GB"
)

print(
    f"Estimated FP16 storage   : "
    f"{estimated_float16_gb:.2f} GB"
)

print("=" * 90)


# =============================================================================
# CLEANUP
# =============================================================================

del all_embeddings
del embeddings
del dataset
del loader
del hf_model
del cpu_samples
del gpu_samples

gc.collect()
torch.cuda.empty_cache()

print(
    "\nBGE-M3 1,280-token optimization pilot completed."
)

BGE-M3 FINAL SEQUENCE-LENGTH OPTIMIZATION PILOT
Pilot chunks            : 5,000
Batch size              : 128
CPU workers             : 2
Max sequence length     : 1280
Model                   : BAAI/bge-m3

Preparing local SSD dataset...


Preparing local shards:   0%|          | 0/151 [00:00<?, ?shard/s]

Copied shards           : 0
Reused local shards     : 151
Preparation time        : 0.09 sec
Chunks loaded           : 5,000

Loading tokenizer...
Fast tokenizer          : True

Loading BGE-M3 base model...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

GPU baseline memory     : 1.08 GB

Starting optimized embedding...


BGE-M3 embedding:   0%|          | 0/40 [00:00<?, ?batch/s]


BGE-M3 FINAL OPTIMIZATION PILOT SUMMARY
Chunks embedded          : 5,000
Batch size               : 128
CPU workers              : 2
Max sequence length      : 1280
Embedding dimensions     : 1024
Embedding runtime        : 344.32 sec
Throughput               : 14.52 chunks/sec
Throughput               : 871.28 chunks/min
------------------------------------------------------------------------------------------
Average CPU utilisation  : 74.3%
Peak CPU utilisation     : 100.0%
Average GPU utilisation  : 98.8%
Peak GPU utilisation     : 100.0%
------------------------------------------------------------------------------------------
GPU baseline memory      : 1.08 GB
GPU peak memory          : 5.02 GB
GPU memory after         : 1.08 GB
------------------------------------------------------------------------------------------
Estimated full runtime   : 28.82 hours
Estimated FP32 storage   : 5.75 GB
Estimated FP16 storage   : 2.87 GB

BGE-M3 1,280-token optimization pilot completed.


##### **Final BGE-M3 Embedding Configuration**

- **Model:** `BAAI/bge-m3`
- **Sequence length:** 1,280 tokens
- **Batch size:** 128
- **CPU workers:** 2
- **Precision:** FP16
- **Data:** Local Colab SSD
- **GPU utilisation:** ~99% average
- **Vector dimensions:** 1,024
- **Estimated FP16 storage:** ~2.87 GB
- **Estimated full runtime:** ~28.8 hours
- **Full dataset:** 1,506,367 unique chunks
- **Next:** Full resumable embedding with 10,000-chunk persistence shards

# **Embedding & Vector Storage Architecture — Design Alignment Notes**

- **Embedding model:** `BAAI/bge-m3`
- **Embedding dimensions:** 1,024
- **Embedding generation:** GPU-based, batched and resumable
- **Embedding persistence:** Google Drive
- **Embedding strategy:** Process in batches/shards rather than one large in-memory dataset
- **Vector index:** FAISS
- **FAISS persistence:** Google Drive
- **Index construction:** Build from persisted embedding shards
- **Chunk-to-vector mapping:** Preserve `vector_id → chunk_id → document_id → source/provenance`
- **Recovery:** Checkpoints enable embedding to resume after Colab interruption
- **Retrieval flow:** Query → BGE-M3 embedding → FAISS → vector IDs → chunk metadata → context
- **Advanced retrieval:** Hybrid search and reranking to be evaluated after baseline RAG
- **MCP:** May expose retrieval/search functions and later external tools
- **Raw corpus:** Not required for normal inference once the validated chunk, embedding and index layers are complete
- **Current status:** 5,000-chunk BGE-M3 pilot in progress

## **Resumable Embedding Pipeline**

In [ ]:
# =============================================================================
# MODULE 2 — TELECOM RAG IMPLEMENTATION
# CELL 2.12 — FULL RESUMABLE BGE-M3 EMBEDDING
# =============================================================================
#
# Final configuration:
#   Model          : BAAI/bge-m3
#   Batch size     : 128
#   Max sequence   : 1280
#   CPU workers    : 2
#   Precision      : FP16
#   Input          : Reconciled chunk shards
#   Output         : FP16 embedding shards + metadata
#   Persistence    : Google Drive
#   Resume         : Shard-level checkpoint
# =============================================================================

import gc
import json
import shutil
import time

from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm


# =============================================================================
# CONFIGURATION
# =============================================================================

MODEL_NAME = "BAAI/bge-m3"

BATCH_SIZE = 128
NUM_WORKERS = 2
MAX_SEQ_LENGTH = 1280

CHUNKS_PER_SHARD = 10_000

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/telecom_knowledge_base"
)

DRIVE_CHUNK_DIR = (
    DRIVE_ROOT
    / "processed"
    / "reconciled_chunks"
)

DRIVE_EMBED_DIR = (
    DRIVE_ROOT
    / "processed"
    / "embeddings"
)

LOCAL_CHUNK_DIR = Path(
    "/content/reconciled_chunks"
)

CHECKPOINT_PATH = (
    DRIVE_EMBED_DIR
    / "embedding_checkpoint.json"
)

MANIFEST_PATH = (
    DRIVE_EMBED_DIR
    / "embedding_manifest.json"
)

TOTAL_CHUNKS = 1_506_367


# =============================================================================
# CREATE DIRECTORIES
# =============================================================================

DRIVE_EMBED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

LOCAL_CHUNK_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# HEADER
# =============================================================================

print("=" * 90)
print("FULL RESUMABLE BGE-M3 EMBEDDING")
print("=" * 90)

print(f"Model                  : {MODEL_NAME}")
print(f"Batch size             : {BATCH_SIZE}")
print(f"CPU workers            : {NUM_WORKERS}")
print(f"Max sequence length    : {MAX_SEQ_LENGTH}")
print(f"Total chunks           : {TOTAL_CHUNKS:,}")
print(f"Embedding persistence  : {DRIVE_EMBED_DIR}")


# =============================================================================
# DISCOVER RECONCILED SHARDS
# =============================================================================

drive_shards = sorted(
    DRIVE_CHUNK_DIR.glob("chunks_*.jsonl")
)

if not drive_shards:
    raise FileNotFoundError(
        f"No reconciled shards found:\n{DRIVE_CHUNK_DIR}"
    )

print(
    f"Input shards            : {len(drive_shards):,}"
)


# =============================================================================
# COPY SHARDS TO LOCAL SSD
# =============================================================================
# Local SSD is used for the hot embedding path.
# Existing identical files are reused.

print("\nPreparing local SSD...")

copy_start = time.time()

copied = 0
reused = 0

for shard in tqdm(
    drive_shards,
    desc="Preparing local shards",
    unit="shard",
):

    local_path = (
        LOCAL_CHUNK_DIR
        / shard.name
    )

    if (
        local_path.exists()
        and local_path.stat().st_size
        == shard.stat().st_size
    ):
        reused += 1
        continue

    shutil.copy2(
        shard,
        local_path
    )

    copied += 1

copy_elapsed = (
    time.time() - copy_start
)

print(
    f"Copied shards          : {copied:,}"
)

print(
    f"Reused shards          : {reused:,}"
)

print(
    f"Preparation time       : "
    f"{copy_elapsed:.2f} sec"
)


# =============================================================================
# CHECKPOINT
# =============================================================================

if CHECKPOINT_PATH.exists():

    with open(
        CHECKPOINT_PATH,
        "r",
        encoding="utf-8"
    ) as file:

        checkpoint = json.load(file)

else:

    checkpoint = {
        "completed_shards": [],
        "completed_chunks": 0,
        "last_completed_shard": None,
        "status": "INITIALIZED",
    }


completed_shards = set(
    checkpoint.get(
        "completed_shards",
        []
    )
)

completed_chunks = checkpoint.get(
    "completed_chunks",
    0
)

print(
    f"Previously completed    : "
    f"{len(completed_shards):,} shards"
)

print(
    f"Previously embedded      : "
    f"{completed_chunks:,} chunks"
)


# =============================================================================
# TOKENIZER
# =============================================================================

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

print(
    f"Fast tokenizer           : "
    f"{tokenizer.is_fast}"
)


# =============================================================================
# DATASET
# =============================================================================

class TextChunkDataset(Dataset):

    def __init__(self, records):

        self.records = records

    def __len__(self):

        return len(self.records)

    def __getitem__(self, index):

        return self.records[index]["text"]


# =============================================================================
# COLLATE FUNCTION
# =============================================================================

def collate_fn(batch):

    return tokenizer(
        batch,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )


# =============================================================================
# MODEL
# =============================================================================

print("\nLoading BGE-M3...")

hf_model = AutoModel.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
)

hf_model = (
    hf_model
    .eval()
    .cuda()
)

torch.cuda.empty_cache()

print(
    f"GPU                   : "
    f"{torch.cuda.get_device_name(0)}"
)


# =============================================================================
# SHARD PROCESSING
# =============================================================================

run_start = time.time()

for shard_number, drive_shard in enumerate(
    drive_shards,
    start=1
):

    shard_name = drive_shard.name

    # -------------------------------------------------------------------------
    # Resume check
    # -------------------------------------------------------------------------

    if shard_name in completed_shards:

        print(
            f"\nSKIP {shard_name} — already completed."
        )

        continue


    print("\n" + "=" * 90)

    print(
        f"PROCESSING SHARD {shard_number}/"
        f"{len(drive_shards)} : "
        f"{shard_name}"
    )

    print("=" * 90)


    # -------------------------------------------------------------------------
    # Load records
    # -------------------------------------------------------------------------

    local_shard = (
        LOCAL_CHUNK_DIR
        / shard_name
    )

    records = []

    with open(
        local_shard,
        "r",
        encoding="utf-8"
    ) as file:

        for line in file:

            if line.strip():

                records.append(
                    json.loads(line)
                )


    shard_chunk_count = len(
        records
    )

    print(
        f"Chunks in shard        : "
        f"{shard_chunk_count:,}"
    )


    # -------------------------------------------------------------------------
    # DataLoader
    # -------------------------------------------------------------------------

    dataset = TextChunkDataset(
        records
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(
            NUM_WORKERS > 0
        ),
        prefetch_factor=2,
        collate_fn=collate_fn,
    )


    # -------------------------------------------------------------------------
    # GPU memory reset
    # -------------------------------------------------------------------------

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()


    # -------------------------------------------------------------------------
    # Embed shard
    # -------------------------------------------------------------------------

    shard_start = time.time()

    shard_embeddings = []

    progress = tqdm(
        loader,
        total=len(loader),
        desc=shard_name,
        unit="batch",
        dynamic_ncols=True,
    )


    with torch.inference_mode():

        for batch in progress:

            batch = {
                key: value.cuda(
                    non_blocking=True
                )
                for key, value in batch.items()
            }

            outputs = hf_model(
                **batch
            )

            # BGE-M3 CLS pooling
            embeddings = (
                outputs
                .last_hidden_state[:, 0]
            )

            # L2 normalize
            embeddings = (
                torch.nn.functional.normalize(
                    embeddings,
                    p=2,
                    dim=1
                )
            )

            # Store compact FP16 representation
            embeddings = (
                embeddings
                .cpu()
                .numpy()
                .astype(
                    np.float16,
                    copy=False
                )
            )

            shard_embeddings.append(
                embeddings
            )

            processed = min(
                sum(
                    x.shape[0]
                    for x in shard_embeddings
                ),
                shard_chunk_count
            )

            elapsed = (
                time.time()
                - shard_start
            )

            rate = (
                processed / elapsed
                if elapsed > 0
                else 0
            )

            progress.set_postfix(
                chunks=f"{processed:,}",
                rate=f"{rate:.2f}/s",
            )


    progress.close()


    # -------------------------------------------------------------------------
    # Combine shard embeddings
    # -------------------------------------------------------------------------

    shard_matrix = np.concatenate(
        shard_embeddings,
        axis=0
    )

    del shard_embeddings


    # -------------------------------------------------------------------------
    # Prepare output paths
    # -------------------------------------------------------------------------

    embedding_output = (
        DRIVE_EMBED_DIR
        / shard_name.replace(
            ".jsonl",
            ".npy"
        )
    )

    metadata_output = (
        DRIVE_EMBED_DIR
        / shard_name.replace(
            ".jsonl",
            "_metadata.jsonl"
        )
    )

    embedding_temp = (
        DRIVE_EMBED_DIR
        / (
            shard_name.replace(
                ".jsonl",
                ".tmp.npy"
            )
        )
    )

    metadata_temp = (
        DRIVE_EMBED_DIR
        / (
            shard_name.replace(
                ".jsonl",
                "_metadata.tmp.jsonl"
            )
        )
    )


    # -------------------------------------------------------------------------
    # Persist embeddings
    # -------------------------------------------------------------------------

    np.save(
        embedding_temp,
        shard_matrix
    )

    # Atomic rename after successful write
    embedding_temp.replace(
        embedding_output
    )


    # -------------------------------------------------------------------------
    # Persist metadata mapping
    # -------------------------------------------------------------------------

    with open(
        metadata_temp,
        "w",
        encoding="utf-8"
    ) as file:

        for record in records:

            metadata = {
                "chunk_id": record.get(
                    "chunk_id"
                ),
                "document_id": record.get(
                    "document_id"
                ),
                "source": record.get(
                    "source"
                ),
                "title": record.get(
                    "title"
                ),
                "path": record.get(
                    "path"
                ),
            }

            file.write(
                json.dumps(
                    metadata,
                    ensure_ascii=False
                ) + "\n"
            )

    metadata_temp.replace(
        metadata_output
    )


    # -------------------------------------------------------------------------
    # Update checkpoint
    # -------------------------------------------------------------------------

    shard_elapsed = (
        time.time()
        - shard_start
    )

    completed_shards.add(
        shard_name
    )

    completed_chunks += (
        shard_chunk_count
    )

    checkpoint = {

        "status": "IN_PROGRESS",

        "total_chunks": TOTAL_CHUNKS,

        "completed_chunks":
            completed_chunks,

        "completed_shards":
            sorted(completed_shards),

        "total_shards":
            len(drive_shards),

        "last_completed_shard":
            shard_name,

        "model":
            MODEL_NAME,

        "batch_size":
            BATCH_SIZE,

        "max_sequence_length":
            MAX_SEQ_LENGTH,

        "embedding_dimensions":
            int(shard_matrix.shape[1]),

        "embedding_dtype":
            "float16",

        "updated_utc":
            time.strftime(
                "%Y-%m-%dT%H:%M:%SZ",
                time.gmtime()
            ),
    }


    with open(
        CHECKPOINT_PATH,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            checkpoint,
            file,
            indent=2
        )


    # -------------------------------------------------------------------------
    # Shard summary
    # -------------------------------------------------------------------------

    shard_rate = (
        shard_chunk_count
        / shard_elapsed
    )

    print("\n" + "-" * 90)

    print(
        f"Shard completed         : "
        f"{shard_name}"
    )

    print(
        f"Chunks embedded         : "
        f"{shard_chunk_count:,}"
    )

    print(
        f"Shard throughput        : "
        f"{shard_rate:.2f} chunks/sec"
    )

    print(
        f"Total embedded          : "
        f"{completed_chunks:,} / "
        f"{TOTAL_CHUNKS:,}"
    )

    print(
        f"Output embeddings       : "
        f"{embedding_output}"
    )

    print(
        f"Checkpoint              : "
        f"{CHECKPOINT_PATH}"
    )

    print("-" * 90)


    # -------------------------------------------------------------------------
    # Cleanup shard memory
    # -------------------------------------------------------------------------

    del records
    del dataset
    del loader
    del shard_matrix

    gc.collect()

    torch.cuda.empty_cache()


# =============================================================================
# COMPLETION
# =============================================================================

total_elapsed = (
    time.time()
    - run_start
)

checkpoint["status"] = (
    "COMPLETED"
)

checkpoint["completed_chunks"] = (
    TOTAL_CHUNKS
)

checkpoint["updated_utc"] = (
    time.strftime(
        "%Y-%m-%dT%H:%M:%SZ",
        time.gmtime()
    )
)


with open(
    CHECKPOINT_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        checkpoint,
        file,
        indent=2
    )


# =============================================================================
# EMBEDDING MANIFEST
# =============================================================================

manifest = {

    "model": MODEL_NAME,

    "embedding_dimensions": 1024,

    "embedding_dtype": "float16",

    "max_sequence_length":
        MAX_SEQ_LENGTH,

    "batch_size":
        BATCH_SIZE,

    "cpu_workers":
        NUM_WORKERS,

    "total_chunks":
        TOTAL_CHUNKS,

    "total_shards":
        len(drive_shards),

    "embedding_directory":
        str(DRIVE_EMBED_DIR),

    "checkpoint":
        str(CHECKPOINT_PATH),

    "status":
        "COMPLETED",
}


with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        indent=2
    )


# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("FULL BGE-M3 EMBEDDING RUN COMPLETED")
print("=" * 90)

print(
    f"Total chunks            : "
    f"{TOTAL_CHUNKS:,}"
)

print(
    f"Total shards             : "
    f"{len(drive_shards):,}"
)

print(
    f"Embedding dimensions     : "
    f"1,024"
)

print(
    f"Embedding dtype          : "
    f"float16"
)

print(
    f"Total runtime            : "
    f"{total_elapsed / 3600:.2f} hours"
)

print(
    f"Embedding output         : "
    f"{DRIVE_EMBED_DIR}"
)

print(
    f"Checkpoint               : "
    f"{CHECKPOINT_PATH}"
)

print(
    f"Manifest                 : "
    f"{MANIFEST_PATH}"
)

print("=" * 90)
print("EMBEDDING PIPELINE : COMPLETED")
print("=" * 90)


# =============================================================================
# RELEASE GPU
# =============================================================================

del hf_model

gc.collect()

torch.cuda.empty_cache()

print(
    "\nBGE-M3 model released from GPU."
)

FULL RESUMABLE BGE-M3 EMBEDDING
Model                  : BAAI/bge-m3
Batch size             : 128
CPU workers            : 2
Max sequence length    : 1280
Total chunks           : 1,506,367
Embedding persistence  : /content/drive/MyDrive/telecom_knowledge_base/processed/embeddings
Input shards            : 151

Preparing local SSD...


Preparing local shards:   0%|          | 0/151 [00:00<?, ?shard/s]

Copied shards          : 0
Reused shards          : 151
Preparation time       : 0.06 sec
Previously completed    : 0 shards
Previously embedded      : 0 chunks

Loading tokenizer...
Fast tokenizer           : True

Loading BGE-M3...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

GPU                   : Tesla T4

PROCESSING SHARD 1/151 : chunks_000001.jsonl
Chunks in shard        : 10,000


chunks_000001.jsonl:   0%|          | 0/79 [00:00<?, ?batch/s]


------------------------------------------------------------------------------------------
Shard completed         : chunks_000001.jsonl
Chunks embedded         : 10,000
Shard throughput        : 14.62 chunks/sec
Total embedded          : 10,000 / 1,506,367
Output embeddings       : /content/drive/MyDrive/telecom_knowledge_base/processed/embeddings/chunks_000001.npy
Checkpoint              : /content/drive/MyDrive/telecom_knowledge_base/processed/embeddings/embedding_checkpoint.json
------------------------------------------------------------------------------------------

PROCESSING SHARD 2/151 : chunks_000002.jsonl
Chunks in shard        : 10,000


chunks_000002.jsonl:   0%|          | 0/79 [00:00<?, ?batch/s]

In [ ]:
# =============================================================================
# KAGGLE — BGE-M3 DUAL-T4 OPTIMIZATION PILOT
# =============================================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer


# =============================================================================
# CONFIGURATION
# =============================================================================

MODEL_NAME = "BAAI/bge-m3"

PILOT_CHUNKS = 5_000

MAX_SEQ_LENGTH = 1_280

# Batch size PER GPU
BATCH_SIZE = 128

# Work dispatched to each worker process
POOL_CHUNK_SIZE = 1_000

INPUT_DIR = Path(
    "/kaggle/input/datasets/"
    "cliffordimaguezegie/"
    "telecom-reconciled-chunks"
)


# =============================================================================
# GPU VERIFICATION
# =============================================================================

print("=" * 90)
print("KAGGLE BGE-M3 DUAL-T4 OPTIMIZATION PILOT")
print("=" * 90)

print(
    f"PyTorch version        : "
    f"{torch.__version__}"
)

print(
    f"CUDA available         : "
    f"{torch.cuda.is_available()}"
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not available. "
        "Enable GPU in Kaggle notebook settings."
    )

gpu_count = torch.cuda.device_count()

print(
    f"GPUs available         : "
    f"{gpu_count}"
)

if gpu_count < 2:
    raise RuntimeError(
        f"Expected 2 GPUs but found {gpu_count}."
    )

for i in range(gpu_count):

    props = torch.cuda.get_device_properties(i)

    print(
        f"\nGPU {i}"
    )

    print(
        f"  Name                : "
        f"{props.name}"
    )

    print(
        f"  Memory              : "
        f"{props.total_memory / (1024**3):.2f} GB"
    )


# =============================================================================
# VERIFY INPUT DATASET
# =============================================================================

if not INPUT_DIR.exists():

    raise FileNotFoundError(
        f"Kaggle dataset not found:\n{INPUT_DIR}"
    )

shards = sorted(
    INPUT_DIR.rglob("*.jsonl")
)

print(
    f"\nInput shards           : "
    f"{len(shards):,}"
)

if len(shards) != 151:

    print(
        "WARNING: Expected 151 shards."
    )


# =============================================================================
# LOAD DISTRIBUTED 5,000-CHUNK SAMPLE
# =============================================================================
# Sample evenly across all shards rather than taking only the first 5,000
# chunks. This keeps the benchmark representative of the corpus.

print("\nLoading distributed pilot sample...")

base_per_shard = (
    PILOT_CHUNKS // len(shards)
)

remainder = (
    PILOT_CHUNKS % len(shards)
)

texts = []
sampled_per_shard = 0

for shard_index, shard_path in enumerate(shards):

    target = (
        base_per_shard
        + (
            1
            if shard_index < remainder
            else 0
        )
    )

    shard_count = 0

    with open(
        shard_path,
        "r",
        encoding="utf-8"
    ) as file:

        for line in file:

            if not line.strip():
                continue

            record = json.loads(line)

            texts.append(
                record["text"]
            )

            shard_count += 1

            if shard_count >= target:
                break


if len(texts) != PILOT_CHUNKS:

    raise RuntimeError(
        f"Expected {PILOT_CHUNKS:,} chunks but loaded "
        f"{len(texts):,}."
    )

print(
    f"Chunks loaded          : "
    f"{len(texts):,}"
)


# =============================================================================
# LOAD BGE-M3
# =============================================================================

print("\nLoading BGE-M3...")

model = SentenceTransformer(
    MODEL_NAME,
    device="cuda:0"
)

model.max_seq_length = (
    MAX_SEQ_LENGTH
)

print(
    f"Model                  : "
    f"{MODEL_NAME}"
)

print(
    f"Sequence length        : "
    f"{MAX_SEQ_LENGTH}"
)

print(
    f"Batch size / GPU       : "
    f"{BATCH_SIZE}"
)

print(
    f"Pool chunk size        : "
    f"{POOL_CHUNK_SIZE}"
)


# =============================================================================
# START MULTI-GPU POOL
# =============================================================================

print("\nStarting dual-GPU pool...")

pool_start = time.time()

pool = model.start_multi_process_pool(
    target_devices=[
        "cuda:0",
        "cuda:1",
    ]
)

pool_start_time = (
    time.time()
    - pool_start
)

print(
    f"Pool startup time      : "
    f"{pool_start_time:.2f} sec"
)


# =============================================================================
# EMBEDDING BENCHMARK
# =============================================================================

print("\nGenerating embeddings...")

start_time = time.time()

embeddings = model.encode(
    texts,
    pool=pool,
    batch_size=BATCH_SIZE,
    chunk_size=POOL_CHUNK_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

elapsed_seconds = (
    time.time()
    - start_time
)


# =============================================================================
# STOP MULTI-GPU POOL
# =============================================================================

model.stop_multi_process_pool(
    pool
)


# =============================================================================
# PERFORMANCE METRICS
# =============================================================================

num_embeddings = (
    embeddings.shape[0]
)

embedding_dimension = (
    embeddings.shape[1]
)

throughput = (
    num_embeddings
    / elapsed_seconds
)

chunks_per_minute = (
    throughput
    * 60
)

estimated_full_hours = (
    1_506_367
    / throughput
    / 3600
)

estimated_fp16_storage_gb = (
    1_506_367
    * embedding_dimension
    * 2
    / (1024**3)
)

estimated_fp32_storage_gb = (
    1_506_367
    * embedding_dimension
    * 4
    / (1024**3)
)


# =============================================================================
# VECTOR SANITY CHECK
# =============================================================================

sample_norms = np.linalg.norm(
    embeddings[:10],
    axis=1
)


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("KAGGLE DUAL-T4 BGE-M3 PILOT SUMMARY")
print("=" * 90)

print(
    f"Chunks embedded          : "
    f"{num_embeddings:,}"
)

print(
    f"GPUs                     : "
    f"{gpu_count}"
)

print(
    f"Batch size / GPU         : "
    f"{BATCH_SIZE}"
)

print(
    f"Pool chunk size          : "
    f"{POOL_CHUNK_SIZE}"
)

print(
    f"Max sequence length      : "
    f"{MAX_SEQ_LENGTH}"
)

print(
    f"Embedding dimensions     : "
    f"{embedding_dimension}"
)

print(
    f"Embedding runtime        : "
    f"{elapsed_seconds:.2f} sec"
)

print(
    f"Throughput               : "
    f"{throughput:.2f} chunks/sec"
)

print(
    f"Throughput               : "
    f"{chunks_per_minute:.2f} chunks/min"
)

print(
    f"Estimated 1.5M runtime   : "
    f"{estimated_full_hours:.2f} hours"
)

print(
    f"Estimated FP16 storage   : "
    f"{estimated_fp16_storage_gb:.2f} GB"
)

print(
    f"Estimated FP32 storage   : "
    f"{estimated_fp32_storage_gb:.2f} GB"
)

print(
    f"Vector norm range        : "
    f"{sample_norms.min():.4f} - "
    f"{sample_norms.max():.4f}"
)

print("=" * 90)


# =============================================================================
# CLEANUP
# =============================================================================

del embeddings
del texts
del model
del sample_norms

gc.collect()

torch.cuda.empty_cache()

print(
    "\nKaggle dual-T4 benchmark completed."
)

KAGGLE BGE-M3 DUAL-T4 OPTIMIZATION PILOT
PyTorch version        : 2.10.0+cu128
CUDA available         : True
GPUs available         : 2

GPU 0
  Name                : Tesla T4
  Memory              : 14.56 GB

GPU 1
  Name                : Tesla T4
  Memory              : 14.56 GB

Input shards           : 151

Loading distributed pilot sample...


Chunks loaded          : 5,000

Loading BGE-M3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Model                  : BAAI/bge-m3
Sequence length        : 1280
Batch size / GPU       : 128
Pool chunk size        : 1000

Starting dual-GPU pool...
Pool startup time      : 26.74 sec

Generating embeddings...


Chunks:   0%|          | 0/5 [00:00<?, ?it/s]

Process SpawnProcess-1:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/sentence_transformer/model.py", line 907, in _multi_process_worker
    embeddings = model.encode(inputs, device=target_device, **kwargs)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/util/decorators.py", line 41, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/sentence_transformer/model.p

In [4]:
# =============================================================================
# KAGGLE BGE-M3 DUAL-T4 OPTIMIZATION PILOT (5,000 CHUNKS @ 1280 SEQ LEN)
# =============================================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer


# =============================================================================
# CONFIGURATION
# =============================================================================

MODEL_NAME = "BAAI/bge-m3"

PILOT_CHUNKS = 5_000

# Retained strictly per workflow requirements
MAX_SEQ_LENGTH = 1_280

# Batch size PER GPU (128 per card = 256 combined concurrent batch)
BATCH_SIZE = 128

# Text chunks dispatched per worker process loop
POOL_CHUNK_SIZE = 1_000

INPUT_DIR = Path(
    "/kaggle/input/datasets/"
    "cliffordimaguezegie/"
    "telecom-reconciled-chunks"
)


# =============================================================================
# GPU VERIFICATION
# =============================================================================

print("=" * 90)
print("KAGGLE BGE-M3 DUAL-T4 OPTIMIZATION PILOT")
print("=" * 90)

print(f"PyTorch version        : {torch.__version__}")
print(f"CUDA available         : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not available. Enable GPU in Kaggle notebook settings."
    )

gpu_count = torch.cuda.device_count()

print(f"GPUs available         : {gpu_count}")

if gpu_count < 2:
    raise RuntimeError(
        f"Expected 2 GPUs but found {gpu_count}."
    )

for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"\nGPU {i}")
    print(f"  Name                 : {props.name}")
    print(f"  Memory               : {props.total_memory / (1024**3):.2f} GB")


# =============================================================================
# VERIFY INPUT DATASET
# =============================================================================

if not INPUT_DIR.exists():
    raise FileNotFoundError(
        f"Kaggle dataset not found:\n{INPUT_DIR}"
    )

shards = sorted(INPUT_DIR.rglob("*.jsonl"))

print(f"\nInput shards           : {len(shards):,}")

if len(shards) != 151:
    print("WARNING: Expected 151 shards.")


# =============================================================================
# LOAD DISTRIBUTED 5,000-CHUNK SAMPLE
# =============================================================================

print("\nLoading distributed pilot sample...")

base_per_shard = PILOT_CHUNKS // len(shards)
remainder = PILOT_CHUNKS % len(shards)

texts = []

for shard_index, shard_path in enumerate(shards):
    target = base_per_shard + (1 if shard_index < remainder else 0)
    shard_count = 0

    with open(shard_path, "r", encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue

            record = json.loads(line)
            texts.append(record["text"])
            shard_count += 1

            if shard_count >= target:
                break

if len(texts) != PILOT_CHUNKS:
    raise RuntimeError(
        f"Expected {PILOT_CHUNKS:,} chunks but loaded {len(texts):,}."
    )

print(f"Chunks loaded          : {len(texts):,}")


# =============================================================================
# LOAD BGE-M3 & CONVERT TO FP16
# =============================================================================

print("\nLoading BGE-M3 model on CPU baseline...")

# Load on CPU first to avoid binding main process CUDA context prior to pool fork
model = SentenceTransformer(MODEL_NAME, device="cpu")
model.max_seq_length = MAX_SEQ_LENGTH

# Enable FP16 (Half Precision) to accelerate T4 Tensor Core matrix multiplication
print("Converting model weights to FP16 (Half Precision)...")
model.half()

print(f"Model                  : {MODEL_NAME}")
print(f"Sequence length        : {MAX_SEQ_LENGTH}")
print(f"Batch size / GPU       : {BATCH_SIZE}")
print(f"Pool chunk size        : {POOL_CHUNK_SIZE}")


# =============================================================================
# START MULTI-GPU POOL
# =============================================================================

print("\nStarting dual-GPU pool...")

pool_start = time.time()

# Target devices automatically move process copies to GPU memory
pool = model.start_multi_process_pool(target_devices=["cuda:0", "cuda:1"])

pool_start_time = time.time() - pool_start

print(f"Pool startup time      : {pool_start_time:.2f} sec")


# =============================================================================
# EMBEDDING BENCHMARK
# =============================================================================

print("\nGenerating embeddings...")

start_time = time.time()

# Use encode_multi_process directly to maximize pool efficiency
embeddings = model.encode_multi_process(
    texts,
    pool=pool,
    batch_size=BATCH_SIZE,
    chunk_size=POOL_CHUNK_SIZE,
    normalize_embeddings=True,
)

elapsed_seconds = time.time() - start_time


# =============================================================================
# STOP MULTI-GPU POOL
# =============================================================================

model.stop_multi_process_pool(pool)


# =============================================================================
# PERFORMANCE METRICS
# =============================================================================

num_embeddings = embeddings.shape[0]
embedding_dimension = embeddings.shape[1]

throughput = num_embeddings / elapsed_seconds
chunks_per_minute = throughput * 60

# Full scale calculation based on 1,506,367 chunks
estimated_full_hours = 1_506_367 / throughput / 3600
estimated_fp16_storage_gb = (1_506_367 * embedding_dimension * 2) / (1024**3)
estimated_fp32_storage_gb = (1_506_367 * embedding_dimension * 4) / (1024**3)


# =============================================================================
# VECTOR SANITY CHECK
# =============================================================================

sample_norms = np.linalg.norm(embeddings[:10], axis=1)


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("KAGGLE DUAL-T4 BGE-M3 PILOT SUMMARY")
print("=" * 90)

print(f"Chunks embedded          : {num_embeddings:,}")
print(f"GPUs                     : {gpu_count}")
print(f"Batch size / GPU         : {BATCH_SIZE}")
print(f"Pool chunk size          : {POOL_CHUNK_SIZE}")
print(f"Max sequence length      : {MAX_SEQ_LENGTH}")
print(f"Embedding dimensions     : {embedding_dimension}")
print(f"Embedding runtime        : {elapsed_seconds:.2f} sec")
print(f"Throughput               : {throughput:.2f} chunks/sec")
print(f"Throughput               : {chunks_per_minute:.2f} chunks/min")
print(f"Estimated 1.5M runtime   : {estimated_full_hours:.2f} hours")
print(f"Estimated FP16 storage   : {estimated_fp16_storage_gb:.2f} GB")
print(f"Estimated FP32 storage   : {estimated_fp32_storage_gb:.2f} GB")
print(f"Vector norm range        : {sample_norms.min():.4f} - {sample_norms.max():.4f}")
print("=" * 90)


# =============================================================================
# CLEANUP
# =============================================================================

del embeddings
del texts
del model
del sample_norms

gc.collect()
torch.cuda.empty_cache()

print("\nKaggle dual-T4 benchmark completed.")

KAGGLE BGE-M3 DUAL-T4 OPTIMIZATION PILOT
PyTorch version        : 2.10.0+cu128
CUDA available         : True
GPUs available         : 2

GPU 0
  Name                 : Tesla T4
  Memory               : 14.56 GB

GPU 1
  Name                 : Tesla T4
  Memory               : 14.56 GB

Input shards           : 151

Loading distributed pilot sample...


Chunks loaded          : 5,000

Loading BGE-M3 model on CPU baseline...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Converting model weights to FP16 (Half Precision)...
Model                  : BAAI/bge-m3
Sequence length        : 1280
Batch size / GPU       : 128
Pool chunk size        : 1000

Starting dual-GPU pool...
Pool startup time      : 24.87 sec

Generating embeddings...


/tmp/ipykernel_58/2931760505.py:168: DeprecationWarning: The `encode_multi_process` method has been deprecated, and its functionality has been integrated into `encode`. You can now call `encode` with the same parameters to achieve multi-process encoding.
  embeddings = model.encode_multi_process(



KAGGLE DUAL-T4 BGE-M3 PILOT SUMMARY
Chunks embedded          : 5,000
GPUs                     : 2
Batch size / GPU         : 128
Pool chunk size          : 1000
Max sequence length      : 1280
Embedding dimensions     : 1024
Embedding runtime        : 140.81 sec
Throughput               : 35.51 chunks/sec
Throughput               : 2130.58 chunks/min
Estimated 1.5M runtime   : 11.78 hours
Estimated FP16 storage   : 2.87 GB
Estimated FP32 storage   : 5.75 GB
Vector norm range        : 1.0000 - 1.0000

Kaggle dual-T4 benchmark completed.


In [5]:
# =============================================================================
# KAGGLE BGE-M3 DUAL-T4 OPTIMIZATION PILOT
# Conservative multi-GPU configuration after OOM
# =============================================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer


# =============================================================================
# CONFIGURATION
# =============================================================================

MODEL_NAME = "BAAI/bge-m3"

PILOT_CHUNKS = 5_000

MAX_SEQ_LENGTH = 1_280

# IMPORTANT:
# This is PER GPU.
# 64 x 2 GPUs = 128 effective concurrent batch.
BATCH_SIZE = 64

# Work dispatched to each GPU worker at a time.
POOL_CHUNK_SIZE = 500

INPUT_DIR = Path(
    "/kaggle/input/datasets/"
    "cliffordimaguezegie/"
    "telecom-reconciled-chunks"
)


# =============================================================================
# GPU VERIFICATION
# =============================================================================

print("=" * 90)
print("KAGGLE BGE-M3 DUAL-T4 OPTIMIZATION PILOT")
print("=" * 90)

print(
    f"PyTorch version        : {torch.__version__}"
)

print(
    f"CUDA available         : "
    f"{torch.cuda.is_available()}"
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not available."
    )

gpu_count = torch.cuda.device_count()

print(
    f"GPUs available         : "
    f"{gpu_count}"
)

if gpu_count < 2:
    raise RuntimeError(
        f"Expected 2 GPUs but found {gpu_count}."
    )

for i in range(gpu_count):

    props = torch.cuda.get_device_properties(i)

    print(f"\nGPU {i}")

    print(
        f"  Name                 : "
        f"{props.name}"
    )

    print(
        f"  Memory               : "
        f"{props.total_memory / (1024**3):.2f} GB"
    )


# =============================================================================
# VERIFY INPUT DATASET
# =============================================================================

if not INPUT_DIR.exists():
    raise FileNotFoundError(
        f"Kaggle dataset not found:\n{INPUT_DIR}"
    )

shards = sorted(
    INPUT_DIR.rglob("*.jsonl")
)

print(
    f"\nInput shards           : "
    f"{len(shards):,}"
)


# =============================================================================
# LOAD DISTRIBUTED 5,000-CHUNK SAMPLE
# =============================================================================

print("\nLoading distributed pilot sample...")

base_per_shard = (
    PILOT_CHUNKS // len(shards)
)

remainder = (
    PILOT_CHUNKS % len(shards)
)

texts = []

for shard_index, shard_path in enumerate(shards):

    target = (
        base_per_shard
        + (
            1
            if shard_index < remainder
            else 0
        )
    )

    shard_count = 0

    with open(
        shard_path,
        "r",
        encoding="utf-8"
    ) as file:

        for line in file:

            if not line.strip():
                continue

            record = json.loads(line)

            texts.append(
                record["text"]
            )

            shard_count += 1

            if shard_count >= target:
                break


if len(texts) != PILOT_CHUNKS:

    raise RuntimeError(
        f"Expected {PILOT_CHUNKS:,} chunks but loaded "
        f"{len(texts):,}."
    )

print(
    f"Chunks loaded          : "
    f"{len(texts):,}"
)


# =============================================================================
# LOAD MODEL ON CPU
# =============================================================================

print("\nLoading BGE-M3 on CPU...")

model = SentenceTransformer(
    MODEL_NAME,
    device="cpu"
)

model.max_seq_length = (
    MAX_SEQ_LENGTH
)

# FP16 weights before worker creation.
model.half()

print(
    f"Model                  : "
    f"{MODEL_NAME}"
)

print(
    f"Sequence length        : "
    f"{MAX_SEQ_LENGTH}"
)

print(
    f"Batch size / GPU       : "
    f"{BATCH_SIZE}"
)

print(
    f"Effective batch        : "
    f"{BATCH_SIZE * gpu_count}"
)

print(
    f"Pool chunk size        : "
    f"{POOL_CHUNK_SIZE}"
)


# =============================================================================
# START MULTI-GPU POOL
# =============================================================================

print("\nStarting dual-GPU pool...")

pool_start = time.time()

pool = model.start_multi_process_pool(
    target_devices=[
        "cuda:0",
        "cuda:1"
    ]
)

pool_start_time = (
    time.time()
    - pool_start
)

print(
    f"Pool startup time      : "
    f"{pool_start_time:.2f} sec"
)


# =============================================================================
# EMBEDDING BENCHMARK
# =============================================================================

print("\nGenerating embeddings...")

start_time = time.time()

# Current Sentence Transformers path:
# encode() + reusable multi-GPU pool.
embeddings = model.encode(
    texts,
    pool=pool,
    batch_size=BATCH_SIZE,
    chunk_size=POOL_CHUNK_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

elapsed_seconds = (
    time.time()
    - start_time
)


# =============================================================================
# STOP POOL
# =============================================================================

model.stop_multi_process_pool(
    pool
)


# =============================================================================
# METRICS
# =============================================================================

num_embeddings = (
    embeddings.shape[0]
)

embedding_dimension = (
    embeddings.shape[1]
)

throughput = (
    num_embeddings
    / elapsed_seconds
)

chunks_per_minute = (
    throughput
    * 60
)

estimated_full_hours = (
    1_506_367
    / throughput
    / 3600
)

estimated_fp16_storage_gb = (
    1_506_367
    * embedding_dimension
    * 2
    / (1024**3)
)

sample_norms = np.linalg.norm(
    embeddings[:10],
    axis=1
)


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 90)
print("KAGGLE DUAL-T4 BGE-M3 PILOT SUMMARY")
print("=" * 90)

print(
    f"Chunks embedded          : "
    f"{num_embeddings:,}"
)

print(
    f"GPUs                     : "
    f"{gpu_count}"
)

print(
    f"Batch size / GPU         : "
    f"{BATCH_SIZE}"
)

print(
    f"Effective batch          : "
    f"{BATCH_SIZE * gpu_count}"
)

print(
    f"Pool chunk size          : "
    f"{POOL_CHUNK_SIZE}"
)

print(
    f"Max sequence length      : "
    f"{MAX_SEQ_LENGTH}"
)

print(
    f"Embedding dimensions     : "
    f"{embedding_dimension}"
)

print(
    f"Embedding runtime        : "
    f"{elapsed_seconds:.2f} sec"
)

print(
    f"Throughput               : "
    f"{throughput:.2f} chunks/sec"
)

print(
    f"Throughput               : "
    f"{chunks_per_minute:.2f} chunks/min"
)

print(
    f"Estimated 1.5M runtime   : "
    f"{estimated_full_hours:.2f} hours"
)

print(
    f"Estimated FP16 storage   : "
    f"{estimated_fp16_storage_gb:.2f} GB"
)

print(
    f"Vector norm range        : "
    f"{sample_norms.min():.4f} - "
    f"{sample_norms.max():.4f}"
)

print("=" * 90)


# =============================================================================
# CLEANUP
# =============================================================================

del embeddings
del texts
del model
del sample_norms

gc.collect()

torch.cuda.empty_cache()

print(
    "\nKaggle dual-T4 benchmark completed."
)

KAGGLE BGE-M3 DUAL-T4 OPTIMIZATION PILOT
PyTorch version        : 2.10.0+cu128
CUDA available         : True
GPUs available         : 2

GPU 0
  Name                 : Tesla T4
  Memory               : 14.56 GB

GPU 1
  Name                 : Tesla T4
  Memory               : 14.56 GB

Input shards           : 151

Loading distributed pilot sample...
Chunks loaded          : 5,000

Loading BGE-M3 on CPU...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model                  : BAAI/bge-m3
Sequence length        : 1280
Batch size / GPU       : 64
Effective batch        : 128
Pool chunk size        : 500

Starting dual-GPU pool...
Pool startup time      : 24.07 sec

Generating embeddings...


Chunks:   0%|          | 0/10 [00:00<?, ?it/s]


KAGGLE DUAL-T4 BGE-M3 PILOT SUMMARY
Chunks embedded          : 5,000
GPUs                     : 2
Batch size / GPU         : 64
Effective batch          : 128
Pool chunk size          : 500
Max sequence length      : 1280
Embedding dimensions     : 1024
Embedding runtime        : 138.72 sec
Throughput               : 36.04 chunks/sec
Throughput               : 2162.64 chunks/min
Estimated 1.5M runtime   : 11.61 hours
Estimated FP16 storage   : 2.87 GB
Vector norm range        : 1.0000 - 1.0000

Kaggle dual-T4 benchmark completed.


### **Final Reusable Embedding**

2 × Tesla T4
BGE-M3
1,280 tokens
64 chunks/GPU
128 effective batch
FP16
~36 chunks/sec
~4.6 min per 10,000 chunks

In [6]:
# =============================================================================
# KAGGLE — FULL RESUMABLE BGE-M3 DUAL-T4 EMBEDDING PIPELINE
# =============================================================================
#
# FINAL VALIDATED CONFIGURATION
#   Model              : BAAI/bge-m3
#   GPUs               : 2 x Tesla T4
#   Sequence length    : 1280
#   Batch / GPU        : 64
#   Effective batch    : 128
#   Pool chunk size    : 500
#   Precision          : FP16
#   Checkpoint         : Every 10,000 chunks
#   Embedding storage  : FP16
#
# IMPORTANT
#   Enable Kaggle Notebook:
#       Settings -> Persistence -> Files only
#
#   This allows checkpoint/output files in /kaggle/working to persist
#   across interactive sessions. Kaggle also saves /kaggle/working as
#   notebook output when a version is saved.
# =============================================================================


# =============================================================================
# IMPORTS
# =============================================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


# =============================================================================
# CONFIGURATION
# =============================================================================

MODEL_NAME = "BAAI/bge-m3"

TOTAL_CHUNKS = 1_506_367

MAX_SEQ_LENGTH = 1_280

BATCH_SIZE = 64

POOL_CHUNK_SIZE = 500

CHECKPOINT_CHUNKS = 10_000


# =============================================================================
# INPUT DATASET
# =============================================================================

INPUT_DIR = Path(
    "/kaggle/input/datasets/"
    "cliffordimaguezegie/"
    "telecom-reconciled-chunks"
)


# =============================================================================
# OUTPUT / CHECKPOINT DIRECTORIES
# =============================================================================

OUTPUT_DIR = Path(
    "/kaggle/working/"
    "bge_m3_embeddings"
)

CHECKPOINT_DIR = Path(
    "/kaggle/working/"
    "bge_m3_checkpoint"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CHECKPOINT_FILE = (
    CHECKPOINT_DIR
    / "embedding_checkpoint.json"
)

MANIFEST_FILE = (
    CHECKPOINT_DIR
    / "embedding_manifest.json"
)


# =============================================================================
# HEADER
# =============================================================================

print("=" * 100)
print("FULL RESUMABLE BGE-M3 DUAL-T4 EMBEDDING PIPELINE")
print("=" * 100)

print(
    f"Model                  : {MODEL_NAME}"
)

print(
    f"Total chunks            : "
    f"{TOTAL_CHUNKS:,}"
)

print(
    f"Sequence length         : "
    f"{MAX_SEQ_LENGTH}"
)

print(
    f"Batch / GPU             : "
    f"{BATCH_SIZE}"
)

print(
    f"Effective batch         : "
    f"{BATCH_SIZE * 2}"
)

print(
    f"Pool chunk size         : "
    f"{POOL_CHUNK_SIZE}"
)

print(
    f"Checkpoint interval     : "
    f"{CHECKPOINT_CHUNKS:,}"
)

print(
    f"Output directory        : "
    f"{OUTPUT_DIR}"
)

print(
    f"Checkpoint file         : "
    f"{CHECKPOINT_FILE}"
)


# =============================================================================
# GPU VERIFICATION
# =============================================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU not available."
    )

GPU_COUNT = torch.cuda.device_count()

if GPU_COUNT < 2:

    raise RuntimeError(
        f"Expected 2 GPUs but found {GPU_COUNT}."
    )

print("\nGPU configuration")

for gpu_id in range(GPU_COUNT):

    props = (
        torch.cuda.get_device_properties(
            gpu_id
        )
    )

    print(
        f"GPU {gpu_id}: "
        f"{props.name} | "
        f"{props.total_memory / (1024**3):.2f} GB"
    )


# =============================================================================
# VERIFY INPUT DATASET
# =============================================================================

if not INPUT_DIR.exists():

    raise FileNotFoundError(
        f"Input dataset not found:\n"
        f"{INPUT_DIR}"
    )

input_shards = sorted(
    INPUT_DIR.rglob("*.jsonl")
)

print(
    f"\nInput shards            : "
    f"{len(input_shards):,}"
)

if len(input_shards) != 151:

    print(
        "WARNING: Expected 151 input shards."
    )


# =============================================================================
# LOAD / INITIALIZE CHECKPOINT
# =============================================================================

if CHECKPOINT_FILE.exists():

    with open(
        CHECKPOINT_FILE,
        "r",
        encoding="utf-8"
    ) as file:

        checkpoint = json.load(file)

    print(
        "\nExisting checkpoint found."
    )

else:

    checkpoint = {
        "status": "INITIALIZED",
        "completed_chunks": 0,
        "completed_shards": [],
        "last_completed_shard": None,
        "total_chunks": TOTAL_CHUNKS,
        "embedding_dimensions": 1024,
        "embedding_dtype": "float16",
        "model": MODEL_NAME,
        "sequence_length": MAX_SEQ_LENGTH,
        "batch_size_per_gpu": BATCH_SIZE,
        "pool_chunk_size": POOL_CHUNK_SIZE,
        "checkpoint_chunks": CHECKPOINT_CHUNKS,
    }

    print(
        "\nNo previous checkpoint found."
    )


completed_shards = set(
    checkpoint.get(
        "completed_shards",
        []
    )
)

completed_chunks = checkpoint.get(
    "completed_chunks",
    0
)

print(
    f"Completed shards        : "
    f"{len(completed_shards):,}"
)

print(
    f"Completed chunks        : "
    f"{completed_chunks:,}"
)


# =============================================================================
# MODEL
# =============================================================================
#
# Load the model on CPU before starting the multi-GPU workers.
# This avoids creating the initial SentenceTransformer model directly
# on a GPU before worker processes are spawned.
# =============================================================================

print("\nLoading BGE-M3 on CPU...")

model = SentenceTransformer(
    MODEL_NAME,
    device="cpu"
)

model.max_seq_length = (
    MAX_SEQ_LENGTH
)

# FP16 before pool creation.
model.half()

print(
    "BGE-M3 loaded on CPU."
)


# =============================================================================
# START MULTI-GPU POOL
# =============================================================================

print(
    "\nStarting dual-GPU pool..."
)

pool_start = time.time()

pool = model.start_multi_process_pool(
    target_devices=[
        "cuda:0",
        "cuda:1"
    ]
)

pool_elapsed = (
    time.time()
    - pool_start
)

print(
    f"Pool startup time      : "
    f"{pool_elapsed:.2f} sec"
)


# =============================================================================
# PROCESS EACH RECONCILED SHARD
# =============================================================================

overall_start = time.time()

for shard_number, shard_path in enumerate(
    input_shards,
    start=1
):

    shard_name = shard_path.name


    # -------------------------------------------------------------------------
    # RESUME CHECK
    # -------------------------------------------------------------------------

    if shard_name in completed_shards:

        print(
            f"\nSKIP "
            f"{shard_name} "
            f"— already completed."
        )

        continue


    print("\n" + "=" * 100)

    print(
        f"PROCESSING SHARD "
        f"{shard_number}/{len(input_shards)}"
    )

    print(
        f"Shard                  : "
        f"{shard_name}"
    )

    print("=" * 100)


    # -------------------------------------------------------------------------
    # LOAD SHARD
    # -------------------------------------------------------------------------

    records = []

    with open(
        shard_path,
        "r",
        encoding="utf-8"
    ) as file:

        for line in file:

            if line.strip():

                records.append(
                    json.loads(line)
                )


    shard_count = len(records)

    print(
        f"Chunks in shard        : "
        f"{shard_count:,}"
    )


    # -------------------------------------------------------------------------
    # CREATE TEXT LIST
    # -------------------------------------------------------------------------

    texts = [
        record["text"]
        for record in records
    ]


    # -------------------------------------------------------------------------
    # EMBEDDING
    # -------------------------------------------------------------------------

    shard_start = time.time()

    print(
        "\nGenerating embeddings..."
    )

    embeddings = model.encode(
        texts,
        pool=pool,
        batch_size=BATCH_SIZE,
        chunk_size=POOL_CHUNK_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    shard_elapsed = (
        time.time()
        - shard_start
    )

    shard_rate = (
        shard_count
        / shard_elapsed
    )


    # -------------------------------------------------------------------------
    # VALIDATE SHARD
    # -------------------------------------------------------------------------

    if embeddings.shape[0] != shard_count:

        raise RuntimeError(
            f"Embedding count mismatch for "
            f"{shard_name}: "
            f"expected {shard_count}, "
            f"got {embeddings.shape[0]}"
        )

    if embeddings.shape[1] != 1024:

        raise RuntimeError(
            f"Unexpected embedding dimension: "
            f"{embeddings.shape[1]}"
        )


    # -------------------------------------------------------------------------
    # CONVERT TO FP16
    # -------------------------------------------------------------------------

    embeddings_fp16 = (
        embeddings
        .astype(
            np.float16,
            copy=False
        )
    )


    # -------------------------------------------------------------------------
    # OUTPUT FILENAMES
    # -------------------------------------------------------------------------

    output_file = (
        OUTPUT_DIR
        / shard_name.replace(
            ".jsonl",
            "_embeddings.npy"
        )
    )

    metadata_file = (
        OUTPUT_DIR
        / shard_name.replace(
            ".jsonl",
            "_metadata.jsonl"
        )
    )


    # -------------------------------------------------------------------------
    # SAVE EMBEDDINGS
    # -------------------------------------------------------------------------

    np.save(
        output_file,
        embeddings_fp16
    )


    # -------------------------------------------------------------------------
    # SAVE VECTOR → CHUNK MAPPING
    # -------------------------------------------------------------------------

    with open(
        metadata_file,
        "w",
        encoding="utf-8"
    ) as file:

        for vector_id, record in enumerate(
            records
        ):

            metadata = {
                "vector_id": vector_id,
                "chunk_id": record.get(
                    "chunk_id"
                ),
                "document_id": record.get(
                    "document_id"
                ),
                "source": record.get(
                    "source"
                ),
                "title": record.get(
                    "title"
                ),
                "path": record.get(
                    "path"
                ),
            }

            file.write(
                json.dumps(
                    metadata,
                    ensure_ascii=False
                )
                + "\n"
            )


    # -------------------------------------------------------------------------
    # CHECK OUTPUT FILES
    # -------------------------------------------------------------------------

    if not output_file.exists():

        raise RuntimeError(
            f"Embedding output was not created:\n"
            f"{output_file}"
        )

    if not metadata_file.exists():

        raise RuntimeError(
            f"Metadata output was not created:\n"
            f"{metadata_file}"
        )


    # -------------------------------------------------------------------------
    # UPDATE CHECKPOINT
    # -------------------------------------------------------------------------

    completed_shards.add(
        shard_name
    )

    completed_chunks += (
        shard_count
    )

    checkpoint.update({

        "status": "IN_PROGRESS",

        "completed_chunks":
            completed_chunks,

        "completed_shards":
            sorted(
                completed_shards
            ),

        "last_completed_shard":
            shard_name,

        "updated_epoch":
            time.time(),

        "updated_utc":
            time.strftime(
                "%Y-%m-%dT%H:%M:%SZ",
                time.gmtime()
            ),
    })


    with open(
        CHECKPOINT_FILE,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            checkpoint,
            file,
            indent=2
        )


    # -------------------------------------------------------------------------
    # SHARD SUMMARY
    # -------------------------------------------------------------------------

    print("\n" + "-" * 90)

    print(
        f"Shard completed         : "
        f"{shard_name}"
    )

    print(
        f"Shard chunks            : "
        f"{shard_count:,}"
    )

    print(
        f"Shard throughput        : "
        f"{shard_rate:.2f} chunks/sec"
    )

    print(
        f"Completed chunks        : "
        f"{completed_chunks:,} / "
        f"{TOTAL_CHUNKS:,}"
    )

    print(
        f"Progress                : "
        f"{completed_chunks / TOTAL_CHUNKS * 100:.2f}%"
    )

    print(
        f"Embedding file          : "
        f"{output_file.name}"
    )

    print(
        f"Metadata file           : "
        f"{metadata_file.name}"
    )

    print(
        f"Checkpoint updated      : "
        f"{CHECKPOINT_FILE}"
    )

    print("-" * 90)


    # -------------------------------------------------------------------------
    # CLEANUP SHARD MEMORY
    # -------------------------------------------------------------------------

    del records
    del texts
    del embeddings
    del embeddings_fp16

    gc.collect()

    torch.cuda.empty_cache()


# =============================================================================
# FINALIZE
# =============================================================================

total_elapsed = (
    time.time()
    - overall_start
)


if completed_chunks >= TOTAL_CHUNKS:

    checkpoint.update({

        "status": "COMPLETED",

        "completed_chunks":
            TOTAL_CHUNKS,

        "updated_epoch":
            time.time(),

        "updated_utc":
            time.strftime(
                "%Y-%m-%dT%H:%M:%SZ",
                time.gmtime()
            ),
    })


    with open(
        CHECKPOINT_FILE,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            checkpoint,
            file,
            indent=2
        )


    manifest = {

        "status": "COMPLETED",

        "model": MODEL_NAME,

        "total_chunks":
            TOTAL_CHUNKS,

        "completed_chunks":
            completed_chunks,

        "embedding_dimensions":
            1024,

        "embedding_dtype":
            "float16",

        "sequence_length":
            MAX_SEQ_LENGTH,

        "batch_size_per_gpu":
            BATCH_SIZE,

        "effective_batch_size":
            BATCH_SIZE * GPU_COUNT,

        "pool_chunk_size":
            POOL_CHUNK_SIZE,

        "embedding_directory":
            str(OUTPUT_DIR),

        "checkpoint_file":
            str(CHECKPOINT_FILE),

        "total_runtime_hours":
            total_elapsed / 3600,

    }


    with open(
        MANIFEST_FILE,
        "w",
        encoding="utf-8"
    ) as file:

        json.dump(
            manifest,
            file,
            indent=2
        )

    print(
        "\nFULL EMBEDDING RUN COMPLETED."
    )

else:

    print(
        "\nRUN STOPPED / INTERRUPTED."
    )

    print(
        f"Completed chunks: "
        f"{completed_chunks:,}"
    )

    print(
        "Re-run the same cell to resume."
    )


# =============================================================================
# STOP MULTI-GPU POOL
# =============================================================================

model.stop_multi_process_pool(
    pool
)


# =============================================================================
# CLEANUP
# =============================================================================

del model

gc.collect()

torch.cuda.empty_cache()


# =============================================================================
# FINAL STATUS
# =============================================================================

print("\n" + "=" * 100)

print(
    f"Completed chunks        : "
    f"{completed_chunks:,} / {TOTAL_CHUNKS:,}"
)

print(
    f"Completed shards        : "
    f"{len(completed_shards):,} / "
    f"{len(input_shards):,}"
)

print(
    f"Output directory        : "
    f"{OUTPUT_DIR}"
)

print(
    f"Checkpoint file         : "
    f"{CHECKPOINT_FILE}"
)

print("=" * 100)

FULL RESUMABLE BGE-M3 DUAL-T4 EMBEDDING PIPELINE
Model                  : BAAI/bge-m3
Total chunks            : 1,506,367
Sequence length         : 1280
Batch / GPU             : 64
Effective batch         : 128
Pool chunk size         : 500
Checkpoint interval     : 10,000
Output directory        : /kaggle/working/bge_m3_embeddings
Checkpoint file         : /kaggle/working/bge_m3_checkpoint/embedding_checkpoint.json

GPU configuration
GPU 0: Tesla T4 | 14.56 GB
GPU 1: Tesla T4 | 14.56 GB

Input shards            : 151

No previous checkpoint found.
Completed shards        : 0
Completed chunks        : 0

Loading BGE-M3 on CPU...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE-M3 loaded on CPU.

Starting dual-GPU pool...
Pool startup time      : 23.14 sec

PROCESSING SHARD 1/151
Shard                  : chunks_000001.jsonl
Chunks in shard        : 10,000

Generating embeddings...


Chunks:   0%|          | 0/20 [00:00<?, ?it/s]


------------------------------------------------------------------------------------------
Shard completed         : chunks_000001.jsonl
Shard chunks            : 10,000
Shard throughput        : 33.80 chunks/sec
Completed chunks        : 10,000 / 1,506,367
Progress                : 0.66%
Embedding file          : chunks_000001_embeddings.npy
Metadata file           : chunks_000001_metadata.jsonl
Checkpoint updated      : /kaggle/working/bge_m3_checkpoint/embedding_checkpoint.json
------------------------------------------------------------------------------------------

PROCESSING SHARD 2/151
Shard                  : chunks_000002.jsonl
Chunks in shard        : 10,000

Generating embeddings...


...


------------------------------------------------------------------------------------------
Shard completed         : chunks_000149.jsonl
Shard chunks            : 10,000
Shard throughput        : 44.86 chunks/sec
Completed chunks        : 1,490,000 / 1,506,367
Progress                : 98.91%
Embedding file          : chunks_000149_embeddings.npy
Metadata file           : chunks_000149_metadata.jsonl
Checkpoint updated      : /kaggle/working/bge_m3_checkpoint/embedding_checkpoint.json
------------------------------------------------------------------------------------------

PROCESSING SHARD 150/151
Shard                  : chunks_000150.jsonl
Chunks in shard        : 10,000

Generating embeddings...


Chunks:   0%|          | 0/20 [00:00<?, ?it/s]


------------------------------------------------------------------------------------------
Shard completed         : chunks_000150.jsonl
Shard chunks            : 10,000
Shard throughput        : 44.18 chunks/sec
Completed chunks        : 1,500,000 / 1,506,367
Progress                : 99.58%
Embedding file          : chunks_000150_embeddings.npy
Metadata file           : chunks_000150_metadata.jsonl
Checkpoint updated      : /kaggle/working/bge_m3_checkpoint/embedding_checkpoint.json
------------------------------------------------------------------------------------------

PROCESSING SHARD 151/151
Shard                  : chunks_000151.jsonl
Chunks in shard        : 6,367

Generating embeddings...


Chunks:   0%|          | 0/13 [00:00<?, ?it/s]


------------------------------------------------------------------------------------------
Shard completed         : chunks_000151.jsonl
Shard chunks            : 6,367
Shard throughput        : 45.21 chunks/sec
Completed chunks        : 1,506,367 / 1,506,367
Progress                : 100.00%
Embedding file          : chunks_000151_embeddings.npy
Metadata file           : chunks_000151_metadata.jsonl
Checkpoint updated      : /kaggle/working/bge_m3_checkpoint/embedding_checkpoint.json
------------------------------------------------------------------------------------------

FULL EMBEDDING RUN COMPLETED.

Completed chunks        : 1,506,367 / 1,506,367
Completed shards        : 151 / 151
Output directory        : /kaggle/working/bge_m3_embeddings
Checkpoint file         : /kaggle/working/bge_m3_checkpoint/embedding_checkpoint.json


##### **Observation**

- Embeddings Completed
- Verification and backup will take place in next step

#### **Verify Embedding Size Details**

In [7]:
from pathlib import Path
import numpy as np

OUTPUT_DIR = Path(
    "/kaggle/working/bge_m3_embeddings"
)

embedding_files = sorted(
    OUTPUT_DIR.glob("*_embeddings.npy")
)

metadata_files = sorted(
    OUTPUT_DIR.glob("*_metadata.jsonl")
)

embedding_size = sum(
    f.stat().st_size
    for f in embedding_files
)

metadata_size = sum(
    f.stat().st_size
    for f in metadata_files
)

print("=" * 90)
print("FINAL EMBEDDING OUTPUT VERIFICATION")
print("=" * 90)

print(f"Embedding shards : {len(embedding_files):,}")
print(f"Metadata shards  : {len(metadata_files):,}")

print(
    f"Embedding size   : "
    f"{embedding_size / (1024**3):.2f} GB"
)

print(
    f"Metadata size    : "
    f"{metadata_size / (1024**3):.2f} GB"
)

print(
    f"Total            : "
    f"{(embedding_size + metadata_size) / (1024**3):.2f} GB"
)

# Check each embedding shard dimension
for f in embedding_files[:3]:

    arr = np.load(f, mmap_mode="r")

    print(
        f"{f.name}: "
        f"shape={arr.shape}, "
        f"dtype={arr.dtype}"
    )

print("=" * 90)

FINAL EMBEDDING OUTPUT VERIFICATION
Embedding shards : 151
Metadata shards  : 151
Embedding size   : 2.87 GB
Metadata size    : 0.47 GB
Total            : 3.34 GB
chunks_000001_embeddings.npy: shape=(10000, 1024), dtype=float16
chunks_000002_embeddings.npy: shape=(10000, 1024), dtype=float16
chunks_000003_embeddings.npy: shape=(10000, 1024), dtype=float16


#### **Make Embeddings Persistent Kaggle Dataset**

In [1]:
from pathlib import Path

SOURCE = Path(
    "/kaggle/working/bge_m3_embeddings"
)

print("Exists:", SOURCE.exists())
print("Files:", len(list(SOURCE.iterdir())))

print("\nSample files:")
for f in sorted(SOURCE.iterdir())[:5]:
    print(f.name)

Exists: True
Files: 302

Sample files:
chunks_000001_embeddings.npy
chunks_000001_metadata.jsonl
chunks_000002_embeddings.npy
chunks_000002_metadata.jsonl
chunks_000003_embeddings.npy


In [2]:
from pathlib import Path
import shutil

SOURCE = Path(
    "/kaggle/working/bge_m3_embeddings"
)

BACKUP = Path(
    "/kaggle/working/telecom_bge_m3_embeddings_dataset"
)

BACKUP.mkdir(
    parents=True,
    exist_ok=True
)

# Copy embedding and metadata files only
for file in SOURCE.iterdir():

    if file.is_file() and (
        file.name.endswith("_embeddings.npy")
        or file.name.endswith("_metadata.jsonl")
    ):

        shutil.copy2(
            file,
            BACKUP / file.name
        )

print(
    f"Backup files prepared: "
    f"{len(list(BACKUP.iterdir())):,}"
)

Backup files prepared: 302


In [3]:
import json

metadata = {
    "title": "Telecom BGE-M3 Embeddings",
    "id": "cliffordimaguezegie/telecom-bge-m3-embeddings",
    "description": (
        "BGE-M3 embeddings for the reconciled telecom "
        "knowledge base. 1,506,367 chunks, 1,024-dimensional "
        "FP16 vectors generated using a 1,280-token sequence length."
    ),
    "licenses": [
        {
            "name": "unknown"
        }
    ]
}

metadata_path = (
    BACKUP / "dataset-metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

print(metadata_path)

/kaggle/working/telecom_bge_m3_embeddings_dataset/dataset-metadata.json


In [4]:
!kaggle datasets create \
    -p /kaggle/working/telecom_bge_m3_embeddings_dataset \
    -r skip

Starting upload for file chunks_000056_metadata.jsonl
100%|██████████████████████████████████████| 2.93M/2.93M [00:00<00:00, 5.63MB/s]
Upload successful: chunks_000056_metadata.jsonl (3MB)
Starting upload for file chunks_000057_embeddings.npy
100%|██████████████████████████████████████| 19.5M/19.5M [00:00<00:00, 34.4MB/s]
Upload successful: chunks_000057_embeddings.npy (20MB)
Starting upload for file chunks_000001_embeddings.npy
100%|██████████████████████████████████████| 19.5M/19.5M [00:00<00:00, 27.8MB/s]
Upload successful: chunks_000001_embeddings.npy (20MB)
Starting upload for file chunks_000001_metadata.jsonl
100%|██████████████████████████████████████| 2.95M/2.95M [00:00<00:00, 6.79MB/s]
Upload successful: chunks_000001_metadata.jsonl (3MB)
Starting upload for file chunks_000035_embeddings.npy
100%|██████████████████████████████████████| 19.5M/19.5M [00:00<00:00, 31.3MB/s]
Upload successful: chunks_000035_embeddings.npy (20MB)
Starting upload for file chunks_000133_embeddings.np

##### **Observations**

- Embeddings Successfully Converted to Persistent Dataset in Kaggle